# nuPlan semantic-interaction visual inspection, Excel export, and map-only video

This notebook selects a scenario and frame, draws the map and directed semantic graph, and saves the complete inspection tables to an Excel workbook.

**v9.5.26 interaction update:**
- Existing `np:follows` and `np:overtakes` visualization is preserved.
- Active `np:changesLane(vehicle_like, target_lane)` assertions are visualized independently of the ordinary agent-to-agent edge table.
- No source/target lane highlighting is added. The purple arrow starts at the current subject position and points to the fixed target-lane completion point from `evidence_json`.
- Both `ego_to_structure` and `agent_to_structure` relation switches must be enabled to display every lane-change case.

**v9.5.38 traffic-light visualization fix:**
- the general video pipeline now reads `np:controls`, `np:hasSignalState`, and
  `np:isRelevantSignal` directly from the current-frame assertions;
- canonical `traffic_signal:*` and `controlled_movement:*` resources are
  grounded back to their native nuPlan lane connector for visualization;
- `isRelevantSignal` is drawn for all active agents in the scene;
- the physical traffic-light map object is selected near the controlled
  connector entry and labeled with the current canonical signal state;
- traffic-light predicates are written to `video_edge_events.csv`, so
  `run_parallel_exports.sh` and `run_category_videos.sh` can be verified by
  predicate name.


**v9.5.39 yieldsTo update:**
- `np:yieldsTo(S,O)` is rendered as an independent gold directed semantic arrow;
- the active yield assertion is grounded to its stored crossing conflict reference point;
- a dashed subject-to-reference segment and an `X` marker make the yielding location visible in both static maps and generated videos;
- `video_edge_events.csv` contains `yieldsTo`, so per-predicate video coverage can be checked automatically.


In [ ]:
import os
# ============================================================
# 1. PATHS AND USER SETTINGS
# ============================================================

from pathlib import Path

PROJECT_ROOT = Path.cwd()

DATASET_ROOT = Path(os.environ.get("NUPLAN_DATASET_ROOT", PROJECT_ROOT / "data" / "nuplan"))
MAP_ROOT = Path(os.environ.get("NUPLAN_MAP_ROOT", PROJECT_ROOT / "data" / "maps"))
MAP_VERSION = "nuplan-maps-v1.0"

SCENARIO_FILTER_YAML = PROJECT_ROOT / "configs/test14-random_reduced.yaml"

## v9.5.26 interaction output: follows + lane changes + merges + overtakes.
OUTPUT_NAME = "interaction_merge_v9_5_31"
OUTPUT_DIR = PROJECT_ROOT / "outputs_python" / OUTPUT_NAME

SAMPLE_INTERVAL_S = 0.5
DEFAULT_MAP_RADIUS_M = 80.0
DEFAULT_MAX_TABLE_ROWS = 200

print("Project root:", PROJECT_ROOT)
print("Output directory:", OUTPUT_DIR)
print("Output exists:", OUTPUT_DIR.exists())


In [ ]:
# ============================================================
# 2. IMPORTS AND PROJECT INITIALIZATION
# ============================================================

import json
import math
import random
import sys
import textwrap
from collections import Counter
from functools import lru_cache
from pathlib import Path
from typing import Any, Iterable, Optional, Set

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Polygon as MplPolygon
from IPython.display import display

# Prefer the adapter bundled with this package.  The wider project can also
# contain an older ``src/nuplan_predicate_kg`` checkout; placing that path
# ahead of the package runtime breaks video export with newer nuPlan versions.
_runtime_src_candidates = []
if globals().get("BUNDLED_SRC") is not None:
    _runtime_src_candidates.append(Path(globals()["BUNDLED_SRC"]))
_runtime_src_candidates.extend([
    Path.cwd() / "src",
    PROJECT_ROOT / "nuPlan_Predicates_KG_v9_5_39" / "src",
    PROJECT_ROOT / "src",
])

project_src = next(
    (p for p in _runtime_src_candidates
     if (p / "nuplan_predicate_kg" / "adapters" / "nuplan_adapter.py").exists()),
    PROJECT_ROOT / "src",
)
# Move the selected runtime to the front even if it is already present later.
while str(project_src) in sys.path:
    sys.path.remove(str(project_src))
sys.path.insert(0, str(project_src))

from nuplan_predicate_kg.adapters.nuplan_adapter import build_mini_scenarios, iter_frames

# Debug lane labels first use already-exported map assertions and safely
# fall back to direct nuPlan map-API containment when those assertions are absent.
# Do not import the extractor map implementation here: that module initializes
# the extractor CLI and would conflict with batch-video arguments.

try:
    import yaml
except ImportError as exc:
    raise ImportError("Install PyYAML in the active environment: pip install pyyaml") from exc

try:
    from nuplan.common.actor_state.state_representation import Point2D
    from nuplan.common.maps.maps_datatypes import SemanticMapLayer
    NUPLAN_MAP_IMPORTS_AVAILABLE = True
except Exception as exc:
    print("Warning: nuPlan map imports are unavailable:", exc)
    NUPLAN_MAP_IMPORTS_AVAILABLE = False

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 220)

for required_path, label in [
    (PROJECT_ROOT, "PROJECT_ROOT"),
    (DATASET_ROOT, "DATASET_ROOT"),
    (MAP_ROOT, "MAP_ROOT"),
    (OUTPUT_DIR, "OUTPUT_DIR"),
]:
    if not required_path.exists():
        raise FileNotFoundError(f"{label} does not exist: {required_path}")

print("Initialization completed.")


In [ ]:
# ============================================================
# 3. LOAD PARALLEL EXTRACTOR OUTPUT
# ============================================================

def _safe_json_load(value: Any, default=None):
    if value is None:
        return default
    if isinstance(value, (dict, list, bool, int, float)):
        return value
    try:
        return json.loads(value)
    except Exception:
        return default


def _is_positive_value(value: Any) -> bool:
    parsed = _safe_json_load(value, default=value)
    if isinstance(parsed, bool):
        return parsed
    if parsed is None:
        return False
    text = str(parsed).strip().lower()
    return text not in {"false", "0", "none", "null", "nan", ""}


def _unique_existing_paths(paths):
    result = []
    seen = set()

    for path in paths:
        path = Path(path)
        try:
            key = str(path.resolve())
        except Exception:
            key = str(path)

        if path.exists() and key not in seen:
            result.append(path)
            seen.add(key)

    return result


def discover_metadata_files(output_dir: Path, filename: str) -> list[Path]:
    """
    Discover metadata in both layouts:

      output_dir/<filename>
      output_dir/workers/worker_*/<filename>
      output_dir/workers/worker_*/batches/<filename>

    Recursive fallback is included for future versions.
    """
    output_dir = Path(output_dir)

    candidates = [
        output_dir / filename,
        *sorted((output_dir / "workers").glob(f"worker_*/{filename}")),
        *sorted((output_dir / "workers").glob(f"worker_*/batches/{filename}")),
    ]

    candidates.extend(sorted(output_dir.rglob(filename)))
    return _unique_existing_paths(candidates)


def discover_parquet_parts(output_dir: Path) -> list[Path]:
    """
    Return all assertion Parquet parts from parallel or single-process runs.
    Metadata Parquet files are excluded.
    """
    output_dir = Path(output_dir)

    candidates = [
        *sorted((output_dir / "batches").glob("*.parquet")),
        *sorted((output_dir / "workers").glob("worker_*/batches/*.parquet")),
        *sorted((output_dir / "workers").glob("worker_*/*.parquet")),
    ]

    # Future-proof recursive fallback.
    candidates.extend(sorted(output_dir.rglob("*.parquet")))
    candidates = _unique_existing_paths(candidates)

    excluded_fragments = {
        "scenario_catalog",
        "predicate_definitions",
        "validation",
        "relevance_audit",
        "summary",
        "manifest",
    }

    result = []
    for path in candidates:
        name = path.name.lower()

        if any(fragment in name for fragment in excluded_fragments):
            continue

        # Assertion files normally contain "batch", "assertion", or "part".
        # Keep all remaining files because package versions may rename them.
        result.append(path)

    return result


@lru_cache(maxsize=8)
def load_predicate_definitions(output_dir_string: str) -> pd.DataFrame:
    output_dir = Path(output_dir_string)
    paths = discover_metadata_files(output_dir, "predicate_definitions.csv")

    if not paths:
        raise FileNotFoundError(
            f"No predicate_definitions.csv found under {output_dir}"
        )

    frames = [pd.read_csv(path) for path in paths]
    definitions = pd.concat(frames, ignore_index=True)

    if "predicate_id" in definitions.columns:
        definitions["predicate_id"] = definitions["predicate_id"].astype(str)
        definitions = definitions.drop_duplicates(
            subset=["predicate_id"],
            keep="first",
        )

    return definitions.reset_index(drop=True)


@lru_cache(maxsize=8)
def load_output_scenario_catalog(output_dir_string: str) -> pd.DataFrame:
    output_dir = Path(output_dir_string)
    paths = discover_metadata_files(output_dir, "scenario_catalog.csv")

    if not paths:
        raise FileNotFoundError(
            f"No scenario_catalog.csv found under {output_dir}"
        )

    frames = []

    for path in paths:
        frame = pd.read_csv(path)
        frame["_catalog_source"] = str(path)
        frames.append(frame)

    catalog = pd.concat(frames, ignore_index=True)

    token_column = None
    for candidate in ("token", "scenario_token"):
        if candidate in catalog.columns:
            token_column = candidate
            break

    if token_column is None:
        raise KeyError(
            "The discovered scenario catalogs contain neither 'token' "
            "nor 'scenario_token'."
        )

    if token_column != "token":
        catalog = catalog.rename(columns={token_column: "token"})

    catalog["token"] = catalog["token"].astype(str)
    catalog = catalog.drop_duplicates(subset=["token"], keep="first")

    # Build one stable notebook index independent of worker-local indices.
    if "index" in catalog.columns:
        catalog = catalog.rename(columns={"index": "extractor_index"})

    catalog = catalog.reset_index(drop=True)
    catalog.insert(0, "index", np.arange(len(catalog), dtype=int))

    return catalog


@lru_cache(maxsize=8)
def parquet_inventory(output_dir_string: str) -> pd.DataFrame:
    output_dir = Path(output_dir_string)
    paths = discover_parquet_parts(output_dir)

    rows = []
    for path in paths:
        worker_name = None
        for parent in path.parents:
            if parent.name.startswith("worker_"):
                worker_name = parent.name
                break

        rows.append(
            {
                "worker": worker_name or "single_process",
                "path": str(path),
                "filename": path.name,
                "size_mb": path.stat().st_size / (1024 ** 2),
            }
        )

    return pd.DataFrame(rows)


def _normalise_scenario_token(value) -> str:
    """Normalise Parquet/catalog scenario-token representations."""
    if value is None:
        return ""
    if isinstance(value, (bytes, bytearray)):
        try:
            value = value.decode("utf-8")
        except Exception:
            value = str(value)
    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass
    return str(value).strip()


@lru_cache(maxsize=8)
def parquet_scenario_index(output_dir_string: str) -> pd.DataFrame:
    """
    Build an index from the scenario tokens physically present in assertion
    Parquet files. This prevents a stale or mismatched scenario_catalog.csv
    from silently producing an empty assertion table.
    """
    output_dir = Path(output_dir_string)
    parquet_paths = discover_parquet_parts(output_dir)
    rows = []
    scenario_candidates = (
        "prov_scenario_token",
        "scenario_token",
        "provenance_scenario_token",
    )

    for path in parquet_paths:
        scenario_column = None
        token_frame = None

        # Read only one candidate column. This is inexpensive even for large
        # assertion outputs and avoids loading all assertion columns merely to
        # discover which scenario is stored in a part.
        for candidate in scenario_candidates:
            try:
                candidate_frame = pd.read_parquet(path, columns=[candidate])
            except Exception:
                continue
            if candidate in candidate_frame.columns:
                scenario_column = candidate
                token_frame = candidate_frame
                break

        if scenario_column is None or token_frame is None:
            rows.append({
                "path": str(path),
                "scenario_column": None,
                "scenario_token": None,
                "row_count": 0,
                "status": "missing_scenario_column",
            })
            continue

        normalised = token_frame[scenario_column].map(_normalise_scenario_token)
        tokens = sorted({value for value in normalised if value})
        if not tokens:
            rows.append({
                "path": str(path),
                "scenario_column": scenario_column,
                "scenario_token": None,
                "row_count": int(len(token_frame)),
                "status": "empty_scenario_token",
            })
            continue

        for token in tokens:
            rows.append({
                "path": str(path),
                "scenario_column": scenario_column,
                "scenario_token": token,
                "row_count": int((normalised == token).sum()),
                "status": "ok",
            })

    return pd.DataFrame(
        rows,
        columns=[
            "path", "scenario_column", "scenario_token",
            "row_count", "status",
        ],
    )


@lru_cache(maxsize=32)
def load_assertions_for_scenario(
    output_dir_string: str,
    scenario_token: str,
) -> pd.DataFrame:
    """
    Load assertions using the scenario tokens actually stored in Parquet.

    The previous implementation first used Parquet predicate pushdown and
    silently returned a columnless empty DataFrame when the catalog token and
    physical Parquet contents disagreed. This implementation verifies the
    physical token index first, reads only matching parts, and raises a useful
    diagnostic instead of hiding the mismatch.
    """
    output_dir = Path(output_dir_string)
    scenario_token = _normalise_scenario_token(scenario_token)
    index = parquet_scenario_index(str(output_dir))

    if index.empty:
        raise FileNotFoundError(
            f"No assertion Parquet files found under {output_dir}. "
            "Expected output_dir/batches/*.parquet or "
            "output_dir/workers/worker_*/batches/*.parquet."
        )

    valid_index = index[index["status"].astype(str) == "ok"].copy()
    available_tokens = sorted(
        set(valid_index["scenario_token"].dropna().map(_normalise_scenario_token))
    )
    matching = valid_index[
        valid_index["scenario_token"].map(_normalise_scenario_token)
        == scenario_token
    ].copy()

    if matching.empty:
        preview = available_tokens[:20]
        message = (
            "The selected scenario exists in scenario_catalog.csv but not in "
            "the assertion Parquet files. "
            f"Selected token: {scenario_token}. "
            f"Output directory: {output_dir}. "
            f"Parquet tokens ({len(available_tokens)}): {preview}. "
            "This normally means that the output directory contains files "
            "from different or incomplete runs. Generate into a fresh output "
            "name, or remove the stale target output directory before rerunning."
        )
        raise RuntimeError(message)

    frames = []
    for path_string in matching["path"].drop_duplicates().astype(str):
        path = Path(path_string)
        try:
            frame = pd.read_parquet(path)
        except Exception as exc:
            print(f"Skipping unreadable Parquet file {path}: {exc}")
            continue

        scenario_column = None
        for candidate in (
            "prov_scenario_token",
            "scenario_token",
            "provenance_scenario_token",
        ):
            if candidate in frame.columns:
                scenario_column = candidate
                break

        if scenario_column is None:
            continue

        token_values = frame[scenario_column].map(_normalise_scenario_token)
        frame = frame[token_values == scenario_token].copy()
        if not frame.empty:
            frame[scenario_column] = token_values[token_values == scenario_token]
            frame["_source_parquet"] = str(path)
            frames.append(frame)

    if not frames:
        raise RuntimeError(
            f"Parquet index found scenario {scenario_token}, but no assertion "
            "rows could be read. Check the Parquet engine and file integrity."
        )

    result = pd.concat(frames, ignore_index=True)

    # Protect against accidental duplicate discovery through recursive paths.
    dedup_columns = [
        column
        for column in (
            "assertion_id",
            "predicate_id",
            "subject_id",
            "object_id",
            "valid_time_us",
        )
        if column in result.columns
    ]
    if dedup_columns:
        result = result.drop_duplicates(subset=dedup_columns, keep="first")

    return result.reset_index(drop=True)


def _interval_predicate_ids_for_frame_filter(
    definitions=None,
):
    """Return predicates whose displayed frame is the interval endpoint.

    This includes every definition with ``temporal_scope == interval`` such as
    ``np:follows`` as well as temporal-category predicates retained for backward
    compatibility with older definition files.
    """
    if definitions is None:
        definitions = globals().get("DEFINITIONS")

    if (
        definitions is None
        or getattr(definitions, "empty", True)
        or "predicate_id" not in definitions.columns
    ):
        return set()

    mask = pd.Series(False, index=definitions.index, dtype=bool)
    if "temporal_scope" in definitions.columns:
        mask |= (
            definitions["temporal_scope"]
            .fillna("")
            .astype(str)
            .str.lower()
            .eq("interval")
        )
    if "category" in definitions.columns:
        mask |= (
            definitions["category"]
            .fillna("")
            .astype(str)
            .str.lower()
            .eq("temporal")
        )

    return set(definitions.loc[mask, "predicate_id"].astype(str))


def _temporal_evidence_dict(value):
    if isinstance(value, dict):
        return value

    if value is None:
        return {}

    try:
        if pd.isna(value):
            return {}
    except (TypeError, ValueError):
        pass

    if isinstance(value, str):
        text = value.strip()

        if not text:
            return {}

        try:
            parsed = json.loads(text)
            return parsed if isinstance(parsed, dict) else {}
        except (json.JSONDecodeError, TypeError):
            return {}

    return {}


def _timestamp_from_temporal_identifier(value):
    """
    Extract a nuPlan microsecond timestamp embedded in an entity or pair ID.
    """
    import re

    text = str(value or "").strip()

    if not text:
        return None

    for candidate in re.findall(
        r"(?<!\d)(\d{13,17})(?!\d)",
        text,
    ):
        try:
            return int(candidate)
        except (TypeError, ValueError):
            continue

    return None


def temporal_assertion_current_timestamp_us(row):
    """
    Resolve the frame at which a temporal assertion was emitted.

    Temporal assertions represent intervals:
      valid_time_us = interval/streak start
      end_time_us   = current frame / interval endpoint

    Therefore end_time_us is the correct primary key for selecting one frame.
    Evidence and entity IDs are compatibility fallbacks for older outputs.
    """
    end_time = pd.to_numeric(
        row.get("end_time_us"),
        errors="coerce",
    )

    if not pd.isna(end_time):
        return int(end_time)

    for column in (
        "prov_timestamp_us",
        "provenance_timestamp_us",
        "provenance_time_us",
    ):
        provenance_time = pd.to_numeric(
            row.get(column),
            errors="coerce",
        )

        if not pd.isna(provenance_time):
            return int(provenance_time)

    evidence = _temporal_evidence_dict(
        row.get("evidence_json")
    )

    for key in (
        "current_timestamp_us",
        "current_time_us",
    ):
        evidence_time = pd.to_numeric(
            evidence.get(key),
            errors="coerce",
        )

        if not pd.isna(evidence_time):
            return int(evidence_time)

    # Prefer explicitly current identifiers.
    for identifier in (
        evidence.get("current_entity_id"),
        evidence.get("current_pair_id"),
    ):
        timestamp = _timestamp_from_temporal_identifier(
            identifier
        )

        if timestamp is not None:
            return timestamp

    raw_subject = str(
        row.get("subject_id") or ""
    )

    raw_object = str(
        row.get("object_id") or ""
    )

    # A pair-state subject is created for the current pair observation.
    if raw_subject.startswith("pair:"):
        timestamp = _timestamp_from_temporal_identifier(
            raw_subject
        )

        if timestamp is not None:
            return timestamp

    # For np:precedes, the object is the current entity and the subject is the
    # previous entity. Check the object before the subject.
    timestamp = _timestamp_from_temporal_identifier(
        raw_object
    )

    if timestamp is not None:
        return timestamp

    timestamp = _timestamp_from_temporal_identifier(
        raw_subject
    )

    if timestamp is not None:
        return timestamp

    fallback = pd.to_numeric(
        row.get("valid_time_us"),
        errors="coerce",
    )

    if not pd.isna(fallback):
        return int(fallback)

    return None


def assertions_at_timestamp(
    scenario_assertions: pd.DataFrame,
    timestamp_us: int,
    definitions=None,
) -> pd.DataFrame:
    """
    Select assertions belonging to one displayed frame.

    Instantaneous behavior:
        valid_time_us == selected timestamp

    Interval behavior (including np:follows):
        end_time_us/current timestamp == selected timestamp

    This prevents a newly appearing final agent from collecting all later
    interval assertions whose valid_time_us remains equal to its first frame.
    """
    if scenario_assertions.empty:
        return scenario_assertions.copy()

    selected_timestamp_us = int(timestamp_us)

    valid_times = pd.to_numeric(
        scenario_assertions["valid_time_us"],
        errors="coerce",
    )

    interval_ids = _interval_predicate_ids_for_frame_filter(
        definitions
    )

    if (
        not interval_ids
        or "predicate_id" not in scenario_assertions.columns
    ):
        return scenario_assertions.loc[
            valid_times.eq(selected_timestamp_us)
        ].copy()

    predicate_ids = (
        scenario_assertions["predicate_id"]
        .astype(str)
    )

    interval_mask = predicate_ids.isin(
        interval_ids
    )

    non_temporal_selected = (
        ~interval_mask
        & valid_times.eq(selected_timestamp_us)
    )

    interval_selected = pd.Series(
        False,
        index=scenario_assertions.index,
        dtype=bool,
    )

    if interval_mask.any():
        interval_current_times = (
            scenario_assertions.loc[interval_mask]
            .apply(
                temporal_assertion_current_timestamp_us,
                axis=1,
            )
        )

        interval_selected.loc[
            interval_current_times.index
        ] = pd.to_numeric(
            interval_current_times,
            errors="coerce",
        ).eq(selected_timestamp_us)

    return scenario_assertions.loc[
        non_temporal_selected
        | interval_selected
    ].copy()


if not OUTPUT_DIR.exists():
    raise FileNotFoundError(
        f"Extractor output directory does not exist: {OUTPUT_DIR}"
    )

DEFINITIONS = load_predicate_definitions(str(OUTPUT_DIR))
RAW_OUTPUT_SCENARIO_CATALOG = load_output_scenario_catalog(str(OUTPUT_DIR))
PARQUET_INVENTORY = parquet_inventory(str(OUTPUT_DIR))
PARQUET_SCENARIO_INDEX = parquet_scenario_index(str(OUTPUT_DIR))

parquet_tokens = set(
    PARQUET_SCENARIO_INDEX.loc[
        PARQUET_SCENARIO_INDEX["status"].astype(str) == "ok",
        "scenario_token",
    ].dropna().map(_normalise_scenario_token)
)
OUTPUT_SCENARIO_CATALOG = RAW_OUTPUT_SCENARIO_CATALOG.copy()
OUTPUT_SCENARIO_CATALOG["token"] = OUTPUT_SCENARIO_CATALOG["token"].map(
    _normalise_scenario_token
)
OUTPUT_SCENARIO_CATALOG["has_assertions"] = OUTPUT_SCENARIO_CATALOG["token"].isin(
    parquet_tokens
)
MISSING_CATALOG_SCENARIOS = OUTPUT_SCENARIO_CATALOG[
    ~OUTPUT_SCENARIO_CATALOG["has_assertions"]
].copy()
OUTPUT_SCENARIO_CATALOG = OUTPUT_SCENARIO_CATALOG[
    OUTPUT_SCENARIO_CATALOG["has_assertions"]
].reset_index(drop=True)
OUTPUT_SCENARIO_CATALOG["index"] = np.arange(
    len(OUTPUT_SCENARIO_CATALOG), dtype=int
)

if OUTPUT_SCENARIO_CATALOG.empty:
    raise RuntimeError(
        "No scenario token is shared by scenario_catalog.csv and the assertion "
        "Parquet files. The output directory mixes different or incomplete runs. "
        f"Catalog tokens: {RAW_OUTPUT_SCENARIO_CATALOG['token'].astype(str).tolist()[:20]} | "
        f"Parquet tokens: {sorted(parquet_tokens)[:20]}"
    )

print(f"Definitions: {len(DEFINITIONS):,}")
print(
    "Scenarios backed by assertions:",
    f"{len(OUTPUT_SCENARIO_CATALOG):,}/"
    f"{len(RAW_OUTPUT_SCENARIO_CATALOG):,}",
)
print(f"Assertion Parquet files: {len(PARQUET_INVENTORY):,}")

if not PARQUET_INVENTORY.empty:
    print(
        "Parquet size:",
        f"{PARQUET_INVENTORY['size_mb'].sum():,.2f} MB",
    )
    display(
        PARQUET_INVENTORY
        .groupby("worker", dropna=False)
        .agg(
            parquet_files=("path", "size"),
            size_mb=("size_mb", "sum"),
        )
        .reset_index()
    )

display(OUTPUT_SCENARIO_CATALOG.head(10))

if not MISSING_CATALOG_SCENARIOS.empty:
    print("Catalog scenarios without assertion rows:")
    display(MISSING_CATALOG_SCENARIOS.head(20))

display(PARQUET_SCENARIO_INDEX.head(20))


In [ ]:
# OPTIONAL: VERIFY PARALLEL OUTPUT DISCOVERY
print("Output directory:", OUTPUT_DIR)
print("Workers discovered:", sorted(PARQUET_INVENTORY['worker'].unique()) if not PARQUET_INVENTORY.empty else [])
display(PARQUET_INVENTORY.head(20))


In [ ]:
# ============================================================
# 4. BUILD AND SELECT NUPLAN SCENARIOS
# ============================================================

def _normalize_filter_values(value):
    if value is None:
        return None
    if isinstance(value, (str, int, float)):
        return {str(value)}
    return {str(item) for item in value}


def _scenario_map_name(scenario) -> str:
    for attribute in ("map_name", "_map_name"):
        value = getattr(scenario, attribute, None)
        if value is not None:
            return str(value)
    return ""


def filter_scenarios_from_yaml(
    scenarios: Iterable[Any],
    yaml_path: Optional[Path],
    random_seed: int,
) -> list[Any]:
    """
    Notebook-side approximation of the extractor filter.

    This is retained for fast lookup, but it is not treated as authoritative
    because the extractor may use nuPlan's official ScenarioFilter semantics,
    including details not reproduced here.
    """
    scenarios = list(scenarios)
    if yaml_path is None:
        return scenarios

    with Path(yaml_path).open("r", encoding="utf-8") as handle:
        config = yaml.safe_load(handle) or {}

    scenario_types = _normalize_filter_values(config.get("scenario_types"))
    scenario_tokens = _normalize_filter_values(config.get("scenario_tokens"))
    log_names = _normalize_filter_values(config.get("log_names"))
    map_names = _normalize_filter_values(config.get("map_names"))

    filtered = []
    for scenario in scenarios:
        if scenario_types is not None and str(scenario.scenario_type) not in scenario_types:
            continue
        if scenario_tokens is not None and str(scenario.token) not in scenario_tokens:
            continue
        if log_names is not None and str(scenario.log_name) not in log_names:
            continue
        if map_names is not None and _scenario_map_name(scenario) not in map_names:
            continue
        filtered.append(scenario)

    if bool(config.get("shuffle", False)):
        random.Random(int(random_seed)).shuffle(filtered)

    num_per_type = config.get("num_scenarios_per_type")
    if num_per_type is not None:
        counts = Counter()
        selected = []
        for scenario in filtered:
            scenario_type = str(scenario.scenario_type)
            if counts[scenario_type] < int(num_per_type):
                selected.append(scenario)
                counts[scenario_type] += 1
        filtered = selected

    total_limit = config.get("limit_total_scenarios")
    if total_limit is not None:
        if isinstance(total_limit, float):
            keep = max(1, int(len(filtered) * total_limit))
        else:
            keep = int(total_limit)
        filtered = filtered[:keep]

    return filtered


@lru_cache(maxsize=2)
def build_all_scenarios(
    dataset_root_string: str,
    map_root_string: str,
    map_version: str,
    sample_interval_s: float,
) -> tuple[Any, ...]:
    """
    Build the complete nuPlan mini scenario list.

    This list is the authoritative fallback for resolving a scenario token
    saved by the extractor.
    """
    config = {
        "dataset_root": dataset_root_string,
        "map_root": map_root_string,
        "map_version": map_version,
        "sample_interval_s": float(sample_interval_s),
        "max_scenarios": None,
        "verbose": True,
    }
    return tuple(build_mini_scenarios(config))


@lru_cache(maxsize=4)
def build_filtered_scenarios(
    dataset_root_string: str,
    map_root_string: str,
    map_version: str,
    scenario_filter_yaml_string: str,
    filter_seed: int,
    sample_interval_s: float,
) -> tuple[Any, ...]:
    scenarios = list(
        build_all_scenarios(
            dataset_root_string,
            map_root_string,
            map_version,
            sample_interval_s,
        )
    )

    yaml_path = (
        Path(scenario_filter_yaml_string)
        if scenario_filter_yaml_string
        else None
    )

    scenarios = filter_scenarios_from_yaml(
        scenarios,
        yaml_path=yaml_path,
        random_seed=int(filter_seed),
    )
    return tuple(scenarios)


@lru_cache(maxsize=2)
def build_all_scenario_lookup(
    dataset_root_string: str,
    map_root_string: str,
    map_version: str,
    sample_interval_s: float,
) -> dict[str, Any]:
    scenarios = build_all_scenarios(
        dataset_root_string,
        map_root_string,
        map_version,
        sample_interval_s,
    )
    return {str(scenario.token): scenario for scenario in scenarios}


def get_scenarios(filter_seed: int = 42) -> list[Any]:
    return list(
        build_filtered_scenarios(
            str(DATASET_ROOT),
            str(MAP_ROOT),
            MAP_VERSION,
            str(SCENARIO_FILTER_YAML) if SCENARIO_FILTER_YAML else "",
            int(filter_seed),
            float(SAMPLE_INTERVAL_S),
        )
    )


def resolve_output_scenario_token(
    token: str,
    *,
    filter_seed: int = 42,
):
    """
    Resolve an extractor-output token to the official nuPlan scenario object.

    Resolution order:
      1. Notebook-side filtered list, for speed.
      2. Complete nuPlan mini lookup, which is authoritative by token.

    This avoids false KeyErrors when the extractor and notebook apply the YAML
    filter with slightly different semantics or ordering.
    """
    token = str(token)

    filtered_lookup = {
        str(scenario.token): scenario
        for scenario in get_scenarios(filter_seed=filter_seed)
    }

    scenario = filtered_lookup.get(token)
    if scenario is not None:
        return scenario

    print(
        "Token was not found in the notebook-side filtered list; "
        "falling back to the complete nuPlan mini scenario lookup."
    )

    full_lookup = build_all_scenario_lookup(
        str(DATASET_ROOT),
        str(MAP_ROOT),
        MAP_VERSION,
        float(SAMPLE_INTERVAL_S),
    )

    scenario = full_lookup.get(token)
    if scenario is None:
        raise KeyError(
            f"Scenario token {token} exists in the extractor output catalog, "
            "but it was not found anywhere in the rebuilt nuPlan mini dataset. "
            "Verify DATASET_ROOT, MAP_ROOT, MAP_VERSION, and that the output "
            "was generated from the same nuPlan dataset."
        )

    return scenario


def select_scenario(
    *,
    selector: str = "index",
    value: Any = 0,
    random_seed: int = 42,
    filter_seed: int = 42,
):
    selector = str(selector).strip().lower()
    catalog = OUTPUT_SCENARIO_CATALOG.copy()

    if catalog.empty:
        raise ValueError("The output scenario catalog is empty.")

    if selector == "token":
        token = str(value)

    elif selector == "index":
        index = int(value)
        matching = catalog[catalog["index"].astype(int) == index]

        if matching.empty:
            if not 0 <= index < len(catalog):
                raise IndexError(f"Scenario index {index} is outside the catalog.")
            token = str(catalog.iloc[index]["token"])
        else:
            token = str(matching.iloc[0]["token"])

    elif selector == "random":
        row_position = random.Random(int(random_seed)).randrange(len(catalog))
        token = str(catalog.iloc[row_position]["token"])

    else:
        raise ValueError("selector must be 'token', 'index', or 'random'.")

    matching_catalog_rows = catalog[
        catalog["token"].astype(str) == token
    ]

    if matching_catalog_rows.empty:
        raise KeyError(
            f"Scenario token {token} is not present in the extractor output catalog."
        )

    scenario = resolve_output_scenario_token(
        token,
        filter_seed=int(filter_seed),
    )

    catalog_row = matching_catalog_rows.iloc[0].to_dict()
    return scenario, catalog_row


def show_scenario_catalog(max_rows: int = 200):
    display(OUTPUT_SCENARIO_CATALOG.head(max_rows))


print(
    "Scenario functions are ready. Output tokens are resolved first against "
    "the notebook-side filtered list and then against the complete nuPlan mini list."
)


In [ ]:
# ============================================================
# 5. BUILD AND SELECT FRAMES
# ============================================================

_FRAME_CACHE: dict[str, list[Any]] = {}


def get_scenario_frames(scenario) -> list[Any]:
    token = str(scenario.token)
    if token not in _FRAME_CACHE:
        _FRAME_CACHE[token] = list(iter_frames(scenario))
    return _FRAME_CACHE[token]


def _frame_native_tokens(frame: Any) -> list[str]:
    tokens = []
    for attribute in (
        "token",
        "frame_token",
        "lidar_pc_token",
        "lidar_token",
        "sample_token",
    ):
        value = getattr(frame, attribute, None)
        if value is not None:
            tokens.append(str(value))

    for container_name in ("lidar_pc", "lidar_point_cloud", "sample"):
        container = getattr(frame, container_name, None)
        if container is None:
            continue
        for attribute in ("token", "lidar_pc_token", "sample_token"):
            value = getattr(container, attribute, None)
            if value is not None:
                tokens.append(str(value))

    return list(dict.fromkeys(tokens))


def frame_catalog(scenario) -> pd.DataFrame:
    rows = []
    for frame_index, frame in enumerate(get_scenario_frames(scenario)):
        timestamp_us = int(frame.timestamp_us)
        scenario_token = str(scenario.token)
        native_tokens = _frame_native_tokens(frame)
        rows.append(
            {
                "frame_index": frame_index,
                "timestamp_us": timestamp_us,
                "native_frame_token": native_tokens[0] if native_tokens else None,
                "all_native_tokens": native_tokens,
                "observation_token": f"observation:{scenario_token}:{timestamp_us}",
                "ego_entity_token": f"ego:{scenario_token}:{timestamp_us}",
                "tracked_object_count": len(list(frame.tracked_objects)),
            }
        )
    return pd.DataFrame(rows)


def _frame_token_matches(
    frame: Any,
    scenario_token: str,
    supplied_token: Any,
) -> bool:
    supplied = str(supplied_token)
    timestamp_us = int(frame.timestamp_us)

    accepted = {
        str(timestamp_us),
        f"observation:{scenario_token}:{timestamp_us}",
        f"ego:{scenario_token}:{timestamp_us}",
    }
    accepted.update(_frame_native_tokens(frame))
    return supplied in accepted


def select_frame(
    scenario,
    *,
    selector: str = "index",
    value: Any = 0,
    random_seed: int = 42,
):
    selector = str(selector).strip().lower()
    frames = get_scenario_frames(scenario)
    if not frames:
        raise ValueError("The selected scenario has no frames.")

    if selector == "index":
        frame_index = int(value)
        if not 0 <= frame_index < len(frames):
            raise IndexError(
                f"Frame index {frame_index} is outside [0, {len(frames) - 1}]."
            )
    elif selector == "timestamp":
        target = int(value)
        matches = [
            index
            for index, frame in enumerate(frames)
            if int(frame.timestamp_us) == target
        ]
        if not matches:
            raise KeyError(f"Timestamp {target} was not found.")
        frame_index = matches[0]
    elif selector == "token":
        scenario_token = str(scenario.token)
        matches = [
            index
            for index, frame in enumerate(frames)
            if _frame_token_matches(frame, scenario_token, value)
        ]
        if not matches:
            raise KeyError(
                f"Frame token {value!r} was not found. "
                "Display frame_catalog(scenario) to see accepted identifiers."
            )
        frame_index = matches[0]
    elif selector == "random":
        frame_index = random.Random(int(random_seed)).randrange(len(frames))
    else:
        raise ValueError(
            "selector must be 'index', 'timestamp', 'token', or 'random'."
        )

    return frames[frame_index], frame_index


print("Frame functions are ready.")


In [ ]:
# ============================================================
# 6. ENTITY GEOMETRY HELPERS
# ============================================================

def get_geometric_center_pose(entity: Any) -> tuple[float, float, float]:
    car_footprint = getattr(entity, "car_footprint", None)
    if car_footprint is not None:
        oriented_box = getattr(car_footprint, "oriented_box", None)
        center = getattr(oriented_box, "center", None)
        if center is not None:
            return float(center.x), float(center.y), float(center.heading)

    center = getattr(entity, "center", None)
    if center is not None:
        return float(center.x), float(center.y), float(center.heading)

    box = getattr(entity, "box", None)
    center = getattr(box, "center", None)
    if center is not None:
        return float(center.x), float(center.y), float(center.heading)

    raise AttributeError(f"No geometric-center pose for {type(entity)}")


def get_dimensions(entity: Any) -> tuple[Optional[float], Optional[float]]:
    candidates = [
        getattr(getattr(entity, "car_footprint", None), "oriented_box", None),
        entity,
        getattr(entity, "box", None),
    ]
    for candidate in candidates:
        if candidate is None:
            continue
        length = getattr(candidate, "length", None)
        width = getattr(candidate, "width", None)
        if length is not None and width is not None:
            return float(length), float(width)
    return None, None


def get_track_token(obj: Any) -> str:
    metadata = getattr(obj, "metadata", None)
    token = getattr(metadata, "track_token", None)
    if token is None:
        token = getattr(obj, "track_token", None)
    if token is None:
        x, y, heading = get_geometric_center_pose(obj)
        token = f"unknown_{x:.3f}_{y:.3f}_{heading:.3f}"
    return str(token)


def get_agent_type(obj: Any) -> str:
    value = getattr(obj, "tracked_object_type", "UNKNOWN")
    return str(value).split(".")[-1]


def entity_id_to_track_token(entity_id: str) -> str:
    entity_id = str(entity_id)
    if entity_id.startswith("ego:"):
        return "ego"
    if entity_id.startswith("agent:"):
        parts = entity_id.split(":")
        if len(parts) >= 3:
            return ":".join(parts[1:-1])
    return entity_id


def short_token(token: Any, max_length: int = 14) -> str:
    text = str(token)
    if text == "ego":
        return "EGO"
    if len(text) <= max_length:
        return text
    return f"{text[:6]}…{text[-5:]}"


def oriented_rectangle_corners(
    x: float,
    y: float,
    heading: float,
    length: Optional[float],
    width: Optional[float],
) -> np.ndarray:
    length = float(length) if length is not None else 4.5
    width = float(width) if width is not None else 2.0

    local = np.array(
        [
            [length / 2, width / 2],
            [length / 2, -width / 2],
            [-length / 2, -width / 2],
            [-length / 2, width / 2],
        ],
        dtype=float,
    )
    rotation = np.array(
        [
            [math.cos(heading), -math.sin(heading)],
            [math.sin(heading), math.cos(heading)],
        ]
    )
    return local @ rotation.T + np.array([x, y])


def build_entity_table(frame, scenario_token: str) -> pd.DataFrame:
    timestamp_us = int(frame.timestamp_us)
    rows = []

    ego_x, ego_y, ego_heading = get_geometric_center_pose(frame.ego_state)
    ego_length, ego_width = get_dimensions(frame.ego_state)
    rows.append(
        {
            "entity_id": f"ego:{scenario_token}:{timestamp_us}",
            "track_token": "ego",
            "agent_type": "EGO",
            "x": ego_x,
            "y": ego_y,
            "heading": ego_heading,
            "length": ego_length,
            "width": ego_width,
            "is_ego": True,
            "source_object": frame.ego_state,
        }
    )

    for obj in frame.tracked_objects:
        token = get_track_token(obj)
        x, y, heading = get_geometric_center_pose(obj)
        length, width = get_dimensions(obj)
        rows.append(
            {
                "entity_id": f"agent:{token}:{timestamp_us}",
                "track_token": token,
                "agent_type": get_agent_type(obj),
                "x": x,
                "y": y,
                "heading": heading,
                "length": length,
                "width": width,
                "is_ego": False,
                "source_object": obj,
            }
        )

    return pd.DataFrame(rows)


print("Entity helpers are ready.")


# ============================================================
# STABLE NUMERIC AGENT LABELS WITHIN EACH SCENARIO
# ============================================================

_SCENE_AGENT_NUMBER_CACHE: dict[str, dict[str, int]] = {}


def scene_agent_number_map(scenario) -> dict[str, int]:
    """Assign stable A1, A2, ... IDs by first appearance in the scenario.

    A tracked object's nuPlan track token is stable over frames. We scan the
    scenario in temporal order once and assign a number at first appearance.
    Therefore the same physical tracked object keeps the same displayed number
    in every inspected frame of the same scenario.
    """
    scenario_token = str(scenario.token)
    if scenario_token in _SCENE_AGENT_NUMBER_CACHE:
        return _SCENE_AGENT_NUMBER_CACHE[scenario_token]

    mapping: dict[str, int] = {}
    next_number = 1
    for scene_frame in get_scenario_frames(scenario):
        for obj in scene_frame.tracked_objects:
            token = get_track_token(obj)
            if token not in mapping:
                mapping[token] = next_number
                next_number += 1

    _SCENE_AGENT_NUMBER_CACHE[scenario_token] = mapping
    return mapping


def add_stable_display_ids(entity_table: pd.DataFrame, scenario) -> pd.DataFrame:
    result = entity_table.copy()
    number_map = scene_agent_number_map(scenario)
    result["display_id"] = result.apply(
        lambda row: "EGO" if bool(row["is_ego"]) else f"A{number_map.get(str(row['track_token']), '?')}",
        axis=1,
    )
    return result


print("Stable scenario-level numeric agent labels are ready.")


In [ ]:
# ============================================================
# 7. GENERAL SEMANTIC FILTERING AND EDGE RECOVERY
# ============================================================

ALL_SEMANTIC_CATEGORIES = [
    "structure", "scenario", "position_values", "geometry", "motion",
    "motion_state", "temporal", "map", "route", "pairwise",
    "relevance", "spatial", "heading", "interaction", "risk",
    "traffic_light", "visibility", "future_observation", "maneuver",
    "intent", "other",
]

ALL_SEMANTIC_RELATIONS = [
    "ego_to_agent", "agent_to_ego", "agent_to_agent",
    "ego_to_structure", "agent_to_structure",
    "structure_to_structure", "structure_to_entity", "other",
]


def _predicate_ids_for_category(definitions: pd.DataFrame, category: str) -> set[str]:
    if (
        definitions is None
        or definitions.empty
        or "category" not in definitions.columns
        or "predicate_id" not in definitions.columns
    ):
        return set()
    return set(definitions.loc[
        definitions["category"].astype(str) == str(category), "predicate_id"
    ].astype(str))


def _pair_tag_from_pair_id(pair_id: str) -> str:
    parts = str(pair_id).split(":")
    return parts[-2] if len(parts) >= 2 else "other"


def _evidence_mapping(value) -> dict:
    if isinstance(value, dict):
        return value
    if value is None:
        return {}
    try:
        parsed = json.loads(value)
    except Exception:
        return {}
    return parsed if isinstance(parsed, dict) else {}


def pair_member_table(frame_assertions: pd.DataFrame) -> pd.DataFrame:
    """
    Resolve a pair-state node to its directed subject and object.

    New v9.5.3 temporal output retains np:pairSubject/np:pairObject through the
    temporal -> pairwise category dependency. For older temporal-only outputs,
    recover the same mapping from evidence_json, where the generator already
    stored subject_id and object_id independently.
    """
    output_columns = ["pair_id", "subject_id", "object_id", "pair_tag"]
    required = {"predicate_id", "subject_id", "object_id"}
    if (
        frame_assertions is None
        or frame_assertions.empty
        or not required.issubset(frame_assertions.columns)
    ):
        return pd.DataFrame(columns=output_columns)

    pair_subject = frame_assertions[
        frame_assertions["predicate_id"].astype(str) == "np:pairSubject"
    ][["subject_id", "object_id"]].rename(
        columns={"subject_id": "pair_id", "object_id": "subject_id"}
    )
    pair_object = frame_assertions[
        frame_assertions["predicate_id"].astype(str) == "np:pairObject"
    ][["subject_id", "object_id"]].rename(
        columns={"subject_id": "pair_id", "object_id": "object_id"}
    )
    pairs = pair_subject.merge(pair_object, on="pair_id", how="outer")

    # Backward-compatible recovery from temporal evidence.
    if "evidence_json" in frame_assertions.columns:
        evidence_rows = []
        possible_pairs = frame_assertions[
            frame_assertions["subject_id"].astype(str).str.startswith("pair:")
        ]
        for row in possible_pairs[["subject_id", "evidence_json"]].itertuples(index=False):
            evidence = _evidence_mapping(row.evidence_json)
            subject_id = evidence.get("subject_id")
            object_id = evidence.get("object_id")
            if subject_id and object_id:
                evidence_rows.append({
                    "pair_id": str(row.subject_id),
                    "subject_id": str(subject_id),
                    "object_id": str(object_id),
                })
        if evidence_rows:
            evidence_pairs = pd.DataFrame(evidence_rows).drop_duplicates("pair_id")
            if pairs.empty:
                pairs = evidence_pairs
            else:
                pairs = pd.concat([pairs, evidence_pairs], ignore_index=True)
                pairs = pairs.drop_duplicates("pair_id", keep="first")

    if pairs.empty:
        return pd.DataFrame(columns=output_columns)

    pairs = pairs.dropna(subset=["pair_id", "subject_id", "object_id"]).copy()
    pairs["pair_id"] = pairs["pair_id"].astype(str)
    pairs["subject_id"] = pairs["subject_id"].astype(str)
    pairs["object_id"] = pairs["object_id"].astype(str)
    pairs["pair_tag"] = pairs["pair_id"].map(_pair_tag_from_pair_id)
    return pairs[output_columns].drop_duplicates("pair_id").reset_index(drop=True)

def _entity_kind(entity_id: Any) -> str:
    value = str(entity_id)
    if value.startswith("ego:"):
        return "ego"
    if value.startswith("agent:"):
        return "agent"
    structure_prefixes = (
        "lane:", "lane_connector:", "roadblock:", "roadblock_connector:",
        "intersection:", "crosswalk:", "stop_line:", "traffic_light:",
        "route:", "map:", "structure:", "walkway:", "carpark:",
    )
    if value.startswith(structure_prefixes):
        return "structure"
    return "other"


def classify_semantic_relation(subject_id: Any, object_id: Any, pair_tag: str = "") -> str:
    tag = str(pair_tag)
    if tag in ALL_SEMANTIC_RELATIONS:
        return tag
    subject_kind = _entity_kind(subject_id)
    object_kind = _entity_kind(object_id)
    if subject_kind == "ego" and object_kind == "agent":
        return "ego_to_agent"
    if subject_kind == "agent" and object_kind == "ego":
        return "agent_to_ego"
    if subject_kind == "agent" and object_kind == "agent":
        return "agent_to_agent"
    if subject_kind == "ego" and object_kind == "structure":
        return "ego_to_structure"
    if subject_kind == "agent" and object_kind == "structure":
        return "agent_to_structure"
    if subject_kind == "structure" and object_kind == "structure":
        return "structure_to_structure"
    if subject_kind == "structure" and object_kind in {"ego", "agent"}:
        return "structure_to_entity"
    return "other"


def enabled_names(switches: dict[str, bool]) -> set[str]:
    return {str(name) for name, enabled in switches.items() if bool(enabled)}


def normalize_semantic_predicate_selection(selection):
    """
    Normalize the exact-predicate visualization filter.

    Accepted forms:
      - "all", "*", or None -> do not filter predicates;
      - "follows" -> {"np:follows"};
      - "np:follows" -> {"np:follows"};
      - ["follows", "crossesInFrontOf"] -> both predicates;
      - "follows,crossesInFrontOf" -> both predicates.
    """
    if selection is None:
        return None

    if isinstance(selection, str):
        text = selection.strip()
        if not text or text.lower() in {"all", "*"}:
            return None
        values = [part.strip() for part in text.split(",") if part.strip()]
    else:
        try:
            values = list(selection)
        except TypeError:
            values = [selection]

    normalized = []
    for value in values:
        text = str(value).strip()
        if not text:
            continue
        if text.lower() in {"all", "*"}:
            return None
        if not text.startswith("np:"):
            text = f"np:{text}"
        if text not in normalized:
            normalized.append(text)

    if not normalized:
        raise ValueError(
            "SEMANTIC_PREDICATES must be 'all' or contain at least one predicate."
        )

    return tuple(normalized)


def semantic_predicate_selection_label(selection) -> str:
    normalized = normalize_semantic_predicate_selection(selection)
    if normalized is None:
        return "all"
    return ",".join(predicate.replace("np:", "", 1) for predicate in normalized)


def semantic_predicate_is_selected(predicate_id, selection) -> bool:
    normalized = normalize_semantic_predicate_selection(selection)
    if normalized is None:
        return True
    predicate_id = str(predicate_id).strip()
    if predicate_id and not predicate_id.startswith("np:"):
        predicate_id = f"np:{predicate_id}"
    return predicate_id in set(normalized)


def validate_semantic_predicate_selection(
    selection,
    definitions,
    active_categories=None,
):
    """Fail early for a typo or a predicate outside the enabled family."""
    normalized = normalize_semantic_predicate_selection(selection)
    if normalized is None:
        return None

    if (
        definitions is None
        or definitions.empty
        or "predicate_id" not in definitions.columns
    ):
        return normalized

    metadata_columns = ["predicate_id"]
    if "category" in definitions.columns:
        metadata_columns.append("category")
    metadata = definitions[metadata_columns].drop_duplicates("predicate_id")
    available = set(metadata["predicate_id"].astype(str))

    unknown = [predicate for predicate in normalized if predicate not in available]
    if unknown:
        raise ValueError(
            "Unknown SEMANTIC_PREDICATES: "
            + ", ".join(unknown)
        )

    if active_categories and "category" in metadata.columns:
        active = {str(value).strip().lower() for value in active_categories}
        category_by_predicate = dict(zip(
            metadata["predicate_id"].astype(str),
            metadata["category"].astype(str).str.strip().str.lower(),
        ))
        outside = [
            predicate
            for predicate in normalized
            if category_by_predicate.get(predicate) not in active
        ]
        if outside:
            details = ", ".join(
                f"{predicate} ({category_by_predicate.get(predicate, 'unknown')})"
                for predicate in outside
            )
            raise ValueError(
                "Selected predicate(s) are not in the enabled semantic family: "
                + details
            )

    return normalized


def _semantic_edge_predicate_list(value):
    if isinstance(value, list):
        return [str(item) for item in value]
    if isinstance(value, (tuple, set, np.ndarray)):
        return [str(item) for item in list(value)]
    if value is None:
        return []
    text = str(value).strip()
    if not text:
        return []
    try:
        parsed = json.loads(text)
        if isinstance(parsed, list):
            return [str(item) for item in parsed]
    except Exception:
        pass
    return [text]


def filter_semantic_edges_by_predicates(edge_table, predicate_selection="all"):
    """
    Keep only the selected predicate(s) inside every semantic edge.

    This deliberately trims the `predicates` list as well as dropping edges.
    Therefore an S->O edge containing both follows and overtakes will contain
    only follows when SEMANTIC_PREDICATES=["follows"]. Arrow styling and
    legends then also use only the selected predicate.
    """
    if edge_table is None:
        return pd.DataFrame()

    normalized = normalize_semantic_predicate_selection(predicate_selection)
    if normalized is None or edge_table.empty:
        return edge_table.copy()

    selected = set(normalized)
    rows = []

    for _, row in edge_table.iterrows():
        predicates = _semantic_edge_predicate_list(row.get("predicates", []))
        kept = [predicate for predicate in predicates if predicate in selected]
        if not kept:
            continue

        updated = row.copy()
        updated["predicates"] = kept
        updated["predicate_count"] = len(kept)

        labels = [
            part.strip()
            for part in str(row.get("relation_labels", "")).split(",")
            if part.strip()
        ]
        bases = [predicate.replace("np:", "", 1) for predicate in kept]
        kept_labels = [
            label
            for label in labels
            if any(label == base or label.startswith(base + "=") for base in bases)
        ]
        if not kept_labels:
            kept_labels = bases

        relation_labels = ", ".join(kept_labels)
        updated["relation_labels"] = relation_labels

        if "semantic_statement" in edge_table.columns:
            source = str(row.get("subject_token", row.get("subject_id", "")))
            target = str(row.get("object_token", row.get("object_id", "")))
            updated["semantic_statement"] = (
                f"{short_token(source)} → {short_token(target)}: {relation_labels}"
            )

        rows.append(updated)

    if not rows:
        return edge_table.iloc[0:0].copy()

    return pd.DataFrame(rows).reset_index(drop=True)


def filter_assertions_by_switches(
    assertions: pd.DataFrame,
    definitions: pd.DataFrame,
    category_switches: dict[str, bool],
) -> pd.DataFrame:
    if assertions.empty:
        return assertions.copy()
    active_categories = enabled_names(category_switches)
    metadata = definitions[["predicate_id", "category"]].drop_duplicates("predicate_id")
    result = assertions.merge(metadata, on="predicate_id", how="left", suffixes=("", "_definition"))
    category_column = "category_definition" if "category_definition" in result.columns else "category"
    return result[result[category_column].astype(str).isin(active_categories)].copy()


def semantic_edge_table(
    frame_assertions: pd.DataFrame,
    definitions: pd.DataFrame,
    category_switches: dict[str, bool],
    relation_switches: dict[str, bool],
    pair_assertions=None,
) -> pd.DataFrame:
    """Build graph edges and preserve the stored direction SUBJECT -> OBJECT."""
    edge_columns = [
        "relation_type", "category", "subject_id", "object_id",
        "subject_token", "object_token", "predicates",
        "predicate_count", "relation_labels",
        "semantic_source_token", "semantic_target_token",
        "semantic_statement",
    ]
    if (
        frame_assertions is None
        or frame_assertions.empty
        or "predicate_id" not in frame_assertions.columns
        or "predicate_id" not in definitions.columns
    ):
        return pd.DataFrame(columns=edge_columns)

    active_categories = enabled_names(category_switches)
    active_relations = enabled_names(relation_switches)
    definition_columns = [
        column for column in [
            "predicate_id",
            "category",
            "label",
            "units",
            "value_type",
        ]
        if column in definitions.columns
    ]
    definition_meta = definitions[definition_columns].drop_duplicates("predicate_id")
    rows = frame_assertions.merge(definition_meta, on="predicate_id", how="left")
    rows = rows[rows["category"].astype(str).isin(active_categories)].copy()

    # Preserve positive-only behaviour for Boolean/numeric semantic states,
    # keep every temporal measurement, and retain entity-valued relations such
    # as np:follows. Entity relations normally encode the target in object_id
    # and therefore legitimately store value_json as null.
    if "value_json" in rows.columns:
        is_temporal = rows["category"].astype(str).str.lower().eq("temporal")
        if "value_type" in rows.columns:
            is_entity_relation = (
                rows["value_type"]
                .fillna("")
                .astype(str)
                .str.strip()
                .str.lower()
                .eq("entity")
            )
        else:
            is_entity_relation = pd.Series(
                False,
                index=rows.index,
                dtype=bool,
            )
        rows = rows[
            is_temporal
            | is_entity_relation
            | rows["value_json"].map(_is_positive_value)
        ]

    pair_source = pair_assertions if pair_assertions is not None else frame_assertions
    pairs = pair_member_table(pair_source).set_index("pair_id", drop=False)
    edge_rows = []
    skip_predicates = {"np:pairSubject", "np:pairObject"}

    for row in rows.itertuples(index=False):
        predicate_id = str(row.predicate_id)
        if predicate_id in skip_predicates:
            continue
        raw_subject = str(row.subject_id)
        raw_object = str(row.object_id)
        pair_tag = ""
        subject_id, object_id = raw_subject, raw_object

        if raw_subject in pairs.index:
            pair = pairs.loc[raw_subject]
            if isinstance(pair, pd.DataFrame):
                pair = pair.iloc[0]
            subject_id = str(pair["subject_id"])
            object_id = str(pair["object_id"])
            pair_tag = str(pair["pair_tag"])
        else:
            # Keep only assertions that actually connect two semantic resources.
            if _entity_kind(raw_object) == "other":
                continue

        relation_type = classify_semantic_relation(subject_id, object_id, pair_tag)
        if relation_type not in active_relations:
            continue

        edge_rows.append({
            "pair_id": raw_subject if raw_subject in pairs.index else None,
            "pair_tag": pair_tag,
            "relation_type": relation_type,
            "category": str(row.category),
            "predicate_id": predicate_id,
            "predicate_label": str(getattr(row, "label", predicate_id)),
            "predicate_value": _decoded_assertion_value(
                getattr(row, "value_json", None)
            ),
            "predicate_unit": getattr(row, "units", None),
            "subject_id": subject_id,
            "object_id": object_id,
            "subject_token": entity_id_to_track_token(subject_id),
            "object_token": entity_id_to_track_token(object_id),
        })

    if not edge_rows:
        return pd.DataFrame(columns=edge_columns)

    edges = pd.DataFrame(edge_rows)

    def _edge_label(row):
        predicate_name = str(row["predicate_id"]).replace("np:", "")
        if str(row["category"]).lower() != "temporal":
            return predicate_name
        value = row.get("predicate_value")
        unit = row.get("predicate_unit")
        if value is None or value == "":
            return predicate_name
        unit_text = "" if unit is None or str(unit).lower() == "nan" else f" {unit}"
        return f"{predicate_name}={value}{unit_text}"

    edges["display_predicate"] = edges.apply(_edge_label, axis=1)

    group_columns = [
        "relation_type", "category", "subject_id", "object_id",
        "subject_token", "object_token",
    ]
    predicate_group = (
        edges.groupby(group_columns, dropna=False)["predicate_id"]
        .agg(lambda values: sorted(set(map(str, values))))
        .reset_index(name="predicates")
    )
    label_group = (
        edges.groupby(group_columns, dropna=False)["display_predicate"]
        .agg(lambda values: list(dict.fromkeys(map(str, values))))
        .reset_index(name="_display_labels")
    )
    grouped = predicate_group.merge(label_group, on=group_columns, how="left")
    grouped["predicate_count"] = grouped["predicates"].map(len)
    grouped["relation_labels"] = grouped["_display_labels"].map(
        lambda values: ", ".join(values)
    )
    grouped = grouped.drop(columns=["_display_labels"])
    grouped["semantic_source_token"] = grouped["subject_token"]
    grouped["semantic_target_token"] = grouped["object_token"]
    grouped["semantic_statement"] = grouped.apply(
        lambda row: f"{short_token(row['subject_token'])} → {short_token(row['object_token'])}: {row['relation_labels']}",
        axis=1,
    )
    return grouped


print("General category/relation filtering is ready. Arrows use SUBJECT -> OBJECT.")

In [ ]:
# ============================================================
# 8. MAP DRAWING HELPERS
# ============================================================

_MAP_LAYER_STYLE = {
    "ROADBLOCK": {"facecolor": "#eeeeee", "edgecolor": "#bdbdbd", "alpha": 0.25, "linewidth": 0.8},
    "ROADBLOCK_CONNECTOR": {"facecolor": "#f5f5f5", "edgecolor": "#bdbdbd", "alpha": 0.20, "linewidth": 0.8},
    "LANE": {"facecolor": "#dfe8f3", "edgecolor": "#8aa4c0", "alpha": 0.30, "linewidth": 0.8},
    "LANE_CONNECTOR": {"facecolor": "#e8e0f3", "edgecolor": "#a58abf", "alpha": 0.30, "linewidth": 0.8},
    "INTERSECTION": {"facecolor": "#f4e3c1", "edgecolor": "#c9a55a", "alpha": 0.25, "linewidth": 0.8},
    "CROSSWALK": {"facecolor": "#fff4b8", "edgecolor": "#b59b31", "alpha": 0.35, "linewidth": 0.8},
    "STOP_LINE": {"facecolor": "#f7cccc", "edgecolor": "#b93b3b", "alpha": 0.65, "linewidth": 1.0},
    "WALKWAYS": {"facecolor": "#e5f2df", "edgecolor": "#83a979", "alpha": 0.25, "linewidth": 0.8},
    "CARPARK_AREA": {"facecolor": "#eeeeee", "edgecolor": "#999999", "alpha": 0.20, "linewidth": 0.7},
}


def _available_semantic_layers() -> list[Any]:
    if not NUPLAN_MAP_IMPORTS_AVAILABLE:
        return []

    layer_names = [
        "ROADBLOCK",
        "ROADBLOCK_CONNECTOR",
        "LANE",
        "LANE_CONNECTOR",
        "INTERSECTION",
        "CROSSWALK",
        "STOP_LINE",
        "WALKWAYS",
        "CARPARK_AREA",
    ]
    return [
        getattr(SemanticMapLayer, name)
        for name in layer_names
        if hasattr(SemanticMapLayer, name)
    ]


def _iter_polygon_parts(geometry):
    if geometry is None:
        return
    geom_type = getattr(geometry, "geom_type", "")
    if geom_type == "Polygon":
        yield geometry
    elif geom_type == "MultiPolygon":
        yield from geometry.geoms


def _plot_polygon_geometry(ax, geometry, style, *, boundary_only=False):
    """Draw polygon geometry; lane-like layers can be boundary-only for debugging."""
    for polygon in _iter_polygon_parts(geometry):
        coords = np.asarray(polygon.exterior.coords)
        if boundary_only:
            ax.plot(
                coords[:, 0],
                coords[:, 1],
                color=style["edgecolor"],
                alpha=max(0.78, float(style.get("alpha", 0.3))),
                linewidth=max(1.0, float(style.get("linewidth", 0.8))),
                zorder=1.5,
            )
        else:
            ax.add_patch(
                MplPolygon(
                    coords,
                    closed=True,
                    facecolor=style["facecolor"],
                    edgecolor=style["edgecolor"],
                    alpha=style["alpha"],
                    linewidth=style["linewidth"],
                    zorder=1,
                )
            )


def _plot_linestring_geometry(ax, geometry, **kwargs):
    if geometry is None:
        return
    geom_type = getattr(geometry, "geom_type", "")
    if geom_type == "LineString":
        coords = np.asarray(geometry.coords)
        ax.plot(coords[:, 0], coords[:, 1], **kwargs)
    elif geom_type == "MultiLineString":
        for part in geometry.geoms:
            coords = np.asarray(part.coords)
            ax.plot(coords[:, 0], coords[:, 1], **kwargs)


def _debug_parent_roadblock_id(map_object):
    """Return the native parent roadblock/roadblock-connector ID when available."""
    try:
        value = map_object.get_roadblock_id()
        if value is not None:
            return str(value)
    except Exception:
        pass
    for name in ("roadblock_id", "parent_id", "parent"):
        value = getattr(map_object, name, None)
        if value is None:
            continue
        if hasattr(value, "id"):
            value = getattr(value, "id")
        if value is not None:
            return str(value)
    return None


def _debug_map_label_anchor(geometry):
    """Choose a stable point inside/near the polygon for a map-ID label."""
    if geometry is None:
        return None
    try:
        point = geometry.representative_point()
        return float(point.x), float(point.y)
    except Exception:
        pass
    try:
        centroid = geometry.centroid
        return float(centroid.x), float(centroid.y)
    except Exception:
        return None


def _debug_map_structure_label(layer_name, map_object):
    object_id = str(getattr(map_object, "id", "?"))
    prefixes = {
        "LANE": "L",
        "LANE_CONNECTOR": "LC",
        "ROADBLOCK": "RB",
        "ROADBLOCK_CONNECTOR": "RBC",
    }
    prefix = prefixes.get(str(layer_name).upper())
    if prefix is None:
        return None
    return f"{prefix}:{object_id}"


def plot_local_map(
    ax,
    map_api,
    center_xy: tuple[float, float],
    radius_m: float,
    *,
    show_map_structure_ids=False,
):
    """Draw local map without lane baselines/centerlines.

    LANE and LANE_CONNECTOR are rendered as polygon boundaries only.  When
    show_map_structure_ids=True, native L/LC/RB/RBC IDs are printed directly
    on the map.  No synthetic corridor IDs are invented here.
    """
    if not NUPLAN_MAP_IMPORTS_AVAILABLE:
        ax.text(
            0.5,
            0.5,
            "nuPlan map imports unavailable",
            transform=ax.transAxes,
            ha="center",
            va="center",
        )
        return

    center = Point2D(float(center_xy[0]), float(center_xy[1]))

    for layer in _available_semantic_layers():
        layer_name = getattr(layer, "name", str(layer).split(".")[-1])
        style = _MAP_LAYER_STYLE.get(
            layer_name,
            {"facecolor": "#eeeeee", "edgecolor": "#aaaaaa", "alpha": 0.2, "linewidth": 0.6},
        )

        try:
            result = map_api.get_proximal_map_objects(
                center,
                float(radius_m),
                [layer],
            )
            objects = result.get(layer, [])
        except Exception:
            continue

        lane_like = str(layer_name).upper() in {"LANE", "LANE_CONNECTOR"}

        for map_object in objects:
            polygon = getattr(map_object, "polygon", None)
            _plot_polygon_geometry(
                ax,
                polygon,
                style,
                boundary_only=lane_like,
            )

            # Intentionally DO NOT draw baseline_path/centerline.  For lane
            # debugging the two physical polygon boundaries are much clearer.

            if show_map_structure_ids:
                label = _debug_map_structure_label(layer_name, map_object)
                anchor = _debug_map_label_anchor(polygon)
                if label and anchor is not None:
                    ax.text(
                        anchor[0],
                        anchor[1],
                        label,
                        fontsize=5.6 if lane_like else 5.0,
                        color=style["edgecolor"],
                        ha="center",
                        va="center",
                        zorder=2.8,
                        clip_on=True,
                        bbox={
                            "boxstyle": "round,pad=0.10",
                            "facecolor": "white",
                            "edgecolor": style["edgecolor"],
                            "alpha": 0.72,
                            "linewidth": 0.45,
                        },
                    )

print("Map helpers are ready.")


In [ ]:
# ============================================================
# 9. CONFIGURABLE AGENT SELECTION + STABLE EDGE PREPARATION
# ============================================================

# Agent-type display controls.
# Keys are normalized to these canonical names.
AGENT_TYPE_COLORS = {
    "vehicle": "#2e7d32",
    "car": "#2e7d32",
    "truck": "#ef6c00",
    "bus": "#6a1b9a",
    "trailer": "#8d6e63",
    "construction_vehicle": "#f9a825",
    "pedestrian": "#d81b60",
    "bicycle": "#00838f",
    "motorcycle": "#3949ab",
    "traffic_cone": "#fb8c00",
    "barrier": "#546e7a",
    "czone_sign": "#7b1fa2",
    "generic_object": "#757575",
    "unknown": "#9e9e9e",
}

AGENT_TYPE_LABELS = {
    "vehicle": "Vehicle",
    "car": "Car",
    "truck": "Truck",
    "bus": "Bus",
    "trailer": "Trailer",
    "construction_vehicle": "Construction vehicle",
    "pedestrian": "Pedestrian",
    "bicycle": "Bicycle",
    "motorcycle": "Motorcycle",
    "traffic_cone": "Traffic cone",
    "barrier": "Barrier",
    "czone_sign": "Construction-zone sign",
    "generic_object": "Generic object",
    "unknown": "Unknown",
}


# Predicate-specific semantic arrow styles. Overtake takes visual priority if
# a Case 1 frame also contains np:follows for the same directed pair.
SEMANTIC_PREDICATE_ARROW_STYLES = {
    "np:follows": {"color": "#1565c0", "linewidth": 2.2, "label": "Follows"},
    "np:overtakes": {"color": "#e53935", "linewidth": 3.2, "label": "Overtakes"},
    "np:mergesInFrontOf": {"color": "#fb8c00", "linewidth": 2.8, "label": "Merge"},
    "np:mergesBehind": {"color": "#fb8c00", "linewidth": 2.8, "label": "Merge"},
    "np:crossesInFrontOf": {"color": "#00897b", "linewidth": 3.0, "label": "Crosses in front"},
    "np:yieldsTo": {"color": "#c79200", "linewidth": 3.2, "label": "Yields to"},
}
DEFAULT_SEMANTIC_ARROW_STYLE = {
    "color": "#757575",
    "linewidth": 1.8,
    "label": "Other semantic relation",
}
LANE_CHANGE_ARROW_STYLE = {
    "color": "#6a1b9a",
    "linewidth": 4.0,
    "label": "Lane change",
}


def semantic_edge_arrow_style(edge):
    predicates = {
        str(value)
        for value in _excel_predicate_list(getattr(edge, "predicates", []))
    }
    if "np:overtakes" in predicates:
        return dict(SEMANTIC_PREDICATE_ARROW_STYLES["np:overtakes"])
    if "np:yieldsTo" in predicates:
        return dict(SEMANTIC_PREDICATE_ARROW_STYLES["np:yieldsTo"])
    if "np:crossesInFrontOf" in predicates:
        return dict(SEMANTIC_PREDICATE_ARROW_STYLES["np:crossesInFrontOf"])
    if "np:mergesInFrontOf" in predicates:
        return dict(SEMANTIC_PREDICATE_ARROW_STYLES["np:mergesInFrontOf"])
    if "np:mergesBehind" in predicates:
        return dict(SEMANTIC_PREDICATE_ARROW_STYLES["np:mergesBehind"])
    if "np:follows" in predicates:
        return dict(SEMANTIC_PREDICATE_ARROW_STYLES["np:follows"])
    return dict(DEFAULT_SEMANTIC_ARROW_STYLE)


def normalize_agent_type(value):
    """
    Normalize nuPlan tracked-object labels.

    Important:
    nuPlan commonly exposes all road vehicles as VEHICLE, without reliably
    distinguishing car, truck, and bus. In that case the canonical type is
    'vehicle'. More specific names are retained only when present.
    """
    text = str(value or "unknown").strip().lower()
    text = text.replace("-", "_").replace(" ", "_")
    text = text.replace("/", "_").replace(".", "_")

    aliases = {
        "pedestrain": "pedestrian",
        "pedistrian": "pedestrian",
        "ped": "pedestrian",
        "person": "pedestrian",
        "human_pedestrian": "pedestrian",
        "cyclist": "bicycle",
        "bike": "bicycle",
        "bicyclist": "bicycle",
        "motorbike": "motorcycle",
        "construction": "construction_vehicle",
        "constructionvehicle": "construction_vehicle",
        "trafficcone": "traffic_cone",
        "traffic_cone": "traffic_cone",
        "czonesign": "czone_sign",
        "czone_sign": "czone_sign",
        "genericobject": "generic_object",
        "generic_object": "generic_object",
        "trackedobjecttype_vehicle": "vehicle",
        "trackedobjecttype_pedestrian": "pedestrian",
        "trackedobjecttype_bicycle": "bicycle",
        "trackedobjecttype_traffic_cone": "traffic_cone",
        "trackedobjecttype_barrier": "barrier",
        "trackedobjecttype_czone_sign": "czone_sign",
        "trackedobjecttype_generic_object": "generic_object",
    }

    text = aliases.get(text, text)

    # Preserve specific vehicle subclasses only when the source label has them.
    for candidate in (
        "construction_vehicle",
        "traffic_cone",
        "generic_object",
        "czone_sign",
        "pedestrian",
        "motorcycle",
        "bicycle",
        "trailer",
        "truck",
        "bus",
        "car",
        "barrier",
        "vehicle",
    ):
        if candidate in text:
            return candidate

    return "unknown"


def show_available_agent_types(entity_table):
    """Display the raw and normalized classes present in the selected frame."""
    table = entity_table.loc[
        ~entity_table["is_ego"].astype(bool),
        ["agent_type"],
    ].copy()

    if table.empty:
        print("No tracked agents are present in this frame.")
        return table

    table["raw_agent_type"] = table["agent_type"].astype(str)
    table["normalized_agent_type"] = table["agent_type"].map(
        normalize_agent_type
    )

    counts = (
        table.groupby(
            ["raw_agent_type", "normalized_agent_type"],
            dropna=False,
        )
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
        .reset_index(drop=True)
    )
    display(counts)
    return counts


def enabled_agent_type_names(agent_type_switches):
    if not agent_type_switches:
        return set(AGENT_TYPE_COLORS)

    return {
        normalize_agent_type(name)
        for name, enabled in agent_type_switches.items()
        if bool(enabled)
    }


def filter_entities_by_agent_type(entity_table, agent_type_switches):
    enabled_types = enabled_agent_type_names(agent_type_switches)
    table = entity_table.copy()
    table["normalized_agent_type"] = table["agent_type"].map(
        normalize_agent_type
    )

    return table[
        table["is_ego"].astype(bool)
        | table["normalized_agent_type"].isin(enabled_types)
    ].copy()


def filter_edges_by_enabled_entity_tokens(
    semantic_edges,
    enabled_entity_table,
):
    if semantic_edges is None or semantic_edges.empty:
        return semantic_edges.copy()

    enabled_tokens = set(
        enabled_entity_table["track_token"].astype(str)
    )
    result = semantic_edges.copy()

    subject_ok = (
        result["subject_token"].isna()
        | result["subject_token"].astype(str).isin(enabled_tokens)
    )
    object_ok = (
        result["object_token"].isna()
        | result["object_token"].astype(str).isin(enabled_tokens)
    )

    return result[subject_ok & object_ok].copy()


def agent_type_color(value):
    return AGENT_TYPE_COLORS.get(
        normalize_agent_type(value),
        AGENT_TYPE_COLORS["unknown"],
    )


def add_agent_type_legend(ax, entity_table, agent_type_switches):
    enabled_types = enabled_agent_type_names(agent_type_switches)
    present_types = {
        normalize_agent_type(value)
        for value in entity_table.loc[
            ~entity_table["is_ego"].astype(bool),
            "agent_type",
        ].tolist()
    }

    shown_types = sorted(enabled_types & present_types)
    handles = [
        Line2D(
            [0],
            [0],
            marker="s",
            linestyle="",
            markerfacecolor="#1565c0",
            markeredgecolor="#0d47a1",
            markersize=9,
            label="Ego",
        )
    ]

    for agent_type in shown_types:
        color = agent_type_color(agent_type)
        handles.append(
            Line2D(
                [0],
                [0],
                marker="o",
                linestyle="",
                markerfacecolor=color,
                markeredgecolor="#212121",
                markersize=9,
                label=AGENT_TYPE_LABELS.get(
                    agent_type,
                    agent_type.replace("_", " ").title(),
                ),
            )
        )

    handles.extend(
        [
            Line2D(
                [0],
                [0],
                color="#212121",
                linewidth=2.0,
                marker=">",
                markersize=6,
                label="Entity heading",
            ),
            Line2D(
                [0], [0],
                color=SEMANTIC_PREDICATE_ARROW_STYLES["np:follows"]["color"],
                linewidth=SEMANTIC_PREDICATE_ARROW_STYLES["np:follows"]["linewidth"],
                label="Follows arrow",
            ),
            Line2D(
                [0], [0],
                color=SEMANTIC_PREDICATE_ARROW_STYLES["np:overtakes"]["color"],
                linewidth=SEMANTIC_PREDICATE_ARROW_STYLES["np:overtakes"]["linewidth"],
                label="Overtake arrow",
            ),
            Line2D(
                [0], [0],
                color=LANE_CHANGE_ARROW_STYLE["color"],
                linewidth=LANE_CHANGE_ARROW_STYLE["linewidth"],
                label="Lane-change arrow",
            ),
            Line2D(
                [0], [0],
                color=SEMANTIC_PREDICATE_ARROW_STYLES["np:mergesInFrontOf"]["color"],
                linewidth=SEMANTIC_PREDICATE_ARROW_STYLES["np:mergesInFrontOf"]["linewidth"],
                label="Merge arrow",
            ),
            Line2D(
                [0], [0],
                color=SEMANTIC_PREDICATE_ARROW_STYLES["np:yieldsTo"]["color"],
                linewidth=SEMANTIC_PREDICATE_ARROW_STYLES["np:yieldsTo"]["linewidth"],
                label="Yield arrow",
            ),
            Line2D(
                [0], [0],
                color=DEFAULT_SEMANTIC_ARROW_STYLE["color"],
                linewidth=DEFAULT_SEMANTIC_ARROW_STYLE["linewidth"],
                label="Other semantic relation",
            ),
        ]
    )

    ax.legend(
        handles=handles,
        loc="upper right",
        fontsize=8,
        title="Agent types",
        title_fontsize=9,
    )


def _world_to_ego_coordinates(x, y, ego_x, ego_y, ego_heading):
    dx = float(x) - float(ego_x)
    dy = float(y) - float(ego_y)
    return (
        math.cos(ego_heading) * dx + math.sin(ego_heading) * dy,
        -math.sin(ego_heading) * dx + math.cos(ego_heading) * dy,
    )


def _extract_velocity_xy(source_object):
    velocity = getattr(source_object, "velocity", None)
    if velocity is not None:
        vx = getattr(velocity, "x", None)
        vy = getattr(velocity, "y", None)
        if vx is not None and vy is not None:
            return float(vx), float(vy)

    dynamic_state = getattr(source_object, "dynamic_car_state", None)
    velocity = getattr(dynamic_state, "rear_axle_velocity_2d", None)
    if velocity is not None:
        vx = getattr(velocity, "x", None)
        vy = getattr(velocity, "y", None)
        if vx is not None and vy is not None:
            return float(vx), float(vy)
    return 0.0, 0.0


def _half_diagonal(row):
    return 0.5 * math.hypot(
        float(row.get("length") or 0.0),
        float(row.get("width") or 0.0),
    )


def _get_lane_ids_at_point(map_api, x, y):
    if map_api is None or not NUPLAN_MAP_IMPORTS_AVAILABLE:
        return set()
    point = Point2D(float(x), float(y))
    result = set()
    for layer_name in ("LANE", "LANE_CONNECTOR"):
        if not hasattr(SemanticMapLayer, layer_name):
            continue
        layer = getattr(SemanticMapLayer, layer_name)
        try:
            objects = map_api.get_all_map_objects(point, layer)
        except Exception:
            objects = []
        for obj in objects or []:
            object_id = getattr(obj, "id", None)
            if object_id is not None:
                result.add(str(object_id))
    return result


def _constant_velocity_clearance(ego_row, agent_row, *, horizon_s, step_s):
    ego_vx, ego_vy = _extract_velocity_xy(ego_row["source_object"])
    agent_vx, agent_vy = _extract_velocity_xy(agent_row["source_object"])
    combined_radius = _half_diagonal(ego_row) + _half_diagonal(agent_row)
    minimum_clearance = float("inf")
    minimum_time = 0.0
    steps = max(1, int(math.ceil(float(horizon_s) / float(step_s))))
    for step_index in range(steps + 1):
        t = min(float(horizon_s), step_index * float(step_s))
        ex = float(ego_row["x"]) + ego_vx * t
        ey = float(ego_row["y"]) + ego_vy * t
        ax_ = float(agent_row["x"]) + agent_vx * t
        ay_ = float(agent_row["y"]) + agent_vy * t
        clearance = math.hypot(ax_ - ex, ay_ - ey) - combined_radius
        if clearance < minimum_clearance:
            minimum_clearance = clearance
            minimum_time = t
    return minimum_clearance, minimum_time


def _diagnostic_primary_sector(
    longitudinal,
    lateral,
    *,
    longitudinal_deadband_m=1.0,
    lateral_deadband_m=0.75,
    side_longitudinal_band_m=2.0,
    front_lateral_base_m=2.5,
    front_lateral_ratio=0.30,
):
    """Notebook-side expectation using the v9.2 body-frame spatial rule."""
    x = float(longitudinal)
    y = float(lateral)
    ax_ = abs(x)
    ay_ = abs(y)

    if ax_ <= float(side_longitudinal_band_m):
        if y >= float(lateral_deadband_m):
            return "leftOf"
        if y <= -float(lateral_deadband_m):
            return "rightOf"
        if x >= float(longitudinal_deadband_m):
            return "inFrontOf"
        if x <= -float(longitudinal_deadband_m):
            return "behind"
        return None

    if x >= float(longitudinal_deadband_m):
        corridor = float(front_lateral_base_m) + float(front_lateral_ratio) * x
        if ay_ <= corridor:
            return "inFrontOf"
        return "frontLeftOf" if y > 0 else "frontRightOf"

    if x <= -float(longitudinal_deadband_m):
        corridor = float(front_lateral_base_m) + float(front_lateral_ratio) * ax_
        if ay_ <= corridor:
            return "behind"
        return "rearLeftOf" if y > 0 else "rearRightOf"

    if y >= float(lateral_deadband_m):
        return "leftOf"
    if y <= -float(lateral_deadband_m):
        return "rightOf"
    return None


def select_interesting_agents_for_plot(
    frame,
    entity_table,
    all_spatial_edges,
    *,
    distance_threshold_m=40.0,
    include_within_distance=True,
    include_forward_corridor=True,
    forward_corridor_length_m=70.0,
    forward_corridor_half_width_m=7.0,
    include_same_lane=True,
    include_predicted_path_intersection=True,
    prediction_horizon_s=5.0,
    prediction_step_s=0.25,
    path_intersection_clearance_m=3.0,
    include_existing_spatial_agents=False,
    manual_tokens=None,
):
    """Select diagnostic candidates with an OR over the enabled rules."""
    manual_tokens = set(map(str, manual_tokens or []))
    ego_row = entity_table[entity_table["is_ego"]].iloc[0]
    map_api = getattr(frame, "map_api", None)
    ego_lane_ids = (
        _get_lane_ids_at_point(map_api, ego_row["x"], ego_row["y"])
        if include_same_lane
        else set()
    )

    existing_spatial_tokens = set()
    if include_existing_spatial_agents and not all_spatial_edges.empty:
        existing_spatial_tokens.update(
            all_spatial_edges["subject_token"].dropna().astype(str)
        )
        existing_spatial_tokens.update(
            all_spatial_edges["object_token"].dropna().astype(str)
        )
        existing_spatial_tokens.discard("ego")

    selected_tokens = set()
    audit_rows = []

    for _, agent_row in entity_table[~entity_table["is_ego"]].iterrows():
        token = str(agent_row["track_token"])
        distance = math.hypot(
            float(agent_row["x"]) - float(ego_row["x"]),
            float(agent_row["y"]) - float(ego_row["y"]),
        )
        longitudinal, lateral = _world_to_ego_coordinates(
            agent_row["x"],
            agent_row["y"],
            ego_row["x"],
            ego_row["y"],
            ego_row["heading"],
        )
        within_distance = bool(
            include_within_distance and distance <= float(distance_threshold_m)
        )
        in_forward_corridor = bool(
            include_forward_corridor
            and 0.0 <= longitudinal <= float(forward_corridor_length_m)
            and abs(lateral) <= float(forward_corridor_half_width_m)
        )

        agent_lane_ids = (
            _get_lane_ids_at_point(map_api, agent_row["x"], agent_row["y"])
            if include_same_lane
            else set()
        )
        shared_lane_ids = ego_lane_ids.intersection(agent_lane_ids)
        same_lane = bool(include_same_lane and shared_lane_ids)

        minimum_clearance, minimum_time = _constant_velocity_clearance(
            ego_row,
            agent_row,
            horizon_s=prediction_horizon_s,
            step_s=prediction_step_s,
        )
        predicted_intersection = bool(
            include_predicted_path_intersection
            and minimum_clearance <= float(path_intersection_clearance_m)
        )
        existing_spatial = bool(
            include_existing_spatial_agents and token in existing_spatial_tokens
        )
        manual = token in manual_tokens

        reasons = []
        if within_distance:
            reasons.append("within_distance")
        if in_forward_corridor:
            reasons.append("forward_corridor")
        if same_lane:
            reasons.append("same_lane")
        if predicted_intersection:
            reasons.append("predicted_path_intersection")
        if existing_spatial:
            reasons.append("existing_spatial_predicate")
        if manual:
            reasons.append("manual")

        selected = bool(reasons)
        if selected:
            selected_tokens.add(token)

        audit_rows.append({
            "display_id": agent_row.get("display_id", token),
            "track_token": token,
            "agent_type": agent_row["agent_type"],
            "selected_candidate": selected,
            "selection_reasons": reasons,
            "distance_to_ego_m": distance,
            "longitudinal_m": longitudinal,
            "lateral_m": lateral,
            "expected_body_frame_spatial": _diagnostic_primary_sector(
                longitudinal, lateral
            ),
            "within_distance": within_distance,
            "in_forward_corridor": in_forward_corridor,
            "same_lane": same_lane,
            "shared_lane_ids": sorted(shared_lane_ids),
            "predicted_path_intersection": predicted_intersection,
            "minimum_predicted_clearance_m": minimum_clearance,
            "time_of_minimum_clearance_s": minimum_time,
            "existing_spatial_relation": existing_spatial,
            "manually_selected": manual,
        })

    audit = pd.DataFrame(audit_rows)
    if not audit.empty:
        audit = audit.sort_values(
            ["selected_candidate", "distance_to_ego_m"],
            ascending=[False, True],
        ).reset_index(drop=True)
    return selected_tokens, audit


def filter_edges_by_selected_agents(
    semantic_edges,
    selected_tokens,
    *,
    keep_ego_structure_edges=True,
):
    if semantic_edges.empty:
        return semantic_edges.copy()
    selected_tokens = set(map(str, selected_tokens))
    keep = []
    for row in semantic_edges.itertuples(index=False):
        subject_kind = _entity_kind(row.subject_id)
        object_kind = _entity_kind(row.object_id)
        agent_tokens = []
        if subject_kind == "agent":
            agent_tokens.append(str(row.subject_token))
        if object_kind == "agent":
            agent_tokens.append(str(row.object_token))
        if not agent_tokens:
            keep.append(bool(keep_ego_structure_edges))
        else:
            keep.append(all(token in selected_tokens for token in agent_tokens))
    return semantic_edges.loc[keep].reset_index(drop=True)


def collapse_edges_for_plot(semantic_edges):
    """Prepare drawable arrows while keeping follows/overtakes separate.

    Most predicates remain combined on one directed pair edge. The two
    interaction relations are split into independent rows so a Case 1 frame
    can show a blue follows arrow and an orange overtake arrow simultaneously.

    The returned table always preserves the endpoint columns required by the
    renderer, including on frames with no active predicates.
    """
    endpoint_columns = [
        "subject_id",
        "object_id",
        "subject_token",
        "object_token",
        "relation_type",
        "predicate_group",
        "categories",
        "predicates",
        "relation_labels",
    ]

    if semantic_edges is None or semantic_edges.empty:
        return pd.DataFrame(columns=endpoint_columns)

    normalized = semantic_edges.copy()

    # Compatibility for entity-valued relations reconstructed directly from
    # assertion rows. Keep the renderer robust even when only IDs or only
    # tokens are present in an input edge table.
    if "subject_token" not in normalized.columns:
        if "subject_id" in normalized.columns:
            normalized["subject_token"] = normalized["subject_id"].map(
                entity_id_to_track_token
            )
        else:
            normalized["subject_token"] = ""
    if "object_token" not in normalized.columns:
        if "object_id" in normalized.columns:
            normalized["object_token"] = normalized["object_id"].map(
                entity_id_to_track_token
            )
        else:
            normalized["object_token"] = ""
    if "subject_id" not in normalized.columns:
        normalized["subject_id"] = normalized["subject_token"]
    if "object_id" not in normalized.columns:
        normalized["object_id"] = normalized["object_token"]
    if "relation_type" not in normalized.columns:
        normalized["relation_type"] = "other"
    if "category" not in normalized.columns:
        normalized["category"] = "other"
    if "predicates" not in normalized.columns:
        normalized["predicates"] = [[] for _ in range(len(normalized))]

    expanded_rows = []
    special_predicates = {"np:follows", "np:overtakes", "np:yieldsTo"}
    for row in normalized.to_dict("records"):
        predicates = [
            str(value)
            for value in _excel_predicate_list(row.get("predicates", []))
        ]
        special = [p for p in predicates if p in special_predicates]
        remaining = [p for p in predicates if p not in special_predicates]
        for predicate in special:
            item = dict(row)
            item["predicates"] = [predicate]
            item["predicate_group"] = predicate
            expanded_rows.append(item)
        if remaining:
            item = dict(row)
            item["predicates"] = remaining
            item["predicate_group"] = "__combined_other__"
            expanded_rows.append(item)

    if not expanded_rows:
        return pd.DataFrame(columns=endpoint_columns)

    expanded = pd.DataFrame(expanded_rows)
    grouped = (
        expanded.groupby(
            [
                "subject_id",
                "object_id",
                "subject_token",
                "object_token",
                "relation_type",
                "predicate_group",
            ],
            dropna=False,
        )
        .agg(
            categories=("category", lambda x: sorted(set(map(str, x)))),
            predicates=(
                "predicates",
                lambda values: sorted({
                    p
                    for group in values
                    for p in _excel_predicate_list(group)
                }),
            ),
        )
        # Do not use drop=True here. The grouping keys are the endpoint
        # columns required by prepare_plot_edges() and plot_map_with_agents().
        .reset_index()
    )
    grouped["relation_labels"] = grouped["predicates"].map(
        lambda values: ", ".join(value.replace("np:", "") for value in values)
    )
    return grouped


def prepare_plot_edges(semantic_edges, entity_table):
    """Assign deterministic E1, E2, ... IDs for the selected frame."""
    plot_edges = collapse_edges_for_plot(semantic_edges)
    if plot_edges.empty:
        return plot_edges

    display_lookup = (
        entity_table.set_index("track_token")["display_id"].astype(str).to_dict()
    )
    plot_edges = plot_edges.copy()
    plot_edges["subject_display_id"] = plot_edges["subject_token"].map(
        display_lookup
    ).fillna(plot_edges["subject_token"])
    plot_edges["object_display_id"] = plot_edges["object_token"].map(
        display_lookup
    ).fillna(plot_edges["object_token"])
    plot_edges = plot_edges.sort_values(
        ["subject_display_id", "object_display_id", "relation_type", "relation_labels"]
    ).reset_index(drop=True)
    plot_edges["edge_id"] = ["E%d" % (index + 1) for index in range(len(plot_edges))]
    return plot_edges


def _agent_tokens_with_displayed_predicates(plot_edges):
    tokens = set()
    if plot_edges is None or plot_edges.empty:
        return tokens
    for row in plot_edges.itertuples(index=False):
        if _entity_kind(row.subject_id) == "agent":
            tokens.add(str(row.subject_token))
        if _entity_kind(row.object_id) == "agent":
            tokens.add(str(row.object_token))
    return tokens


def _entity_visual_style(token, selected_tokens, predicate_tokens, manual_tokens, is_ego):
    if is_ego:
        return {
            "facecolor": "#1565c0",
            "edgecolor": "#0d47a1",
            "alpha": 0.95,
            "linewidth": 2.0,
            "zorder": 8,
        }
    if token in manual_tokens:
        return {
            "facecolor": "#8e24aa",
            "edgecolor": "#4a148c",
            "alpha": 0.95,
            "linewidth": 2.2,
            "zorder": 8,
        }
    if token in selected_tokens and token in predicate_tokens:
        return {
            "facecolor": "#e53935",
            "edgecolor": "#8e0000",
            "alpha": 0.95,
            "linewidth": 2.0,
            "zorder": 7,
        }
    if token in selected_tokens:
        return {
            "facecolor": "#fb8c00",
            "edgecolor": "#a94d00",
            "alpha": 0.95,
            "linewidth": 2.0,
            "zorder": 7,
        }
    return {
        "facecolor": "#9e9e9e",
        "edgecolor": "#4d4d4d",
        "alpha": 0.45,
        "linewidth": 0.8,
        "zorder": 4,
    }


def _draw_edge_id(
    ax, source_x, source_y, target_x, target_y, edge_id, curvature,
    edge_color="#6a1b9a",
):
    dx = target_x - source_x
    dy = target_y - source_y
    length = max(math.hypot(dx, dy), 1e-6)
    midpoint_x = (source_x + target_x) / 2.0
    midpoint_y = (source_y + target_y) / 2.0
    displacement = float(np.clip(0.35 * curvature * length, -5.0, 5.0))
    label_x = midpoint_x + (-dy / length) * displacement
    label_y = midpoint_y + (dx / length) * displacement
    ax.text(
        label_x,
        label_y,
        str(edge_id),
        fontsize=8,
        fontweight="bold",
        ha="center",
        va="center",
        zorder=12,
        clip_on=True,
        bbox={
            "boxstyle": "round,pad=0.16",
            "facecolor": "white",
            "edgecolor": edge_color,
            "alpha": 0.96,
        },
    )


def _debug_compact_map_entity(value):
    value = str(value or "")
    if value.startswith("lane_connector:"):
        return "LC:" + value.split(":", 1)[1]
    if value.startswith("lane:"):
        return "L:" + value.split(":", 1)[1]
    if value.startswith("roadblock_connector:"):
        return "RBC:" + value.split(":", 1)[1]
    if value.startswith("roadblock:"):
        return "RB:" + value.split(":", 1)[1]
    return value or None


def _debug_map_parent_label(map_object, kind):
    """Best-effort roadblock/group ID without importing extractor modules."""
    if map_object is None:
        return None

    parent_id = None
    getter = getattr(map_object, "get_roadblock_id", None)
    if callable(getter):
        try:
            parent_id = getter()
        except Exception:
            parent_id = None

    if parent_id is None:
        for attribute in ("roadblock_id", "parent_id"):
            value = getattr(map_object, attribute, None)
            if value is not None:
                parent_id = value
                break

    if parent_id is None:
        return None

    return ("RBC:" if kind == "lane_connector" else "RB:") + str(parent_id)


def _debug_visual_map_assignment(map_api, row):
    """
    Visualization-only fallback lane lookup.

    This deliberately does not import any predicate/extractor module. It asks
    the nuPlan map API directly which lane/lane-connector polygons contain the
    vehicle center and footprint sample points. A unique center containment is
    treated as the current lane. Multiple candidates are shown as ambiguous.
    """
    empty = {
        "primary": None,
        "parent": None,
        "candidates": [],
        "ambiguous": False,
        "reason": "visual_map_no_match",
        "source": "visual_map",
    }
    if map_api is None or not NUPLAN_MAP_IMPORTS_AVAILABLE:
        return empty

    try:
        center_x = float(getattr(row, "x"))
        center_y = float(getattr(row, "y"))
        heading = float(getattr(row, "heading"))
        length = float(getattr(row, "length") or 4.5)
        width = float(getattr(row, "width") or 2.0)
    except Exception:
        return empty

    # Center is the strongest current-lane observation. Footprint samples are
    # added so a physical boundary crossing is visible before/after the center
    # itself changes polygon.
    sample_points = [(center_x, center_y, "center")]
    try:
        corners = oriented_rectangle_corners(
            center_x, center_y, heading, length, width
        )
        for index, corner in enumerate(corners):
            sample_points.append((float(corner[0]), float(corner[1]), f"corner{index}"))
    except Exception:
        pass

    objects_by_entity = {}
    center_entities = []
    footprint_entities = []

    for x, y, sample_kind in sample_points:
        point = Point2D(float(x), float(y))
        for layer_name, prefix, kind in (
            ("LANE", "lane:", "lane"),
            ("LANE_CONNECTOR", "lane_connector:", "lane_connector"),
        ):
            if not hasattr(SemanticMapLayer, layer_name):
                continue
            layer = getattr(SemanticMapLayer, layer_name)
            try:
                objects = map_api.get_all_map_objects(point, layer) or []
            except Exception:
                objects = []

            for map_object in objects:
                object_id = getattr(map_object, "id", None)
                if object_id is None:
                    continue
                entity = prefix + str(object_id)
                objects_by_entity.setdefault(entity, (map_object, kind))
                if sample_kind == "center" and entity not in center_entities:
                    center_entities.append(entity)
                if entity not in footprint_entities:
                    footprint_entities.append(entity)

    # Prefer a unique center assignment. At a shared boundary, preserve all
    # alternatives instead of inventing a primary lane.
    primary_raw = None
    ambiguous = False
    if len(center_entities) == 1:
        primary_raw = center_entities[0]
    elif len(center_entities) > 1:
        ambiguous = True
    elif len(footprint_entities) == 1:
        # Center can be microscopically outside a polygon due to map precision;
        # a unique footprint lane is still useful for visual debugging.
        primary_raw = footprint_entities[0]
    elif len(footprint_entities) > 1:
        ambiguous = True

    candidates_raw = []
    for entity in center_entities + footprint_entities:
        if entity not in candidates_raw:
            candidates_raw.append(entity)

    parent = None
    if primary_raw in objects_by_entity:
        map_object, kind = objects_by_entity[primary_raw]
        parent = _debug_map_parent_label(map_object, kind)

    return {
        "primary": _debug_compact_map_entity(primary_raw),
        "parent": parent,
        "candidates": [
            _debug_compact_map_entity(value)
            for value in candidates_raw
            if _debug_compact_map_entity(value)
        ],
        "ambiguous": bool(ambiguous),
        "reason": "visual_map_center_or_footprint",
        "source": "visual_map",
    }


def _debug_entity_map_assignment(frame_assertions, row, map_api=None):
    """
    Return lane information for the rendered entity.

    Priority:
      1. exact map assertions already written by the extractor;
      2. safe visualization-only nuPlan map lookup when the extraction output
         contains no map category (for example interaction-only runs).
    """
    empty = {
        "primary": None,
        "parent": None,
        "candidates": [],
        "ambiguous": False,
        "reason": "no_map_assertions",
        "source": "none",
    }

    entity_id = str(getattr(row, "entity_id", ""))
    rows = None
    if (
        frame_assertions is not None
        and not frame_assertions.empty
        and "predicate_id" in frame_assertions.columns
        and "subject_id" in frame_assertions.columns
        and entity_id
    ):
        rows = frame_assertions.loc[
            frame_assertions["subject_id"].astype(str).eq(entity_id)
        ].copy()

    if rows is not None and not rows.empty:
        primary_rows = rows.loc[
            rows["predicate_id"].astype(str).isin(
                {"np:hasPrimaryLane", "np:hasPrimaryLaneConnector"}
            )
        ]
        ambiguous_rows = rows.loc[
            rows["predicate_id"].astype(str).eq("np:hasAmbiguousMapMatch")
        ]
        ambiguous = not ambiguous_rows.empty

        primary_raw = None
        evidence = {}
        if not primary_rows.empty:
            selected = primary_rows.iloc[-1]
            primary_raw = str(selected.get("object_id") or "")
            evidence = _evidence_mapping(selected.get("evidence_json"))
        elif ambiguous:
            selected = ambiguous_rows.iloc[-1]
            evidence = _evidence_mapping(selected.get("evidence_json"))

        candidate_values = []
        membership_ids = {
            "np:inLane", "np:intersectsLane",
            "np:inLaneConnector", "np:intersectsLaneConnector",
        }
        if "object_id" in rows.columns:
            membership_rows = rows.loc[
                rows["predicate_id"].astype(str).isin(membership_ids)
            ]
            for value in membership_rows["object_id"].dropna().astype(str):
                compact = _debug_compact_map_entity(value)
                if compact and compact not in candidate_values:
                    candidate_values.append(compact)

        for item in list(evidence.get("candidate_summaries") or []):
            if not isinstance(item, dict):
                continue
            entity = item.get("entity_id")
            if not entity:
                kind = str(item.get("kind") or "")
                object_id = item.get("object_id")
                if object_id is not None:
                    entity = (
                        f"lane_connector:{object_id}"
                        if kind == "lane_connector"
                        else f"lane:{object_id}"
                    )
            compact = _debug_compact_map_entity(entity)
            if compact and compact not in candidate_values:
                candidate_values.append(compact)

        parent = None
        if primary_raw:
            parent_rows = frame_assertions.loc[
                frame_assertions["subject_id"].astype(str).eq(primary_raw)
                & frame_assertions["predicate_id"].astype(str).isin(
                    {"np:hasParentRoadblock", "np:hasParentRoadblockConnector"}
                )
            ]
            if not parent_rows.empty:
                parent = _debug_compact_map_entity(
                    parent_rows.iloc[-1].get("object_id")
                )

        if parent is None:
            for item in list(evidence.get("candidate_summaries") or []):
                if not isinstance(item, dict):
                    continue
                entity = str(item.get("entity_id") or "")
                if primary_raw and entity != primary_raw:
                    continue
                parent_id = item.get("parent_id")
                if parent_id is None:
                    continue
                kind = str(item.get("kind") or "")
                parent = (
                    "RBC:" if kind == "lane_connector" else "RB:"
                ) + str(parent_id)
                break

        # If any map evidence is actually present, trust the extractor output.
        has_map_evidence = (
            bool(primary_raw)
            or bool(ambiguous)
            or bool(candidate_values)
        )
        if has_map_evidence:
            return {
                "primary": _debug_compact_map_entity(primary_raw),
                "parent": parent,
                "candidates": candidate_values,
                "ambiguous": bool(ambiguous),
                "reason": str(
                    evidence.get("selection_reason")
                    or "exported_map_assertions"
                ),
                "source": "assertions",
            }

    fallback = _debug_visual_map_assignment(map_api, row)
    if fallback.get("primary") or fallback.get("candidates") or fallback.get("ambiguous"):
        return fallback
    return empty


def _debug_agent_lane_text(assignment):
    """Compact text shown above an ego/agent for lane-change debugging."""
    primary = assignment.get("primary")
    candidates = list(assignment.get("candidates") or [])
    parent = assignment.get("parent")
    ambiguous = bool(assignment.get("ambiguous"))

    if primary:
        lines = [f"P={primary}"]
    elif ambiguous:
        lines = ["P=AMBIG"]
    else:
        lines = ["P=NONE"]

    if candidates:
        lines.append("C=" + ",".join(candidates[:3]))
        if len(candidates) > 3:
            lines[-1] += f"+{len(candidates) - 3}"
    if parent:
        lines.append(parent)
    return "\n".join(lines)


def _yield_evidence_for_edge(frame_assertions, edge):
    """Return the active np:yieldsTo evidence for one drawable edge."""
    if frame_assertions is None or frame_assertions.empty:
        return {}
    required = {"predicate_id", "subject_id", "object_id"}
    if not required.issubset(frame_assertions.columns):
        return {}

    subject_id = str(getattr(edge, "subject_id", ""))
    object_id = str(getattr(edge, "object_id", ""))
    rows = frame_assertions.loc[
        frame_assertions["predicate_id"].astype(str).eq("np:yieldsTo")
        & frame_assertions["subject_id"].astype(str).eq(subject_id)
        & frame_assertions["object_id"].astype(str).eq(object_id)
    ]
    if rows.empty:
        return {}
    if "evidence_json" not in rows.columns:
        return {}
    return _evidence_mapping(rows.iloc[-1].get("evidence_json"))


def _draw_yield_reference_overlay(ax, frame_assertions, edge, source_x, source_y):
    """Draw the verified yield reference/conflict point stored by v9.5.39."""
    predicates = {
        str(value)
        for value in _excel_predicate_list(getattr(edge, "predicates", []))
    }
    if "np:yieldsTo" not in predicates:
        return False

    evidence = _yield_evidence_for_edge(frame_assertions, edge)
    x = evidence.get("yield_reference_x_m", evidence.get("conflict_x_m"))
    y = evidence.get("yield_reference_y_m", evidence.get("conflict_y_m"))
    try:
        x = float(x)
        y = float(y)
    except (TypeError, ValueError):
        return False
    if not (math.isfinite(x) and math.isfinite(y)):
        return False

    style = SEMANTIC_PREDICATE_ARROW_STYLES["np:yieldsTo"]
    ax.plot(
        [float(source_x), x],
        [float(source_y), y],
        linestyle="--",
        linewidth=2.0,
        color=style["color"],
        alpha=0.88,
        zorder=8.8,
    )
    ax.scatter(
        [x],
        [y],
        marker="x",
        s=95,
        linewidths=2.5,
        color=style["color"],
        zorder=10.5,
    )
    mode = str(evidence.get("yield_mode") or "yield")
    label = "Yield conflict" if mode == "crossing_conflict" else "Yield merge point"
    ax.text(
        x,
        y + 1.0,
        label,
        fontsize=7.5,
        fontweight="bold",
        color=style["color"],
        ha="center",
        va="bottom",
        zorder=11,
        clip_on=True,
        bbox={
            "boxstyle": "round,pad=0.14",
            "facecolor": "white",
            "edgecolor": style["color"],
            "alpha": 0.86,
            "linewidth": 0.6,
        },
    )
    return True


def plot_map_with_agents(
    ax,
    frame,
    entity_table,
    plot_edges,
    frame_assertions=None,
    *,
    scenario_token,
    frame_index,
    selected_tokens,
    predicate_tokens,
    map_radius_m,
    manual_highlight_tokens=None,
    draw_semantic_arrows=True,
    show_agent_labels=True,
    show_agent_lane_info=False,
    show_map_structure_ids=False,
    show_edge_ids=True,
    agent_type_switches=None,
):
    manual_tokens = set(map(str, manual_highlight_tokens or []))
    ego_row = entity_table[entity_table["is_ego"]].iloc[0]
    ego_xy = (float(ego_row["x"]), float(ego_row["y"]))
    map_api = getattr(frame, "map_api", None)
    if map_api is not None:
        plot_local_map(
            ax,
            map_api,
            ego_xy,
            radius_m=map_radius_m,
            show_map_structure_ids=bool(show_map_structure_ids),
        )

    entity_by_token = entity_table.set_index("track_token", drop=False)
    for row in entity_table.itertuples(index=False):
        token = str(row.track_token)
        style = _entity_visual_style(
            token,
            selected_tokens,
            predicate_tokens,
            manual_tokens,
            bool(row.is_ego),
        )

        if not bool(row.is_ego):
            class_color = agent_type_color(row.agent_type)
            style = dict(style)
            style["facecolor"] = class_color
            style["edgecolor"] = "#212121"
            style["alpha"] = 0.95 if token in selected_tokens else 0.72
            style["linewidth"] = 2.6 if token in selected_tokens else 1.2
        corners = oriented_rectangle_corners(
            row.x, row.y, row.heading, row.length, row.width
        )
        ax.add_patch(MplPolygon(corners, closed=True, **style))
        arrow_length = max(2.5, float(row.length or 4.0) * 0.65)
        ax.arrow(
            row.x,
            row.y,
            arrow_length * math.cos(row.heading),
            arrow_length * math.sin(row.heading),
            width=0.07,
            head_width=0.7,
            head_length=0.9,
            length_includes_head=True,
            color=style["edgecolor"],
            alpha=0.9,
            zorder=style["zorder"] + 0.2,
        )
        if show_agent_labels and (
            bool(row.is_ego)
            or token in selected_tokens
            or token in predicate_tokens
            or token in manual_tokens
        ):
            ax.text(
                row.x,
                row.y,
                str(row.display_id),
                fontsize=8.5,
                fontweight="bold",
                color="white",
                ha="center",
                va="center",
                zorder=11,
                clip_on=True,
            )

        if show_agent_lane_info:
            assignment = _debug_entity_map_assignment(
                frame_assertions,
                row,
                map_api=map_api,
            )
            lane_text = _debug_agent_lane_text(assignment)
            vertical_offset = max(1.5, 0.5 * float(row.width or 2.0) + 1.0)
            ax.text(
                float(row.x),
                float(row.y) + vertical_offset,
                lane_text,
                fontsize=6.0,
                color="#111111",
                ha="center",
                va="bottom",
                zorder=12,
                clip_on=True,
                bbox={
                    "boxstyle": "round,pad=0.18",
                    "facecolor": "white",
                    "edgecolor": style["edgecolor"],
                    "alpha": 0.88,
                    "linewidth": 0.6,
                },
            )

    if draw_semantic_arrows and plot_edges is not None and not plot_edges.empty:
        pair_counts = {}
        for edge in plot_edges.itertuples(index=False):
            subject_token = str(edge.subject_token)
            object_token = str(edge.object_token)
            if (
                subject_token not in entity_by_token.index
                or object_token not in entity_by_token.index
            ):
                continue
            source = entity_by_token.loc[subject_token]
            target = entity_by_token.loc[object_token]
            if isinstance(source, pd.DataFrame):
                source = source.iloc[0]
            if isinstance(target, pd.DataFrame):
                target = target.iloc[0]

            key = tuple(sorted((subject_token, object_token)))
            pair_index = pair_counts.get(key, 0)
            pair_counts[key] = pair_index + 1
            curvature = 0.10 + 0.06 * pair_index
            if subject_token > object_token:
                curvature *= -1.0
            curvature = float(np.clip(curvature, -0.32, 0.32))

            source_x = float(source["x"])
            source_y = float(source["y"])
            target_x = float(target["x"])
            target_y = float(target["y"])
            edge_style = semantic_edge_arrow_style(edge)

            ax.annotate(
                "",
                xy=(target_x, target_y),
                xytext=(source_x, source_y),
                arrowprops={
                    "arrowstyle": "-|>",
                    "linewidth": edge_style["linewidth"],
                    "color": edge_style["color"],
                    "alpha": 0.92,
                    "shrinkA": 10,
                    "shrinkB": 10,
                    "connectionstyle": "arc3,rad=%s" % curvature,
                },
                zorder=9,
            )
            _draw_yield_reference_overlay(
                ax,
                frame_assertions,
                edge,
                source_x,
                source_y,
            )
            if show_edge_ids:
                _draw_edge_id(
                    ax,
                    source_x,
                    source_y,
                    target_x,
                    target_y,
                    edge.edge_id,
                    curvature,
                    edge_color=edge_style["color"],
                )

    ax.set_xlim(ego_xy[0] - map_radius_m, ego_xy[0] + map_radius_m)
    ax.set_ylim(ego_xy[1] - map_radius_m, ego_xy[1] + map_radius_m)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("Global x [m]")
    ax.set_ylabel("Global y [m]")
    ax.grid(True, alpha=0.16)
    ax.set_title(
        "Map and selected semantic relations — %s\nframe=%s, timestamp=%s"
        % (scenario_token, frame_index, int(frame.timestamp_us))
    )

    add_agent_type_legend(
        ax,
        entity_table,
        agent_type_switches,
    )


print("Agent selection, diagnostic colors, and stable edge IDs are ready.")

In [ ]:
# ============================================================
# 10. CLEAN SEMANTIC GRAPH
# - only stable edge IDs are written on arrows
# - full predicates are printed below the figure
# - ego and agent headings are drawn explicitly
# - node IDs are centered inside the nodes
# Python 3.9 compatible
# ============================================================

def _draw_heading_arrow(
    ax,
    x,
    y,
    heading,
    *,
    length,
    color,
    linewidth=2.0,
    zorder=7,
):
    """Draw an entity-orientation arrow from its center."""
    if not all(math.isfinite(float(v)) for v in (x, y, heading, length)):
        return

    end_x = float(x) + float(length) * math.cos(float(heading))
    end_y = float(y) + float(length) * math.sin(float(heading))

    ax.annotate(
        "",
        xy=(end_x, end_y),
        xytext=(float(x), float(y)),
        arrowprops={
            "arrowstyle": "-|>",
            "linewidth": float(linewidth),
            "color": color,
            "alpha": 0.95,
            "shrinkA": 0,
            "shrinkB": 0,
            "mutation_scale": 12,
        },
        zorder=zorder,
    )


def plot_semantic_graph_real_coordinates(
    ax,
    entity_table,
    plot_edges,
    *,
    selected_tokens=None,
    predicate_tokens=None,
    include_all_agents=False,
    label_nodes=True,
    show_edge_ids=True,
    show_heading_arrows=True,
    heading_length_scale=0.75,
    minimum_heading_length_m=2.5,
    padding_m=12.0,
    agent_type_switches=None,
):
    has_edges = plot_edges is not None and not plot_edges.empty
    if not has_edges and not include_all_agents:
        ax.text(
            0.5,
            0.5,
            "No enabled semantic edges for this frame.",
            transform=ax.transAxes,
            ha="center",
            va="center",
            fontsize=13,
        )
        ax.set_title("General semantic graph")
        return

    if plot_edges is None:
        plot_edges = pd.DataFrame(columns=["subject_token", "object_token"])

    selected_tokens = set(map(str, selected_tokens or set()))
    predicate_tokens = set(map(str, predicate_tokens or set()))
    entity_by_token = entity_table.set_index("track_token", drop=False)

    # By default, plot only entities that actually participate in displayed edges.
    graph_tokens = set()
    if has_edges and {"subject_token", "object_token"}.issubset(plot_edges.columns):
        graph_tokens.update(plot_edges["subject_token"].dropna().astype(str))
        graph_tokens.update(plot_edges["object_token"].dropna().astype(str))

    if include_all_agents:
        graph_tokens.update(entity_table["track_token"].astype(str))

    graph_entities = entity_table[
        entity_table["track_token"].astype(str).isin(graph_tokens)
    ].copy()

    if graph_entities.empty:
        ax.text(
            0.5,
            0.5,
            "Enabled edges do not have plottable entity coordinates.",
            transform=ax.transAxes,
            ha="center",
            va="center",
        )
        return

    # --------------------------------------------------------
    # Draw semantic relation arrows.
    # One arrow per directed subject/object pair.
    # Only E1, E2, ... is written on the graph.
    # --------------------------------------------------------
    pair_counts = {}

    for edge in plot_edges.itertuples(index=False):
        subject_token = str(edge.subject_token)
        object_token = str(edge.object_token)

        if (
            subject_token not in entity_by_token.index
            or object_token not in entity_by_token.index
        ):
            continue

        source = entity_by_token.loc[subject_token]
        target = entity_by_token.loc[object_token]

        if isinstance(source, pd.DataFrame):
            source = source.iloc[0]
        if isinstance(target, pd.DataFrame):
            target = target.iloc[0]

        source_x = float(source["x"])
        source_y = float(source["y"])
        target_x = float(target["x"])
        target_y = float(target["y"])

        if not all(
            math.isfinite(value)
            for value in (source_x, source_y, target_x, target_y)
        ):
            continue

        key = tuple(sorted((subject_token, object_token)))
        pair_index = pair_counts.get(key, 0)
        pair_counts[key] = pair_index + 1

        # Opposite directions bend to opposite sides.
        curvature = 0.12 + 0.08 * pair_index
        if subject_token > object_token:
            curvature *= -1.0
        curvature = float(np.clip(curvature, -0.34, 0.34))
        edge_style = semantic_edge_arrow_style(edge)

        ax.annotate(
            "",
            xy=(target_x, target_y),
            xytext=(source_x, source_y),
            arrowprops={
                "arrowstyle": "-|>",
                "linewidth": edge_style["linewidth"],
                "color": edge_style["color"],
                "alpha": 0.92,
                "shrinkA": 16,
                "shrinkB": 16,
                "connectionstyle": "arc3,rad=%s" % curvature,
                "mutation_scale": 13,
            },
            zorder=3,
        )

        if show_edge_ids:
            _draw_edge_id(
                ax,
                source_x,
                source_y,
                target_x,
                target_y,
                edge.edge_id,
                curvature,
                edge_color=edge_style["color"],
            )

    # --------------------------------------------------------
    # Draw nodes, centered IDs, and entity headings.
    # --------------------------------------------------------
    for row in graph_entities.itertuples(index=False):
        token = str(row.track_token)
        is_ego = bool(row.is_ego)

        style = _entity_visual_style(
            token,
            selected_tokens,
            predicate_tokens,
            set(),
            is_ego,
        )

        # Same agent-type colors as the map.
        if not is_ego:
            class_color = agent_type_color(row.agent_type)
            style = dict(style)
            style["facecolor"] = class_color
            style["edgecolor"] = "#212121"
            style["alpha"] = 0.98 if token in selected_tokens else 0.78
            style["linewidth"] = 2.8 if token in selected_tokens else 1.5

        x = float(row.x)
        y = float(row.y)

        ax.scatter(
            [x],
            [y],
            s=300 if is_ego else 210,
            marker="s" if is_ego else "o",
            facecolor=style["facecolor"],
            edgecolor=style["edgecolor"],
            linewidth=style.get("linewidth", 1.8),
            alpha=style.get("alpha", 1.0),
            zorder=5,
        )

        if show_heading_arrows:
            entity_length = getattr(row, "length", None)
            try:
                entity_length = float(entity_length)
            except (TypeError, ValueError):
                entity_length = 4.0 if is_ego else 2.5

            heading_length = max(
                float(minimum_heading_length_m),
                float(entity_length) * float(heading_length_scale),
            )

            _draw_heading_arrow(
                ax,
                x,
                y,
                float(row.heading),
                length=heading_length,
                color="#212121",
                linewidth=2.2 if is_ego else 1.8,
                zorder=7,
            )

        if label_nodes:
            # Stable ID is centered inside the node: no vertical shift.
            ax.text(
                x,
                y,
                str(row.display_id),
                fontsize=8.5,
                fontweight="bold",
                color="white",
                ha="center",
                va="center",
                zorder=9,
                clip_on=True,
            )

    x_values = pd.to_numeric(graph_entities["x"], errors="coerce")
    y_values = pd.to_numeric(graph_entities["y"], errors="coerce")
    finite = np.isfinite(x_values) & np.isfinite(y_values)
    x_values = x_values[finite]
    y_values = y_values[finite]

    if not x_values.empty and not y_values.empty:
        ax.set_xlim(
            float(x_values.min()) - float(padding_m),
            float(x_values.max()) + float(padding_m),
        )
        ax.set_ylim(
            float(y_values.min()) - float(padding_m),
            float(y_values.max()) + float(padding_m),
        )

    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("Global x [m]")
    ax.set_ylabel("Global y [m]")
    ax.grid(True, alpha=0.22)
    ax.set_title(
        "Semantic graph — purple: SUBJECT → OBJECT, black: entity heading"
    )

    handles = [
        Line2D(
            [0], [0],
            color="#6a1b9a",
            linewidth=1.8,
            marker=">",
            markersize=6,
            label="Semantic predicate direction",
        ),
        Line2D(
            [0], [0],
            color="#212121",
            linewidth=2.0,
            marker=">",
            markersize=6,
            label="Ego / agent heading",
        ),
    ]
    ax.legend(handles=handles, loc="best", fontsize=8)


    add_agent_type_legend(
        ax,
        graph_entities,
        agent_type_switches,
    )

def edge_legend_dataframe(plot_edges):
    """
    Return a clean two-column table:

        relation      predicate
        EGO → A1      frontRightOf, near
    """
    if plot_edges is None or plot_edges.empty:
        return pd.DataFrame(
            columns=["relation", "predicate"]
        )

    relation_text = (
        plot_edges["subject_display_id"].astype(str)
        + " → "
        + plot_edges["object_display_id"].astype(str)
    )

    return pd.DataFrame({
        "relation": relation_text,
        "predicate": plot_edges["relation_labels"].astype(str),
    })


def print_edge_legend(edge_legend, max_rows=None):
    """Print the edge/predicate table below the plots, never inside a plot."""
    print("\nDISPLAYED PREDICATES")

    if edge_legend is None or edge_legend.empty:
        print("No displayed semantic edges.")
        return

    shown = edge_legend.copy()

    if max_rows is not None:
        shown = shown.head(int(max_rows))

    display(shown.reset_index(drop=True))

    if max_rows is not None and len(edge_legend) > len(shown):
        print(
            "Showing %d of %d rows. "
            "Use RESULT['edge_legend'] to inspect all rows."
            % (len(shown), len(edge_legend))
        )


print(
    "Clean semantic graph is ready: centered node IDs, heading arrows, "
    "stable E-numbers, and predicates printed below."
)


In [ ]:
# ============================================================
# 11. GENERAL PREDICATE SUMMARY TABLES
# ============================================================

def predicate_count_table(assertions, definitions=DEFINITIONS, *, categories=None):
    output_columns = [
        "category", "predicate_id", "label", "count",
        "assertion_kind", "value_type", "units", "description",
    ]
    if (
        assertions is None
        or assertions.empty
        or "predicate_id" not in assertions.columns
        or "predicate_id" not in definitions.columns
    ):
        return pd.DataFrame(columns=output_columns)
    table = assertions.copy()
    if categories is not None:
        allowed = set(definitions.loc[
            definitions["category"].astype(str).isin(set(map(str, categories))),
            "predicate_id",
        ].astype(str))
        table = table[table["predicate_id"].astype(str).isin(allowed)]
    counts = (
        table.groupby("predicate_id", dropna=False)
        .size()
        .rename("count")
        .reset_index()
    )
    meta_cols = [
        c for c in [
            "predicate_id", "category", "label", "assertion_kind",
            "value_type", "units", "description",
        ]
        if c in definitions.columns
    ]
    counts = counts.merge(
        definitions[meta_cols].drop_duplicates("predicate_id"),
        on="predicate_id",
        how="left",
    )
    cols = [
        c for c in [
            "category", "predicate_id", "label", "count",
            "assertion_kind", "value_type", "units", "description",
        ]
        if c in counts.columns
    ]
    return (
        counts[cols]
        .sort_values(
            ["category", "count", "predicate_id"],
            ascending=[True, False, True],
        )
        .reset_index(drop=True)
    )


def detailed_assertion_table(
    assertions,
    definitions=DEFINITIONS,
    *,
    categories=None,
    max_rows=200,
):
    if (
        assertions is None
        or "predicate_id" not in getattr(assertions, "columns", [])
        or "predicate_id" not in definitions.columns
    ):
        return pd.DataFrame(columns=[
            "category", "predicate_id", "label", "subject_id", "object_id",
            "value_json", "assertion_kind", "units", "valid_time_us",
            "description", "evidence_json",
        ])
    result = assertions.copy()
    if categories is not None:
        allowed = set(definitions.loc[
            definitions["category"].astype(str).isin(set(map(str, categories))),
            "predicate_id",
        ].astype(str))
        result = result[result["predicate_id"].astype(str).isin(allowed)]
    meta_cols = [
        c for c in [
            "predicate_id", "category", "label", "description", "units",
        ]
        if c in definitions.columns
    ]
    result = result.merge(
        definitions[meta_cols].drop_duplicates("predicate_id"),
        on="predicate_id",
        how="left",
    )
    keep = [
        c for c in [
            "category", "predicate_id", "label", "subject_id", "object_id",
            "value_json", "assertion_kind", "units", "valid_time_us",
            "description", "evidence_json",
        ]
        if c in result.columns
    ]
    return result[keep].head(int(max_rows))


def _decoded_assertion_value(value):
    """Decode value_json while preserving strings and numeric values."""
    parsed = _safe_json_load(value, default=value)
    if isinstance(parsed, (dict, list)):
        return json.dumps(parsed, ensure_ascii=False)
    return parsed


def _display_id_lookup(entity_table):
    """Map frame entity IDs and track tokens to stable EGO/A1/A2 labels."""
    lookup = {}
    if entity_table is None or entity_table.empty:
        return lookup

    for row in entity_table.itertuples(index=False):
        display_id = str(getattr(row, "display_id", ""))
        entity_id = str(getattr(row, "entity_id", ""))
        track_token = str(getattr(row, "track_token", ""))

        if entity_id:
            lookup[entity_id] = display_id or entity_id
        if track_token:
            lookup[track_token] = display_id or track_token

    return lookup


def motion_predicate_values_table(
    frame_assertions,
    entity_table,
    definitions=DEFINITIONS,
    *,
    relation_switches=None,
    selected_agent_tokens=None,
    max_rows=500,
):
    """
    Create a readable motion-value table for the selected frame.

    Entity rows:
        entity | predicate | value | unit

    Pairwise rows:
        subject -> object | predicate | value | unit

    The table is intended to be displayed only when the notebook's
    category switch contains: "motion": True.
    """
    columns = [
        "scope",
        "subject",
        "object",
        "relation",
        "predicate",
        "value",
        "unit",
        "description",
    ]

    if frame_assertions is None or frame_assertions.empty:
        return pd.DataFrame(columns=columns)

    motion_ids = set(definitions.loc[
        definitions["category"].astype(str) == "motion",
        "predicate_id",
    ].astype(str))

    rows = frame_assertions[
        frame_assertions["predicate_id"].astype(str).isin(motion_ids)
    ].copy()

    if rows.empty:
        return pd.DataFrame(columns=columns)

    meta_cols = [
        c for c in ["predicate_id", "label", "units", "description"]
        if c in definitions.columns
    ]
    rows = rows.merge(
        definitions[meta_cols].drop_duplicates("predicate_id"),
        on="predicate_id",
        how="left",
    )

    pair_source = frame_assertions
    pairs = pair_member_table(pair_source)
    pair_lookup = {}
    if not pairs.empty:
        for pair in pairs.itertuples(index=False):
            pair_lookup[str(pair.pair_id)] = {
                "subject_id": str(pair.subject_id),
                "object_id": str(pair.object_id),
                "pair_tag": str(pair.pair_tag),
            }

    display_lookup = _display_id_lookup(entity_table)
    enabled_relations = (
        enabled_names(relation_switches)
        if relation_switches is not None
        else set(ALL_SEMANTIC_RELATIONS)
    )

    selected_tokens = None
    if selected_agent_tokens is not None:
        selected_tokens = set(map(str, selected_agent_tokens))

    result_rows = []

    for row in rows.itertuples(index=False):
        predicate_id = str(row.predicate_id)
        raw_subject = str(row.subject_id)
        raw_object = str(row.object_id)
        pair = pair_lookup.get(raw_subject)

        if pair is not None:
            subject_id = pair["subject_id"]
            object_id = pair["object_id"]
            subject_token = entity_id_to_track_token(subject_id)
            object_token = entity_id_to_track_token(object_id)
            relation_type = classify_semantic_relation(
                subject_id,
                object_id,
                pair["pair_tag"],
            )

            if relation_type not in enabled_relations:
                continue

            if selected_tokens is not None:
                non_ego_tokens = {
                    token
                    for token in (subject_token, object_token)
                    if str(token) != "ego"
                }
                if non_ego_tokens and not non_ego_tokens.issubset(selected_tokens):
                    continue

            subject_label = display_lookup.get(
                subject_id,
                display_lookup.get(subject_token, short_token(subject_token)),
            )
            object_label = display_lookup.get(
                object_id,
                display_lookup.get(object_token, short_token(object_token)),
            )

            scope = "pair"
            relation = f"{subject_label} → {object_label}"
        else:
            entity_id = raw_subject
            entity_token = entity_id_to_track_token(entity_id)

            if entity_id not in display_lookup and entity_token not in display_lookup:
                continue

            if (
                selected_tokens is not None
                and entity_token != "ego"
                and entity_token not in selected_tokens
            ):
                continue

            subject_label = display_lookup.get(
                entity_id,
                display_lookup.get(entity_token, short_token(entity_token)),
            )
            object_label = ""
            scope = "entity"
            relation = subject_label

        value = _decoded_assertion_value(
            getattr(row, "value_json", None)
        )

        result_rows.append({
            "scope": scope,
            "subject": subject_label,
            "object": object_label,
            "relation": relation,
            "predicate": predicate_id,
            "value": value,
            "unit": getattr(row, "units", None),
            "description": getattr(row, "description", None),
        })

    if not result_rows:
        return pd.DataFrame(columns=columns)

    result = pd.DataFrame(result_rows)

    # Keep deterministic and readable ordering.
    result["_scope_order"] = result["scope"].map({"entity": 0, "pair": 1})
    result = (
        result.sort_values(
            ["_scope_order", "relation", "predicate"],
            kind="stable",
        )
        .drop(columns=["_scope_order"])
        .reset_index(drop=True)
    )

    return result.head(int(max_rows))



# ============================================================
# TEMPORAL VALUES WITH EGO/A1/A2 LABELS
# ============================================================

def temporal_predicate_values_table(
    frame_assertions,
    entity_table,
    definitions=DEFINITIONS,
    *,
    relation_switches=None,
    selected_agent_tokens=None,
    pair_assertions=None,
    max_rows=500,
):
    """
    Create a readable temporal-value table for the selected frame.

    Entity rows:
        entity | predicate | value | unit

    Pairwise rows:
        subject -> object | predicate | value | unit

    The table is intended to be displayed only when the notebook's
    category switch contains: "temporal": True.
    """
    columns = [
        "scope",
        "subject",
        "object",
        "relation",
        "predicate",
        "value",
        "unit",
        "description",
    ]

    if frame_assertions is None or frame_assertions.empty:
        return pd.DataFrame(columns=columns)

    motion_ids = set(definitions.loc[
        definitions["category"].astype(str) == "temporal",
        "predicate_id",
    ].astype(str))

    rows = frame_assertions[
        frame_assertions["predicate_id"].astype(str).isin(motion_ids)
    ].copy()

    if rows.empty:
        return pd.DataFrame(columns=columns)

    meta_cols = [
        c for c in ["predicate_id", "label", "units", "description"]
        if c in definitions.columns
    ]
    rows = rows.merge(
        definitions[meta_cols].drop_duplicates("predicate_id"),
        on="predicate_id",
        how="left",
    )

    pairs = pair_member_table(frame_assertions)
    pair_lookup = {}
    if not pairs.empty:
        for pair in pairs.itertuples(index=False):
            pair_lookup[str(pair.pair_id)] = {
                "subject_id": str(pair.subject_id),
                "object_id": str(pair.object_id),
                "pair_tag": str(pair.pair_tag),
            }

    display_lookup = _display_id_lookup(entity_table)
    enabled_relations = (
        enabled_names(relation_switches)
        if relation_switches is not None
        else set(ALL_SEMANTIC_RELATIONS)
    )

    selected_tokens = None
    if selected_agent_tokens is not None:
        selected_tokens = set(map(str, selected_agent_tokens))

    result_rows = []

    for row in rows.itertuples(index=False):
        predicate_id = str(row.predicate_id)
        raw_subject = str(row.subject_id)
        raw_object = str(row.object_id)
        pair = pair_lookup.get(raw_subject)

        if pair is not None:
            subject_id = pair["subject_id"]
            object_id = pair["object_id"]
            subject_token = entity_id_to_track_token(subject_id)
            object_token = entity_id_to_track_token(object_id)
            relation_type = classify_semantic_relation(
                subject_id,
                object_id,
                pair["pair_tag"],
            )

            if relation_type not in enabled_relations:
                continue

            if selected_tokens is not None:
                non_ego_tokens = {
                    token
                    for token in (subject_token, object_token)
                    if str(token) != "ego"
                }
                if non_ego_tokens and not non_ego_tokens.issubset(selected_tokens):
                    continue

            subject_label = display_lookup.get(
                subject_id,
                display_lookup.get(subject_token, short_token(subject_token)),
            )
            object_label = display_lookup.get(
                object_id,
                display_lookup.get(object_token, short_token(object_token)),
            )

            scope = "pair"
            relation = f"{subject_label} → {object_label}"
        else:
            entity_id = raw_subject
            entity_token = entity_id_to_track_token(entity_id)

            if entity_id not in display_lookup and entity_token not in display_lookup:
                continue

            if (
                selected_tokens is not None
                and entity_token != "ego"
                and entity_token not in selected_tokens
            ):
                continue

            subject_label = display_lookup.get(
                entity_id,
                display_lookup.get(entity_token, short_token(entity_token)),
            )
            object_label = ""
            scope = "entity"
            relation = subject_label

        value = _decoded_assertion_value(
            getattr(row, "value_json", None)
        )

        result_rows.append({
            "scope": scope,
            "subject": subject_label,
            "object": object_label,
            "relation": relation,
            "predicate": predicate_id,
            "value": value,
            "unit": getattr(row, "units", None),
            "description": getattr(row, "description", None),
        })

    if not result_rows:
        return pd.DataFrame(columns=columns)

    result = pd.DataFrame(result_rows)

    # Keep deterministic and readable ordering.
    result["_scope_order"] = result["scope"].map({"entity": 0, "pair": 1})
    result = (
        result.sort_values(
            ["_scope_order", "relation", "predicate"],
            kind="stable",
        )
        .drop(columns=["_scope_order"])
        .reset_index(drop=True)
    )

    return result.head(int(max_rows))


def show_inspection_tables(
    scenario_assertions,
    frame_assertions,
    semantic_edges,
    entity_table,
    *,
    active_categories,
    max_assertion_rows=200,
):
    print(
        f"Selected frame: {len(frame_assertions):,} total assertions; "
        f"{frame_assertions['predicate_id'].nunique() if not frame_assertions.empty else 0:,} "
        "unique predicates."
    )
    print(f"Displayed semantic edges: {len(semantic_edges):,}.")
    print("\nFRAME PREDICATE COUNTS FOR ENABLED CATEGORIES")
    display(predicate_count_table(frame_assertions, categories=active_categories))
    print("\nSCENARIO PREDICATE COUNTS FOR ENABLED CATEGORIES")
    display(predicate_count_table(scenario_assertions, categories=active_categories))
    print("\nSEMANTIC EDGE TABLE")
    display(semantic_edges)
    print("\nALL ENTITIES AT THE SELECTED FRAME")
    display(entity_table.drop(columns=["source_object"], errors="ignore"))
    print("\nDETAILED ASSERTIONS FOR ENABLED CATEGORIES")
    display(
        detailed_assertion_table(
            frame_assertions,
            categories=active_categories,
            max_rows=max_assertion_rows,
        )
    )


print("General table and motion-value functions are ready.")


In [ ]:
# ============================================================
# 12. ONE GENERAL FUNCTION TO SELECT, FILTER, PLOT, AND INSPECT
# ============================================================

def inspect_semantic_scene(
    *,
    semantic_categories,
    semantic_relations,
    agent_types=None,
    scenario_selector="index",
    scenario_value=0,
    scenario_seed=42,
    filter_seed=42,
    frame_selector="index",
    frame_value=0,
    frame_seed=42,
    output_dir=OUTPUT_DIR,
    map_radius_m=DEFAULT_MAP_RADIUS_M,
    manual_highlight_tokens=None,
    interested_distance_threshold_m=40.0,
    include_within_distance=True,
    include_forward_corridor=True,
    forward_corridor_length_m=70.0,
    forward_corridor_half_width_m=7.0,
    include_same_lane=True,
    include_predicted_path_intersection=True,
    prediction_horizon_s=5.0,
    prediction_step_s=0.25,
    path_intersection_clearance_m=3.0,
    include_existing_spatial_agents=False,
    filter_edges_to_selected_agents=True,
    include_all_agents_in_graph=False,
    draw_semantic_arrows_on_map=True,
    show_agent_labels=True,
    show_edge_ids=True,
    show_heading_arrows=True,
    show_tables=True,
    max_edge_legend_rows=None,
    max_assertion_rows=DEFAULT_MAX_TABLE_ROWS,
):
    scenario, scenario_catalog_row = select_scenario(
        selector=scenario_selector,
        value=scenario_value,
        random_seed=scenario_seed,
        filter_seed=filter_seed,
    )

    scenario_token = str(scenario.token)

    frame, frame_index = select_frame(
        scenario,
        selector=frame_selector,
        value=frame_value,
        random_seed=frame_seed,
    )

    timestamp_us = int(frame.timestamp_us)

    scenario_assertions = load_assertions_for_scenario(
        str(Path(output_dir)),
        scenario_token,
    )

    frame_assertions = assertions_at_timestamp(
        scenario_assertions,
        timestamp_us,
    )

    entity_table_all = add_stable_display_ids(
        build_entity_table(frame, scenario_token),
        scenario,
    )

    # Keep ego plus only the enabled agent classes.
    entity_table = filter_entities_by_agent_type(
        entity_table_all,
        agent_types,
    )

    enabled_agent_tokens = set(
        entity_table.loc[
            ~entity_table["is_ego"].astype(bool),
            "track_token",
        ].astype(str)
    )

    # All enabled semantic predicates before notebook selection.
    semantic_edges_all = semantic_edge_table(
        frame_assertions,
        DEFINITIONS,
        semantic_categories,
        semantic_relations,
        pair_assertions=scenario_assertions,
    )

    semantic_edges_all = filter_edges_by_enabled_entity_tokens(
        semantic_edges_all,
        entity_table,
    )

    spatial_only_switches = {
        name: (name == "spatial")
        for name in ALL_SEMANTIC_CATEGORIES
    }

    all_relation_switches = {
        name: True
        for name in ALL_SEMANTIC_RELATIONS
    }

    all_spatial_edges = semantic_edge_table(
        frame_assertions,
        DEFINITIONS,
        spatial_only_switches,
        all_relation_switches,
        pair_assertions=scenario_assertions,
    )

    all_spatial_edges = filter_edges_by_enabled_entity_tokens(
        all_spatial_edges,
        entity_table,
    )

    # Candidate selection is used only as an edge filter.
    candidate_tokens, notebook_selection_audit = (
        select_interesting_agents_for_plot(
            frame,
            entity_table,
            all_spatial_edges,
            distance_threshold_m=interested_distance_threshold_m,
            include_within_distance=include_within_distance,
            include_forward_corridor=include_forward_corridor,
            forward_corridor_length_m=forward_corridor_length_m,
            forward_corridor_half_width_m=forward_corridor_half_width_m,
            include_same_lane=include_same_lane,
            include_predicted_path_intersection=(
                include_predicted_path_intersection
            ),
            prediction_horizon_s=prediction_horizon_s,
            prediction_step_s=prediction_step_s,
            path_intersection_clearance_m=(
                path_intersection_clearance_m
            ),
            include_existing_spatial_agents=(
                include_existing_spatial_agents
            ),
            manual_tokens=manual_highlight_tokens,
        )
    )

    candidate_tokens = set(candidate_tokens) & enabled_agent_tokens

    if not notebook_selection_audit.empty:
        notebook_selection_audit = notebook_selection_audit[
            notebook_selection_audit["track_token"].astype(str).isin(
                enabled_agent_tokens
            )
        ].copy()

    semantic_edges = (
        filter_edges_by_selected_agents(
            semantic_edges_all,
            candidate_tokens,
        )
        if filter_edges_to_selected_agents
        else semantic_edges_all.copy()
    )

    plot_edges = prepare_plot_edges(
        semantic_edges,
        entity_table,
    )

    predicate_tokens = _agent_tokens_with_displayed_predicates(
        plot_edges
    )

    # Temporal assertions are mainly entity-to-value or pair-state-to-value,
    # so they are not semantic graph edges. Still mark their participating
    # agents as predicate-bearing for map/graph highlighting.
    if bool(semantic_categories.get("temporal", False)):
        predicate_tokens = set(predicate_tokens) | temporal_predicate_agent_tokens(
            frame_assertions=frame_assertions,
            scenario_assertions=scenario_assertions,
            definitions=DEFINITIONS,
            selected_timestamp_us=timestamp_us,
        )

    # IMPORTANT:
    # Only agents satisfying BOTH conditions are highlighted:
    #   1. selected by plot-time rules
    #   2. has at least one displayed predicate
    #
    # Candidate-only agents are neither orange nor highlighted.
    displayed_selected_tokens = (
        set(candidate_tokens)
        & set(predicate_tokens)
    )

    if not notebook_selection_audit.empty:
        notebook_selection_audit = (
            notebook_selection_audit.copy()
        )

        notebook_selection_audit[
            "has_displayed_predicate"
        ] = (
            notebook_selection_audit["track_token"]
            .astype(str)
            .isin(predicate_tokens)
        )

        notebook_selection_audit[
            "displayed_selected"
        ] = (
            notebook_selection_audit["track_token"]
            .astype(str)
            .isin(displayed_selected_tokens)
        )

    edge_legend = edge_legend_dataframe(
        plot_edges
    )

    print("\nAGENT TYPES PRESENT IN THE SELECTED FRAME")
    show_available_agent_types(entity_table_all)

    if len(entity_table) <= 1:
        print(
            "\nWARNING: No non-ego agents match AGENT_TYPES. "
            "nuPlan usually labels cars, trucks, and buses as the generic "
            "'vehicle' class. Set AGENT_TYPES['vehicle'] = True."
        )

    # Two plots only. No table is embedded as a Matplotlib subplot.
    fig, axes = plt.subplots(
        2,
        1,
        figsize=(20, 22),
        constrained_layout=True,
    )

    ax_map, ax_graph = axes

    plot_map_with_agents(
        ax_map,
        frame,
        entity_table,
        plot_edges,
        scenario_token=scenario_token,
        frame_index=frame_index,
        selected_tokens=displayed_selected_tokens,
        predicate_tokens=predicate_tokens,
        map_radius_m=map_radius_m,
        manual_highlight_tokens=manual_highlight_tokens,
        draw_semantic_arrows=draw_semantic_arrows_on_map,
        show_agent_labels=show_agent_labels,
        show_edge_ids=False,
        agent_type_switches=agent_types,
    )

    plot_semantic_graph_real_coordinates(
        ax_graph,
        entity_table,
        plot_edges,
        selected_tokens=displayed_selected_tokens,
        predicate_tokens=predicate_tokens,
        include_all_agents=(
            include_all_agents_in_graph
            or bool(semantic_categories.get("temporal", False))
        ),
        label_nodes=show_agent_labels,
        show_edge_ids=False,
        show_heading_arrows=show_heading_arrows,
        agent_type_switches=agent_types,
    )

    plt.show()

    # Keep the plot clean when notebook tables are disabled.
    if show_tables:
        print_edge_legend(
            edge_legend,
            max_rows=max_edge_legend_rows,
        )

    active_categories = sorted(
        enabled_names(semantic_categories)
    )

    active_relations = sorted(
        enabled_names(semantic_relations)
    )

    metadata = {
        "scenario_catalog_row": scenario_catalog_row,
        "scenario_token": scenario_token,
        "scenario_type": str(scenario.scenario_type),
        "log_name": str(scenario.log_name),
        "map_name": _scenario_map_name(scenario),
        "frame_index": frame_index,
        "timestamp_us": timestamp_us,
        "active_categories": active_categories,
        "active_relations": active_relations,
        "active_agent_types": sorted(enabled_agent_type_names(agent_types)),
        "number_of_all_agents": max(
            0,
            len(entity_table) - 1,
        ),
        "number_of_candidate_agents": len(
            candidate_tokens
        ),
        "number_of_displayed_selected_agents": len(
            displayed_selected_tokens
        ),
        "number_of_edges_before_plot_filter": len(
            semantic_edges_all
        ),
        "number_of_displayed_semantic_edges": len(
            plot_edges
        ),
        "filter_edges_to_selected_agents": (
            filter_edges_to_selected_agents
        ),
    }

    if show_tables:
        print("\nSELECTION METADATA")
        display(
            pd.DataFrame([metadata])
            .T
            .rename(columns={0: "value"})
        )

        # Show only selected candidates that have a displayed predicate.
        print("\nDISPLAYED AGENT SELECTION AUDIT")

        if not notebook_selection_audit.empty:
            displayed_audit = notebook_selection_audit[
                notebook_selection_audit[
                    "displayed_selected"
                ]
            ].copy()

            columns = [
                "display_id",
                "agent_type",
                "selection_reasons",
                "distance_to_ego_m",
                "longitudinal_m",
                "lateral_m",
                "expected_body_frame_spatial",
                "within_distance",
                "in_forward_corridor",
                "same_lane",
                "predicted_path_intersection",
                "has_displayed_predicate",
            ]

            display(
                displayed_audit[
                    [
                        column
                        for column in columns
                        if column in displayed_audit.columns
                    ]
                ].reset_index(drop=True)
            )

    motion_values = pd.DataFrame()

    if show_tables:
        show_inspection_tables(
            scenario_assertions,
            frame_assertions,
            semantic_edges,
            entity_table,
            active_categories=active_categories,
            max_assertion_rows=max_assertion_rows,
        )

        # Show actual motion values only when the motion category is enabled.
        if "motion" in active_categories:
            motion_values = motion_predicate_values_table(
                frame_assertions,
                entity_table,
                DEFINITIONS,
                relation_switches=semantic_relations,
                selected_agent_tokens=(
                    candidate_tokens
                    if filter_edges_to_selected_agents
                    else None
                ),
                max_rows=max_assertion_rows,
            )

            print("\nMOTION PREDICATE VALUES AT THE SELECTED FRAME")
            if motion_values.empty:
                print(
                    "No motion values are available for the current frame, "
                    "enabled agent types, relation directions, and plot filters."
                )
            else:
                display(motion_values)

    return {
        "scenario": scenario,
        "frame": frame,
        "metadata": metadata,
        "scenario_assertions": scenario_assertions,
        "frame_assertions": frame_assertions,
        "entity_table": entity_table,
        "entity_table_all": entity_table_all,
        "semantic_edges": semantic_edges,
        "plot_edges": plot_edges,
        "edge_legend": edge_legend,
        "semantic_edges_before_plot_filter": semantic_edges_all,
        "notebook_selection_audit": notebook_selection_audit,
        "candidate_tokens": candidate_tokens,
        "selected_tokens": displayed_selected_tokens,
        "predicate_tokens": predicate_tokens,
        "frame_catalog": frame_catalog(scenario),
        "motion_values": motion_values,
    }


print(
    "General semantic inspection v9.3.5 parallel-output mode is ready: "
    "predicate-bearing agents only, heading arrows, and tables below plots."
)


## Selection examples

### Random scenario with reproducible seed; frame by index

```python
RESULT = inspect_spatial_scene(
    scenario_selector="random",
    scenario_seed=42,
    frame_selector="index",
    frame_value=20,
)
```

### Scenario by catalog index; random frame

```python
RESULT = inspect_spatial_scene(
    scenario_selector="index",
    scenario_value=0,
    frame_selector="random",
    frame_seed=7,
)
```

### Scenario token and timestamp

```python
RESULT = inspect_spatial_scene(
    scenario_selector="token",
    scenario_value="YOUR_SCENARIO_TOKEN",
    frame_selector="timestamp",
    frame_value=YOUR_TIMESTAMP_US,
)
```

### Frame by token

First inspect all accepted frame identifiers:

```python
SCENARIO, _ = select_scenario(
    selector="token",
    value="YOUR_SCENARIO_TOKEN",
)
display(frame_catalog(SCENARIO))
```

Then use a native token or the generated observation token:

```python
RESULT = inspect_spatial_scene(
    scenario_selector="token",
    scenario_value="YOUR_SCENARIO_TOKEN",
    frame_selector="token",
    frame_value="observation:YOUR_SCENARIO_TOKEN:YOUR_TIMESTAMP_US",
)
```


## Arrow interpretation

Arrows now follow the exact stored semantic pair direction:

```text
subject → object
```

Therefore, enabling only `ego_to_agent` draws arrows from ego toward agents. Enabling `agent_to_ego` draws the inverse arrows from agents toward ego.

## Plot behavior in this corrected notebook

- No predicate names or edge IDs are written on the plots.
- Purple arrows show only the semantic direction (`SUBJECT → OBJECT`).
- Black arrows show ego and agent headings.
- Stable node IDs such as `EGO`, `A1`, and `A2` stay centered inside the nodes.
- The predicate table is printed below the plots with two columns:
  - `relation`, for example `EGO → A1`
  - `predicate`, for example `frontRightOf, near`
- Only selected agents with displayed predicates are highlighted.

- Temporal value tables use readable `EGO`, `A1`, `A2`, ... labels; raw IDs remain in detailed assertions.


In [ ]:


def follow_evidence_table(frame_assertions: pd.DataFrame) -> pd.DataFrame:
    """Flatten the evidence of current-frame np:follows assertions."""
    columns = [
        "subject_id", "object_id", "valid_time_us", "end_time_us",
        "follow_mode", "condition_duration_s", "condition_frame_count",
        "path_relation", "path_hops", "path_object_ids", "path_object_kinds",
        "path_center_distance_m", "path_bumper_gap_m", "bumper_headway_s",
        "subject_forward_speed_mps", "object_forward_speed_mps",
        "leader_rank", "nearest_leader_unique",
        "second_candidate_center_path_distance_m",
        "subject_agent_type", "object_agent_type",
        "subject_track_token", "object_track_token",
        "subject_primary_map_object_id", "object_primary_map_object_id",
        "subject_primary_map_kind", "object_primary_map_kind",
        "free_space_distance_m", "intersection_area_m2", "overlapping",
        "evidence_json",
    ]
    if frame_assertions is None or frame_assertions.empty:
        return pd.DataFrame(columns=columns)
    follows = frame_assertions.loc[
        frame_assertions["predicate_id"].astype(str).eq("np:follows")
    ].copy()
    rows = []
    for _, row in follows.iterrows():
        evidence = _safe_json_load(row.get("evidence_json"), default={}) or {}
        flattened = {
            "subject_id": row.get("subject_id"),
            "object_id": row.get("object_id"),
            "valid_time_us": row.get("valid_time_us"),
            "end_time_us": row.get("end_time_us"),
            **{key: evidence.get(key) for key in columns if key not in {
                "subject_id", "object_id", "valid_time_us", "end_time_us", "evidence_json"
            }},
            "evidence_json": row.get("evidence_json"),
        }
        for key in ("path_object_ids", "path_object_kinds"):
            if isinstance(flattened.get(key), (list, dict)):
                flattened[key] = json.dumps(flattened[key], ensure_ascii=False)
        rows.append(flattened)
    return pd.DataFrame(rows, columns=columns)

# ============================================================
# 12. VISUAL INSPECTION AND XLSX EXPORT FUNCTIONS
# Run this cell once.
# ============================================================

from pathlib import Path
import json
import re

import pandas as pd
import numpy as np

from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.table import Table, TableStyleInfo


# ============================================================
# BASIC HELPERS
# ============================================================

def safe_string(value):
    """
    Convert a value to a clean string without returning 'nan'.
    """
    if value is None:
        return ""

    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass

    return str(value)


def decode_export_value(value):
    """
    Decode an assertion value for readable export.
    """
    try:
        return _decoded_assertion_value(value)
    except Exception:
        pass

    if value is None:
        return None

    if isinstance(
        value,
        (bool, int, float, list, tuple, dict),
    ):
        return value

    text = safe_string(value).strip()

    if not text:
        return None

    try:
        return json.loads(text)
    except Exception:
        return value


def normalize_predicate_items(value):
    """
    Convert a predicate collection into a list.
    """
    if value is None:
        return []

    if isinstance(value, list):
        return value

    if isinstance(value, tuple):
        return list(value)

    if isinstance(value, set):
        return list(value)

    if isinstance(value, dict):
        return [value]

    if isinstance(value, str):
        text = value.strip()

        if not text:
            return []

        try:
            parsed = json.loads(text)

            if isinstance(parsed, list):
                return parsed

            if isinstance(parsed, dict):
                return [parsed]

        except Exception:
            pass

        return [text]

    return [value]


def normalize_predicate_id(value):
    """
    Normalize predicate IDs for comparison.

    Examples:
        np:near -> near
        near    -> near
    """
    text = safe_string(value).strip()

    if ":" in text:
        return text.split(":")[-1]

    return text


def first_existing_column(
    dataframe,
    candidates,
):
    """
    Return the first candidate column present in a DataFrame.
    """
    if dataframe is None:
        return None

    for column in candidates:
        if column in dataframe.columns:
            return column

    return None


# ============================================================
# PREDICATE METADATA
# ============================================================

def build_predicate_definition_lookup(
    definitions,
):
    """
    Build predicate metadata lookup using both full and short IDs.
    """
    lookup = {}

    if definitions is None or definitions.empty:
        return lookup

    for row in definitions.itertuples(index=False):
        row_data = row._asdict()

        predicate_id = safe_string(
            row_data.get(
                "predicate_id",
                "",
            )
        )

        if not predicate_id:
            continue

        metadata = {
            "predicate_id": predicate_id,

            "category": safe_string(
                row_data.get(
                    "category",
                    "",
                )
            ).strip().lower(),

            "label": row_data.get(
                "label",
                None,
            ),

            "description": row_data.get(
                "description",
                None,
            ),

            "units": row_data.get(
                "units",
                row_data.get(
                    "unit",
                    None,
                ),
            ),
        }

        lookup[predicate_id] = metadata

        lookup.setdefault(
            normalize_predicate_id(
                predicate_id
            ),
            metadata,
        )

    return lookup


# ============================================================
# ENTITY DISPLAY LOOKUP
# ============================================================

def build_entity_display_lookup(
    entity_table,
):
    """
    Map entity IDs and track tokens to EGO, A1, A2, etc.
    """
    lookup = {
        "ego": "EGO",
        "EGO": "EGO",
    }

    if entity_table is None or entity_table.empty:
        return lookup

    try:
        existing_lookup = _display_id_lookup(
            entity_table
        )

        if existing_lookup:
            for key, value in existing_lookup.items():
                key = safe_string(key)
                value = safe_string(value)

                if key and value:
                    lookup[key] = value

    except Exception:
        pass

    display_column = first_existing_column(
        entity_table,
        [
            "display_id",
            "display_name",
            "short_id",
            "plot_id",
        ],
    )

    entity_id_column = first_existing_column(
        entity_table,
        [
            "entity_id",
            "node_id",
            "id",
        ],
    )

    track_token_column = first_existing_column(
        entity_table,
        [
            "track_token",
            "token",
            "object_token",
        ],
    )

    if display_column is None:
        return lookup

    for row in entity_table.itertuples(index=False):
        row_data = row._asdict()

        display_id = safe_string(
            row_data.get(
                display_column,
                "",
            )
        )

        if not display_id:
            continue

        entity_id = (
            safe_string(
                row_data.get(
                    entity_id_column,
                    "",
                )
            )
            if entity_id_column
            else ""
        )

        track_token = (
            safe_string(
                row_data.get(
                    track_token_column,
                    "",
                )
            )
            if track_token_column
            else ""
        )

        if entity_id:
            lookup[entity_id] = display_id

        if track_token:
            lookup[track_token] = display_id

        if entity_id:
            try:
                extracted_token = safe_string(
                    entity_id_to_track_token(
                        entity_id
                    )
                )

                if extracted_token:
                    lookup[
                        extracted_token
                    ] = display_id

            except Exception:
                pass

    return lookup


def readable_entity_label(
    entity_id,
    track_token,
    display_lookup,
):
    """
    Resolve an entity into EGO, A1, A2, etc.
    """
    entity_id = safe_string(
        entity_id
    )

    track_token = safe_string(
        track_token
    )

    for candidate in [
        track_token,
        entity_id,
    ]:
        if not candidate:
            continue

        if candidate.lower() == "ego":
            return "EGO"

        if candidate in display_lookup:
            return display_lookup[
                candidate
            ]

    if entity_id:
        try:
            extracted_token = safe_string(
                entity_id_to_track_token(
                    entity_id
                )
            )

            if extracted_token.lower() == "ego":
                return "EGO"

            if extracted_token in display_lookup:
                return display_lookup[
                    extracted_token
                ]

        except Exception:
            pass

    fallback = (
        track_token
        or entity_id
    )

    try:
        return safe_string(
            short_token(
                fallback
            )
        )
    except Exception:
        return fallback[:12]


# ============================================================
# SEMANTIC EDGE PREPARATION
# ============================================================

def prepare_semantic_edges(
    semantic_edges,
    entity_table,
):
    """
    Add readable subject/object IDs to semantic edges.
    """
    if semantic_edges is None:
        return pd.DataFrame()

    edges = semantic_edges.copy()

    if edges.empty:
        return edges

    display_lookup = build_entity_display_lookup(
        entity_table
    )

    subject_id_column = first_existing_column(
        edges,
        [
            "subject_id",
            "source_id",
            "subject_entity_id",
        ],
    )

    object_id_column = first_existing_column(
        edges,
        [
            "object_id",
            "target_id",
            "object_entity_id",
        ],
    )

    subject_token_column = first_existing_column(
        edges,
        [
            "subject_token",
            "source_token",
        ],
    )

    object_token_column = first_existing_column(
        edges,
        [
            "object_token",
            "target_token",
        ],
    )

    existing_subject_display_column = first_existing_column(
        edges,
        [
            "subject_display_id",
            "subject_display",
            "source_display_id",
            "subject_label",
        ],
    )

    existing_object_display_column = first_existing_column(
        edges,
        [
            "object_display_id",
            "object_display",
            "target_display_id",
            "object_label",
        ],
    )

    def resolve_subject(row):
        if existing_subject_display_column:
            existing_value = safe_string(
                row.get(
                    existing_subject_display_column,
                    "",
                )
            )

            if existing_value:
                return existing_value

        return readable_entity_label(
            entity_id=(
                row.get(
                    subject_id_column,
                    "",
                )
                if subject_id_column
                else ""
            ),

            track_token=(
                row.get(
                    subject_token_column,
                    "",
                )
                if subject_token_column
                else ""
            ),

            display_lookup=display_lookup,
        )

    def resolve_object(row):
        if existing_object_display_column:
            existing_value = safe_string(
                row.get(
                    existing_object_display_column,
                    "",
                )
            )

            if existing_value:
                return existing_value

        return readable_entity_label(
            entity_id=(
                row.get(
                    object_id_column,
                    "",
                )
                if object_id_column
                else ""
            ),

            track_token=(
                row.get(
                    object_token_column,
                    "",
                )
                if object_token_column
                else ""
            ),

            display_lookup=display_lookup,
        )

    edges[
        "subject_display_id"
    ] = edges.apply(
        resolve_subject,
        axis=1,
    )

    edges[
        "object_display_id"
    ] = edges.apply(
        resolve_object,
        axis=1,
    )

    edges[
        "relation_display"
    ] = edges.apply(
        lambda row: (
            safe_string(
                row["subject_display_id"]
            )
            if not safe_string(
                row["object_display_id"]
            )
            else (
                f"{safe_string(row['subject_display_id'])}"
                f" → "
                f"{safe_string(row['object_display_id'])}"
            )
        ),
        axis=1,
    )

    return edges


# ============================================================
# SPATIAL VALUES
# ============================================================

def extract_predicate_from_item(
    predicate_item,
):
    """
    Extract predicate information from one item.
    """
    if isinstance(
        predicate_item,
        dict,
    ):
        predicate_id = safe_string(
            predicate_item.get(
                "predicate_id",
                predicate_item.get(
                    "predicate",
                    predicate_item.get(
                        "id",
                        predicate_item.get(
                            "name",
                            "",
                        ),
                    ),
                ),
            )
        )

        predicate_value = predicate_item.get(
            "value",
            predicate_item.get(
                "value_json",
                True,
            ),
        )

        predicate_unit = predicate_item.get(
            "unit",
            predicate_item.get(
                "units",
                None,
            ),
        )

        predicate_description = predicate_item.get(
            "description",
            None,
        )

    else:
        predicate_id = safe_string(
            predicate_item
        )

        predicate_value = True
        predicate_unit = None
        predicate_description = None

    return (
        predicate_id,
        predicate_value,
        predicate_unit,
        predicate_description,
    )


def spatial_values_from_semantic_edges(
    semantic_edges,
    entity_table,
    definitions,
    keep_tokens=False,
):
    """
    Create the Spatial values table using resolved semantic edges.
    """
    output_columns = [
        "scope",
        "subject",
        "object",
        "relation",
        "predicate",
        "value",
        "unit",
        "description",
        "pair_type",
    ]

    if keep_tokens:
        output_columns.extend([
            "subject_token",
            "object_token",
        ])

    if semantic_edges is None or semantic_edges.empty:
        return pd.DataFrame(
            columns=output_columns
        )

    edges = prepare_semantic_edges(
        semantic_edges=semantic_edges,
        entity_table=entity_table,
    )

    definition_lookup = build_predicate_definition_lookup(
        definitions
    )

    category_column = first_existing_column(
        edges,
        [
            "category",
            "predicate_category",
        ],
    )

    predicate_collection_column = first_existing_column(
        edges,
        [
            "predicates",
            "predicate_ids",
            "predicate_list",
        ],
    )

    single_predicate_column = first_existing_column(
        edges,
        [
            "predicate_id",
            "predicate",
        ],
    )

    relation_type_column = first_existing_column(
        edges,
        [
            "relation_type",
            "pair_type",
            "edge_type",
        ],
    )

    subject_token_column = first_existing_column(
        edges,
        [
            "subject_token",
            "source_token",
        ],
    )

    object_token_column = first_existing_column(
        edges,
        [
            "object_token",
            "target_token",
        ],
    )

    export_rows = []

    for edge in edges.itertuples(
        index=False
    ):
        edge_data = edge._asdict()

        edge_category = (
            safe_string(
                edge_data.get(
                    category_column,
                    "",
                )
            ).strip().lower()
            if category_column
            else ""
        )

        subject_display = safe_string(
            edge_data.get(
                "subject_display_id",
                "",
            )
        )

        object_display = safe_string(
            edge_data.get(
                "object_display_id",
                "",
            )
        )

        relation_display = safe_string(
            edge_data.get(
                "relation_display",
                "",
            )
        )

        relation_type = (
            safe_string(
                edge_data.get(
                    relation_type_column,
                    "",
                )
            )
            if relation_type_column
            else ""
        )

        subject_token = (
            safe_string(
                edge_data.get(
                    subject_token_column,
                    "",
                )
            )
            if subject_token_column
            else ""
        )

        object_token = (
            safe_string(
                edge_data.get(
                    object_token_column,
                    "",
                )
            )
            if object_token_column
            else ""
        )

        if predicate_collection_column:
            predicate_items = normalize_predicate_items(
                edge_data.get(
                    predicate_collection_column,
                    [],
                )
            )

        elif single_predicate_column:
            predicate_items = [{
                "predicate_id": edge_data.get(
                    single_predicate_column,
                    "",
                ),

                "value": edge_data.get(
                    "value",
                    edge_data.get(
                        "value_json",
                        True,
                    ),
                ),

                "unit": edge_data.get(
                    "unit",
                    edge_data.get(
                        "units",
                        None,
                    ),
                ),

                "description": edge_data.get(
                    "description",
                    None,
                ),
            }]

        else:
            predicate_items = []

        for predicate_item in predicate_items:
            (
                predicate_id,
                predicate_value,
                predicate_unit,
                predicate_description,
            ) = extract_predicate_from_item(
                predicate_item
            )

            if not predicate_id:
                continue

            metadata = definition_lookup.get(
                predicate_id,
                definition_lookup.get(
                    normalize_predicate_id(
                        predicate_id
                    ),
                    {},
                ),
            )

            predicate_category = safe_string(
                metadata.get(
                    "category",
                    edge_category,
                )
            ).strip().lower()

            if predicate_category != "spatial":
                continue

            if predicate_unit is None:
                predicate_unit = metadata.get(
                    "units",
                    None,
                )

            if predicate_description is None:
                predicate_description = metadata.get(
                    "description",
                    None,
                )

            row = {
                "scope": "pair",
                "subject": subject_display,
                "object": object_display,
                "relation": relation_display,
                "predicate": predicate_id,
                "value": decode_export_value(
                    predicate_value
                ),
                "unit": predicate_unit,
                "description":
                    predicate_description,
                "pair_type": relation_type,
            }

            if keep_tokens:
                row[
                    "subject_token"
                ] = subject_token

                row[
                    "object_token"
                ] = object_token

            export_rows.append(
                row
            )

    result = pd.DataFrame(
        export_rows,
        columns=output_columns,
    )

    if result.empty:
        return result

    return (
        result
        .drop_duplicates()
        .sort_values(
            by=[
                "subject",
                "object",
                "predicate",
            ],
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )


# ============================================================
# COMPLETE MOTION VALUES
# ============================================================

def complete_motion_values_table(
    frame_assertions,
    entity_table,
    definitions,
    relation_switches,
    selected_agent_tokens=None,
):
    """
    Rebuild the complete Motion values table for exactly the selected frame.

    Pair-member assertions are already contained in ``frame_assertions``.
    The notebook's ``motion_predicate_values_table`` reads both entity-level
    and pairwise motion rows from that selected-frame DataFrame.
    """
    if frame_assertions is None or frame_assertions.empty:
        return pd.DataFrame()

    maximum_rows = max(
        len(frame_assertions) * 2,
        1,
    )

    result = motion_predicate_values_table(
        frame_assertions,
        entity_table,
        definitions,
        relation_switches=relation_switches,
        selected_agent_tokens=selected_agent_tokens,
        max_rows=maximum_rows,
    )

    if result is None:
        return pd.DataFrame()

    return result.copy()


# ============================================================
# COMPLETE TEMPORAL VALUES
# ============================================================

def _strict_assertions_at_selected_timestamp(
    assertions,
    selected_timestamp_us,
):
    """
    Apply the common category-aware selected-frame filter.

    Temporal predicates use end_time_us/current_timestamp_us.
    Spatial, motion, and every non-temporal category still use valid_time_us.
    """
    if assertions is None or assertions.empty:
        return pd.DataFrame(
            columns=getattr(
                assertions,
                "columns",
                None,
            )
        )

    return assertions_at_timestamp(
        assertions,
        int(selected_timestamp_us),
        definitions=DEFINITIONS,
    )



def complete_temporal_values_table(
    frame_assertions,
    scenario_assertions,
    entity_table,
    definitions,
    relation_switches,
    selected_timestamp_us,
    selected_agent_tokens=None,
):
    """
    Export temporal predicates for exactly the selected *actual* frame.

    The generator's temporal assertions can use the preceding timestamp in
    valid_time_us while storing the real current frame in evidence_json as
    current_timestamp_us. Therefore temporal filtering must not rely only on
    valid_time_us.

    Filtering priority:
      1. evidence_json.current_timestamp_us
      2. timestamp embedded in the temporal subject/pair ID
      3. timestamp embedded in evidence subject_id/object_id
      4. valid_time_us only as a final fallback

    Spatial and motion processing are not changed.
    """
    selected_timestamp_us = int(selected_timestamp_us)

    output_columns = [
        "selected_timestamp_us",
        "actual_timestamp_us",
        "valid_time_us",
        "scope",
        "subject",
        "object",
        "relation",
        "predicate",
        "value",
        "unit",
        "description",
        "subject_id",
        "object_id",
    ]

    assertion_sources = []
    if frame_assertions is not None and not frame_assertions.empty:
        assertion_sources.append(frame_assertions)
    if scenario_assertions is not None and not scenario_assertions.empty:
        assertion_sources.append(scenario_assertions)

    if not assertion_sources:
        return pd.DataFrame(columns=output_columns)

    all_assertions = pd.concat(
        assertion_sources,
        ignore_index=True,
        sort=False,
    )

    temporal_ids = set(
        definitions.loc[
            definitions["category"].astype(str).str.lower().eq("temporal"),
            "predicate_id",
        ].astype(str)
    )

    temporal_rows = all_assertions.loc[
        all_assertions["predicate_id"].astype(str).isin(temporal_ids)
    ].copy()

    if temporal_rows.empty:
        return pd.DataFrame(columns=output_columns)

    def _parse_evidence(value):
        if isinstance(value, dict):
            return value
        if value is None:
            return {}
        try:
            if pd.isna(value):
                return {}
        except (TypeError, ValueError):
            pass
        if isinstance(value, str):
            text = value.strip()
            if not text:
                return {}
            try:
                parsed = json.loads(text)
                return parsed if isinstance(parsed, dict) else {}
            except (json.JSONDecodeError, TypeError):
                return {}
        return {}

    def _timestamp_from_identifier(identifier):
        text = safe_string(identifier)
        if not text:
            return None

        # nuPlan entity/pair IDs contain microsecond timestamps as standalone
        # colon-delimited numeric components.
        candidates = re.findall(r"(?<!\d)(\d{13,17})(?!\d)", text)
        for candidate in candidates:
            try:
                return int(candidate)
            except (TypeError, ValueError):
                continue
        return None

    def _actual_timestamp(row):
        end_time = pd.to_numeric(
            getattr(row, "end_time_us", None),
            errors="coerce",
        )
        if not pd.isna(end_time):
            return int(end_time)

        evidence = _parse_evidence(getattr(row, "evidence_json", None))

        evidence_current = pd.to_numeric(
            evidence.get("current_timestamp_us"),
            errors="coerce",
        )
        if not pd.isna(evidence_current):
            return int(evidence_current)

        timestamp = _timestamp_from_identifier(
            getattr(row, "subject_id", None)
        )
        if timestamp is not None:
            return timestamp

        timestamp = _timestamp_from_identifier(
            evidence.get("subject_id")
        )
        if timestamp is not None:
            return timestamp

        timestamp = _timestamp_from_identifier(
            evidence.get("object_id")
        )
        if timestamp is not None:
            return timestamp

        fallback = pd.to_numeric(
            getattr(row, "valid_time_us", None),
            errors="coerce",
        )
        if not pd.isna(fallback):
            return int(fallback)

        return None

    temporal_rows["actual_timestamp_us_internal"] = [
        _actual_timestamp(row)
        for row in temporal_rows.itertuples(index=False)
    ]

    # This is the decisive fix: select by actual current frame, not by the
    # generator's valid_time_us, which may point to the preceding frame.
    temporal_rows = temporal_rows.loc[
        temporal_rows["actual_timestamp_us_internal"].eq(selected_timestamp_us)
    ].copy()

    if temporal_rows.empty:
        return pd.DataFrame(columns=output_columns)

    metadata_columns = [
        column
        for column in ["predicate_id", "label", "units", "description"]
        if column in definitions.columns
    ]
    definition_metadata = (
        definitions[metadata_columns]
        .drop_duplicates(subset=["predicate_id"], keep="first")
        .copy()
    )
    temporal_rows = temporal_rows.merge(
        definition_metadata,
        on="predicate_id",
        how="left",
        suffixes=("", "_definition"),
    )

    display_lookup = build_entity_display_lookup(entity_table)

    enabled_relations = (
        enabled_names(relation_switches)
        if relation_switches is not None
        else set(ALL_SEMANTIC_RELATIONS)
    )

    selected_tokens = (
        set(map(str, selected_agent_tokens))
        if selected_agent_tokens is not None
        else None
    )

    result_rows = []

    for row in temporal_rows.itertuples(index=False):
        predicate_id = safe_string(getattr(row, "predicate_id", ""))
        evidence = _parse_evidence(getattr(row, "evidence_json", None))

        raw_assertion_subject = safe_string(
            getattr(row, "subject_id", "")
        )
        raw_assertion_object = safe_string(
            getattr(row, "object_id", "")
        )

        evidence_subject_id = safe_string(
            evidence.get("subject_id")
        )
        evidence_object_id = safe_string(
            evidence.get("object_id")
        )

        # Pair predicates use a pair-state as assertion subject. Their real
        # members are stored in evidence_json.
        is_pair = bool(
            evidence_subject_id
            and evidence_object_id
            and (
                raw_assertion_subject.startswith("pair:")
                or safe_string(getattr(row, "object", ""))
                or safe_string(getattr(row, "relation", "")).count("→") >= 1
            )
        )

        if is_pair:
            member_subject_id = evidence_subject_id
            member_object_id = evidence_object_id

            member_subject_token = safe_string(
                evidence.get("subject_track_token")
                or entity_id_to_track_token(member_subject_id)
            )
            member_object_token = safe_string(
                evidence.get("object_track_token")
                or entity_id_to_track_token(member_object_id)
            )

            relation_hint = safe_string(
                evidence.get("pair_tag")
                or evidence.get("relation_type")
                or getattr(row, "relation", "")
            )
            relation_type = classify_semantic_relation(
                member_subject_id,
                member_object_id,
                relation_hint,
            )
            if relation_type not in enabled_relations:
                continue

            if selected_tokens is not None:
                non_ego_tokens = {
                    token
                    for token in (
                        member_subject_token,
                        member_object_token,
                    )
                    if token and token.lower() != "ego"
                }
                if (
                    non_ego_tokens
                    and not non_ego_tokens.issubset(selected_tokens)
                ):
                    continue

            subject_label = readable_entity_label(
                member_subject_id,
                member_subject_token,
                display_lookup,
            )
            object_label = readable_entity_label(
                member_object_id,
                member_object_token,
                display_lookup,
            )

            scope = "pair"
            relation = (
                f"{subject_label} → "
                f"{normalize_predicate_id(predicate_id)} → "
                f"{object_label}"
            )
            exported_subject_id = member_subject_id
            exported_object_id = member_object_id

        else:
            entity_id = evidence_subject_id or raw_assertion_subject
            entity_token = safe_string(
                evidence.get("track_key")
                or evidence.get("subject_track_token")
                or entity_id_to_track_token(entity_id)
            )

            if (
                selected_tokens is not None
                and entity_token.lower() != "ego"
                and entity_token not in selected_tokens
            ):
                continue

            subject_label = readable_entity_label(
                entity_id,
                entity_token,
                display_lookup,
            )
            object_label = ""
            scope = "entity"
            relation = (
                f"{subject_label} → "
                f"{normalize_predicate_id(predicate_id)}"
            )
            exported_subject_id = entity_id
            exported_object_id = raw_assertion_object

        value = decode_export_value(
            getattr(row, "value_json", None)
        )

        unit = getattr(row, "units", None)
        if unit is None:
            unit = getattr(row, "units_definition", None)

        description = getattr(row, "description", None)
        if description is None:
            description = getattr(row, "description_definition", None)

        valid_time = pd.to_numeric(
            getattr(row, "valid_time_us", None),
            errors="coerce",
        )

        result_rows.append({
            "selected_timestamp_us": selected_timestamp_us,
            "actual_timestamp_us": int(
                getattr(row, "actual_timestamp_us_internal")
            ),
            "valid_time_us": (
                int(valid_time)
                if not pd.isna(valid_time)
                else None
            ),
            "scope": scope,
            "subject": subject_label,
            "object": object_label,
            "relation": relation,
            "predicate": predicate_id,
            "value": value,
            "unit": unit,
            "description": description,
            "subject_id": exported_subject_id,
            "object_id": exported_object_id,
        })

    if not result_rows:
        return pd.DataFrame(columns=output_columns)

    result = pd.DataFrame(result_rows, columns=output_columns)

    def _temporal_key(value):
        if isinstance(value, np.ndarray):
            value = value.tolist()
        if isinstance(value, np.generic):
            value = value.item()
        if isinstance(value, dict):
            return json.dumps(
                value,
                sort_keys=True,
                ensure_ascii=False,
                default=str,
            )
        if isinstance(value, (list, tuple, set)):
            return json.dumps(
                list(value),
                ensure_ascii=False,
                default=str,
            )
        try:
            if pd.isna(value):
                return "<NA>"
        except (TypeError, ValueError):
            pass
        return value

    dedup_columns = [
        "actual_timestamp_us",
        "scope",
        "subject_id",
        "object_id",
        "predicate",
        "value",
    ]
    dedup_keys = result[dedup_columns].apply(
        lambda column: column.map(_temporal_key)
    )

    result = result.loc[
        ~dedup_keys.duplicated(keep="first")
    ].copy()

    return (
        result.sort_values(
            by=[
                "scope",
                "subject",
                "object",
                "predicate",
            ],
            kind="stable",
        )
        .reset_index(drop=True)
    )

def add_readable_entity_labels(
    table,
    entity_table,
    frame_assertions,
):
    """
    Add readable_subject/readable_object columns using EGO/A1/A2.
    Raw subject_id/object_id columns are preserved for traceability.
    Pair-state subjects are expanded to their ordered members.
    """
    if table is None or table.empty:
        return table

    result = table.copy()
    display_lookup = _display_id_lookup(entity_table)

    pair_source = frame_assertions
    pairs = pair_member_table(pair_source)
    pair_lookup = {}
    if pairs is not None and not pairs.empty:
        for pair in pairs.itertuples(index=False):
            pair_lookup[str(pair.pair_id)] = (
                str(pair.subject_id),
                str(pair.object_id),
            )

    def label_entity(raw_id):
        raw_id = safe_string(raw_id)
        token = entity_id_to_track_token(raw_id)
        return display_lookup.get(
            raw_id,
            display_lookup.get(token, short_token(token)),
        )

    readable_subject = []
    readable_object = []
    readable_relation = []

    for row in result.itertuples(index=False):
        raw_subject = safe_string(getattr(row, "subject_id", ""))
        raw_object = safe_string(getattr(row, "object_id", ""))

        if raw_subject in pair_lookup:
            member_subject, member_object = pair_lookup[raw_subject]
            subject_label = label_entity(member_subject)
            object_label = label_entity(member_object)
        else:
            subject_label = label_entity(raw_subject) if raw_subject else ""
            object_label = label_entity(raw_object) if raw_object else ""

        readable_subject.append(subject_label)
        readable_object.append(object_label)
        readable_relation.append(
            f"{subject_label} → {object_label}"
            if object_label else subject_label
        )

    result.insert(0, "relation", readable_relation)
    result.insert(1, "subject", readable_subject)
    result.insert(2, "object", readable_object)
    return result

def temporal_predicate_agent_tokens(
    frame_assertions,
    scenario_assertions,
    definitions,
    selected_timestamp_us,
):
    """
    Return agents participating in temporal assertions at exactly one frame.

    Pair membership and temporal values are both selected with the common
    endpoint-aware filter. This prevents future temporal observations from
    highlighting or connecting the final agent of the selected frame.
    """
    if (
        frame_assertions is None
        or frame_assertions.empty
        or "predicate_id" not in frame_assertions.columns
        or definitions is None
        or definitions.empty
        or "category" not in definitions.columns
        or "predicate_id" not in definitions.columns
    ):
        return set()

    selected_timestamp_us = int(
        selected_timestamp_us
    )

    selected_frame_assertions = (
        assertions_at_timestamp(
            frame_assertions,
            selected_timestamp_us,
            definitions=definitions,
        )
    )

    selected_pair_context = (
        assertions_at_timestamp(
            scenario_assertions,
            selected_timestamp_us,
            definitions=definitions,
        )
    )

    temporal_ids = set(
        definitions.loc[
            definitions["category"]
            .astype(str)
            .str.lower()
            .eq("temporal"),
            "predicate_id",
        ].astype(str)
    )

    temporal_rows = selected_frame_assertions.loc[
        selected_frame_assertions[
            "predicate_id"
        ]
        .astype(str)
        .isin(temporal_ids)
    ].copy()

    if temporal_rows.empty:
        return set()

    pairs = pair_member_table(
        selected_pair_context
    )

    pair_lookup = {}

    if pairs is not None and not pairs.empty:
        for pair in pairs.itertuples(
            index=False
        ):
            pair_lookup[
                str(pair.pair_id)
            ] = (
                str(pair.subject_id),
                str(pair.object_id),
            )

    tokens = set()

    for row in temporal_rows.itertuples(
        index=False
    ):
        subject_id = safe_string(
            getattr(row, "subject_id", "")
        )

        object_id = safe_string(
            getattr(row, "object_id", "")
        )

        if subject_id in pair_lookup:
            member_subject, member_object = (
                pair_lookup[subject_id]
            )

            for entity_id in (
                member_subject,
                member_object,
            ):
                token = entity_id_to_track_token(
                    entity_id
                )

                if token and token != "ego":
                    tokens.add(token)

        else:
            token = entity_id_to_track_token(
                subject_id
            )

            if token and token != "ego":
                tokens.add(token)

            if object_id.startswith(
                ("ego:", "agent:")
            ):
                token = entity_id_to_track_token(
                    object_id
                )

                if token and token != "ego":
                    tokens.add(token)

    return tokens


def selected_tokens_from_edges(
    semantic_edges,
):
    """
    Obtain all agent tokens participating in the selected edges.
    """
    if semantic_edges is None or semantic_edges.empty:
        return None

    tokens = set()

    for column in [
        "subject_token",
        "object_token",
    ]:
        if column not in semantic_edges.columns:
            continue

        tokens.update(
            semantic_edges[
                column
            ]
            .dropna()
            .astype(str)
            .tolist()
        )

    tokens.discard("")
    tokens.discard("ego")
    tokens.discard("EGO")

    return (
        tokens
        if tokens
        else None
    )


# ============================================================
# EXCEL FORMATTING
# ============================================================

def sanitize_excel_table_name(
    sheet_name,
    sheet_index,
):
    """
    Create a valid Excel table name.
    """
    cleaned = re.sub(
        r"[^A-Za-z0-9_]",
        "",
        safe_string(
            sheet_name
        ),
    )

    if not cleaned:
        cleaned = f"Sheet{sheet_index}"

    if cleaned[0].isdigit():
        cleaned = f"T{cleaned}"

    return f"Table{cleaned}"[:250]


def format_excel_workbook(
    excel_path,
):
    """
    Format all workbook sheets.
    """
    workbook = load_workbook(
        excel_path
    )

    used_table_names = set()

    for sheet_index, worksheet in enumerate(
        workbook.worksheets,
        start=1,
    ):
        if worksheet.max_row < 1:
            continue

        if worksheet.max_column < 1:
            continue

        worksheet.freeze_panes = "A2"
        worksheet.auto_filter.ref = (
            worksheet.dimensions
        )

        for cell in worksheet[1]:
            cell.font = Font(
                bold=True,
            )

            cell.fill = PatternFill(
                fill_type="solid",
                fgColor="D9EAF7",
            )

            cell.alignment = Alignment(
                horizontal="center",
                vertical="center",
                wrap_text=True,
            )

        worksheet.row_dimensions[1].height = 30

        for row in worksheet.iter_rows(
            min_row=2,
        ):
            for cell in row:
                cell.alignment = Alignment(
                    vertical="top",
                    wrap_text=True,
                )

                if isinstance(
                    cell.value,
                    float,
                ):
                    cell.number_format = (
                        "0.000000"
                    )

        for column_index in range(
            1,
            worksheet.max_column + 1,
        ):
            column_letter = get_column_letter(
                column_index
            )

            maximum_length = 0

            for cell in worksheet[
                column_letter
            ]:
                text = (
                    ""
                    if cell.value is None
                    else str(cell.value)
                )

                maximum_length = max(
                    maximum_length,
                    min(
                        len(text),
                        80,
                    ),
                )

            worksheet.column_dimensions[
                column_letter
            ].width = min(
                max(
                    maximum_length + 2,
                    12,
                ),
                60,
            )

        if worksheet.max_row >= 2:
            base_name = sanitize_excel_table_name(
                worksheet.title,
                sheet_index,
            )

            table_name = base_name
            suffix = 1

            while table_name in used_table_names:
                suffix += 1
                table_name = (
                    f"{base_name}{suffix}"
                )[:250]

            used_table_names.add(
                table_name
            )

            excel_table = Table(
                displayName=table_name,
                ref=worksheet.dimensions,
            )

            excel_table.tableStyleInfo = TableStyleInfo(
                name="TableStyleMedium2",
                showFirstColumn=False,
                showLastColumn=False,
                showRowStripes=True,
                showColumnStripes=False,
            )

            worksheet.add_table(
                excel_table
            )

    workbook.save(
        excel_path
    )


def save_inspection_workbook(
    excel_path,
    tables,
):
    """
    Save all tables in one formatted XLSX workbook.
    """
    excel_path = Path(
        excel_path
    )

    excel_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with pd.ExcelWriter(
        excel_path,
        engine="openpyxl",
    ) as writer:

        for sheet_name, table in tables.items():
            if table is None:
                table = pd.DataFrame()

            if (
                table.empty
                and len(table.columns) == 0
            ):
                table = pd.DataFrame({
                    "message": [
                        "No rows are available for this selection."
                    ]
                })

            table.to_excel(
                writer,
                sheet_name=safe_string(
                    sheet_name
                )[:31],
                index=False,
            )

    format_excel_workbook(
        excel_path
    )

    return excel_path


# ============================================================
# MAIN INSPECTION FUNCTION
# ============================================================

def run_and_export_semantic_inspection(
    *,
    semantic_categories,
    semantic_relations,
    agent_types,

    scenario_selector="random",
    scenario_value=0,
    scenario_seed=42,
    filter_seed=42,

    frame_selector="index",
    frame_value=0,
    frame_seed=42,

    export_root=None,
    save_xlsx=True,
    show_tables_in_notebook=True,
    keep_spatial_tokens=False,

    max_saved_assertion_rows=None,
    max_notebook_assertion_rows=200,

    map_radius_m=40.0,

    interested_distance_threshold_m=40.0,
    include_within_distance=True,

    include_forward_corridor=True,
    forward_corridor_length_m=40.0,
    forward_corridor_half_width_m=5.0,

    include_same_lane=True,

    include_predicted_path_intersection=True,
    prediction_horizon_s=5.0,
    prediction_step_s=0.25,
    path_intersection_clearance_m=3.0,

    include_existing_spatial_agents=False,
    filter_edges_to_selected_agents=True,

    manual_highlight_tokens=None,

    include_all_agents_in_graph=False,
    draw_semantic_arrows_on_map=True,
    show_agent_labels=True,
    show_edge_ids=False,
    show_heading_arrows=True,

    max_edge_legend_rows=None,
):
    """
    Run visual inspection and save complete readable tables.
    """
    if manual_highlight_tokens is None:
        manual_highlight_tokens = []

    active_categories = {
        category
        for category, enabled in semantic_categories.items()
        if enabled
    }

    if not active_categories:
        raise ValueError(
            "No category is enabled in semantic_categories."
        )

    # --------------------------------------------------------
    # Run original inspection.
    # The notebook table display may remain limited.
    # --------------------------------------------------------

    result = inspect_semantic_scene(
        semantic_categories=semantic_categories,
        semantic_relations=semantic_relations,
        agent_types=agent_types,

        scenario_selector=scenario_selector,
        scenario_value=scenario_value,
        scenario_seed=scenario_seed,
        filter_seed=filter_seed,

        frame_selector=frame_selector,
        frame_value=frame_value,
        frame_seed=frame_seed,

        map_radius_m=map_radius_m,

        interested_distance_threshold_m=(
            interested_distance_threshold_m
        ),
        include_within_distance=(
            include_within_distance
        ),

        include_forward_corridor=(
            include_forward_corridor
        ),
        forward_corridor_length_m=(
            forward_corridor_length_m
        ),
        forward_corridor_half_width_m=(
            forward_corridor_half_width_m
        ),

        include_same_lane=(
            include_same_lane
        ),

        include_predicted_path_intersection=(
            include_predicted_path_intersection
        ),
        prediction_horizon_s=(
            prediction_horizon_s
        ),
        prediction_step_s=(
            prediction_step_s
        ),
        path_intersection_clearance_m=(
            path_intersection_clearance_m
        ),

        include_existing_spatial_agents=(
            include_existing_spatial_agents
        ),

        filter_edges_to_selected_agents=(
            filter_edges_to_selected_agents
        ),

        manual_highlight_tokens=(
            manual_highlight_tokens
        ),

        include_all_agents_in_graph=(
            include_all_agents_in_graph
        ),
        draw_semantic_arrows_on_map=(
            draw_semantic_arrows_on_map
        ),
        show_agent_labels=(
            show_agent_labels
        ),
        show_edge_ids=(
            show_edge_ids
        ),
        show_heading_arrows=(
            show_heading_arrows
        ),

        show_tables=show_tables_in_notebook,
        max_edge_legend_rows=(
            max_edge_legend_rows
        ),

        # This limit affects notebook display only.
        max_assertion_rows=(
            max_notebook_assertion_rows
        ),
    )

    required_keys = {
        "frame_assertions",
        "scenario_assertions",
        "entity_table",
    }

    missing_keys = (
        required_keys
        - set(result.keys())
    )

    if missing_keys:
        raise KeyError(
            "inspect_semantic_scene() did not return required "
            f"keys: {sorted(missing_keys)}. "
            f"Available keys: {sorted(result.keys())}"
        )

    frame_assertions = (
        result["frame_assertions"]
        .copy()
    )

    scenario_assertions = (
        result["scenario_assertions"]
        .copy()
    )

    entity_table = (
        result["entity_table"]
        .copy()
    )

    entity_table_export = (
        entity_table
        .drop(
            columns=["source_object"],
            errors="ignore",
        )
        .copy()
    )

    semantic_edges = (
        result.get(
            "semantic_edges",
            pd.DataFrame(),
        )
        .copy()
    )

    semantic_edges_export = prepare_semantic_edges(
        semantic_edges=semantic_edges,
        entity_table=entity_table,
    )

    # --------------------------------------------------------
    # Rebuild complete Motion values.
    # This is independent of max_notebook_assertion_rows.
    # --------------------------------------------------------

    selected_agent_tokens = None

    if filter_edges_to_selected_agents:
        selected_agent_tokens = selected_tokens_from_edges(
            semantic_edges_export
        )

    motion_values = pd.DataFrame()

    if "motion" in active_categories:
        motion_values = complete_motion_values_table(
            frame_assertions=frame_assertions,
            entity_table=entity_table,
            definitions=DEFINITIONS,
            relation_switches=semantic_relations,
            selected_agent_tokens=selected_agent_tokens,
        )

    temporal_values = pd.DataFrame()

    if "temporal" in active_categories:
        temporal_values = complete_temporal_values_table(
            frame_assertions=frame_assertions,
            scenario_assertions=scenario_assertions,
            entity_table=entity_table,
            definitions=DEFINITIONS,
            relation_switches=semantic_relations,
            selected_timestamp_us=int(
                result.get(
                    "timestamp_us",
                    result.get("metadata", {}).get(
                        "timestamp_us",
                        int(result["frame"].timestamp_us),
                    ),
                )
            ),
            selected_agent_tokens=selected_agent_tokens,
        )

    frame_predicate_counts = predicate_count_table(
        frame_assertions,
        categories=active_categories,
    )

    scenario_predicate_counts = predicate_count_table(
        scenario_assertions,
        categories=active_categories,
    )

    if max_saved_assertion_rows is None:
        detailed_row_limit = len(
            frame_assertions
        )
    else:
        detailed_row_limit = int(
            max_saved_assertion_rows
        )

    detailed_assertions = detailed_assertion_table(
        frame_assertions,
        categories=active_categories,
        max_rows=detailed_row_limit,
    )

    detailed_assertions = add_readable_entity_labels(
        detailed_assertions,
        entity_table=entity_table,
        frame_assertions=scenario_assertions,
    )

    follow_values = pd.DataFrame()
    if "interaction" in active_categories:
        follow_values = follow_evidence_table(frame_assertions)

    spatial_values = pd.DataFrame()

    if "spatial" in active_categories:
        spatial_values = spatial_values_from_semantic_edges(
            semantic_edges=semantic_edges_export,
            entity_table=entity_table,
            definitions=DEFINITIONS,
            keep_tokens=keep_spatial_tokens,
        )

    tables = {
        "Frame predicate counts":
            frame_predicate_counts,

        "Scenario counts":
            scenario_predicate_counts,

        "Detailed assertions":
            detailed_assertions,

        "Semantic edges":
            semantic_edges_export,

        "Entities":
            entity_table_export,
    }

    if "motion" in active_categories:
        tables[
            "Motion values"
        ] = motion_values

    if "temporal" in active_categories:
        tables[
            "Temporal values"
        ] = temporal_values

    if "spatial" in active_categories:
        tables[
            "Spatial values"
        ] = spatial_values

    if "interaction" in active_categories:
        tables[
            "Follow evidence"
        ] = follow_values

    active_category_name = "_".join(
        sorted(active_categories)
    )

    if export_root is None:
        export_root = (
            Path.cwd()
            / "visual_inspection_exports"
        )
    else:
        export_root = Path(
            export_root
        )

    output_directory = (
        export_root
        / "inspection_tables"
        / active_category_name
        / (
            f"scenario_{scenario_selector}"
            f"_value_{scenario_value}"
            f"_seed_{scenario_seed}"
            f"_frame_{frame_selector}"
            f"_value_{frame_value}"
            f"_seed_{frame_seed}"
        )
    )

    excel_path = (
        output_directory
        / "scene_inspection_tables.xlsx"
    )

    # --------------------------------------------------------
    # Display the complete exported tables.
    # --------------------------------------------------------

    if show_tables_in_notebook:
        if "motion" in active_categories:
            print(
                "\nCOMPLETE MOTION PREDICATE VALUES"
            )

            if motion_values.empty:
                print(
                    "No motion predicate values are available."
                )
            else:
                display(
                    motion_values
                )

                if "scope" in motion_values.columns:
                    print(
                        "\nMotion row counts by scope:"
                    )

                    print(
                        motion_values[
                            "scope"
                        ]
                        .value_counts(
                            dropna=False
                        )
                        .to_string()
                    )

        if "temporal" in active_categories:
            print(
                "\nCOMPLETE TEMPORAL PREDICATE VALUES"
            )

            if temporal_values.empty:
                print(
                    "No temporal predicate values are available."
                )
            else:
                display(temporal_values)

                if "scope" in temporal_values.columns:
                    print(
                        "\nTemporal row counts by scope:"
                    )
                    print(
                        temporal_values[
                            "scope"
                        ]
                        .value_counts(
                            dropna=False
                        )
                        .to_string()
                    )

        if "spatial" in active_categories:
            print(
                "\nSPATIAL PREDICATE VALUES"
            )

            if spatial_values.empty:
                print(
                    "No spatial predicate values are available."
                )
            else:
                display(
                    spatial_values
                )

    if save_xlsx:
        save_inspection_workbook(
            excel_path=excel_path,
            tables=tables,
        )

        if show_tables_in_notebook:
            print(
                "\nXLSX SAVED SUCCESSFULLY"
            )

            print(
                f"File: {excel_path.resolve()}"
            )

            print(
                "\nSheets saved:"
            )

            for sheet_name, table in tables.items():
                print(
                    f"  - {sheet_name}: "
                    f"{len(table):,} rows"
                )

            if (
                "motion" in active_categories
                and not motion_values.empty
                and "scope" in motion_values.columns
            ):
                entity_motion_count = int(
                    motion_values[
                        "scope"
                    ]
                    .astype(str)
                    .str.lower()
                    .eq("entity")
                    .sum()
                )

                pair_motion_count = int(
                    motion_values[
                        "scope"
                    ]
                    .astype(str)
                    .str.lower()
                    .isin([
                        "pair",
                        "pairwise",
                    ])
                    .sum()
                )

                print(
                    "\nMotion values exported:"
                )

                print(
                    f"  - Entity rows: "
                    f"{entity_motion_count:,}"
                )

                print(
                    f"  - Pair rows: "
                    f"{pair_motion_count:,}"
                )

                if pair_motion_count == 0:
                    print(
                        "\nWARNING: No pairwise motion rows were found."
                    )

                    print(
                        "Check that at least one pair relation is enabled, "
                        "for example ego_to_agent=True."
                    )

    result[
        "active_categories"
    ] = active_categories

    result[
        "frame_predicate_counts"
    ] = frame_predicate_counts

    result[
        "scenario_predicate_counts"
    ] = scenario_predicate_counts

    result[
        "detailed_assertions_export"
    ] = detailed_assertions

    result[
        "semantic_edges_export"
    ] = semantic_edges_export

    result[
        "motion_values_export"
    ] = motion_values
    result[
        "temporal_values_export"
    ] = temporal_values

    result[
        "spatial_values_export"
    ] = spatial_values

    result[
        "excel_tables"
    ] = tables

    result[
        "excel_path"
    ] = excel_path

    return result

## Two-stage master Excel workflow

1. **LOAD AND EXPORT ALL TRUE** reads the assertion Parquet data once and saves one complete Excel workbook with every semantic category, relation direction, and agent type enabled. No selected-agent edge filtering is applied to this master workbook.

2. **READ EXCEL, CHANGE CONFIGURATION, AND REPLOT** reloads that workbook, applies the current `True`/`False` plotting configuration, recalculates interested agents, and regenerates only the map, graph, legend, and small filtered tables. It does not reload assertion Parquet files and does not rewrite the workbook.

Rerun Stage 1 only for scenario, frame, seed, source-output, or export-path changes. Rerun Stage 2 for all plotting and filtering changes.

In [ ]:
# ============================================================
# 13. MASTER-EXCEL GENERATION AND EXCEL-BASED REPLOTTING
# Run this helper cell once after the function-definition cells.
#
# Workflow:
#   1. LOAD AND EXPORT ALL TRUE:
#      Reads the assertion Parquet files and writes one complete workbook.
#   2. READ EXCEL, FILTER, AND PLOT:
#      Reads that workbook, applies the current True/False configuration,
#      and creates only the plots and small filtered tables.
#
# The second stage never calls load_assertions_for_scenario() and never
# calls run_and_export_semantic_inspection().
# ============================================================

import ast as _python_ast
from contextlib import contextmanager
from datetime import datetime, timezone


MASTER_EXCEL_FORMAT_VERSION = "working_v8_all_true_master_v1"

ALL_TRUE_SEMANTIC_CATEGORIES = {
    name: True
    for name in ALL_SEMANTIC_CATEGORIES
}

ALL_TRUE_SEMANTIC_RELATIONS = {
    name: True
    for name in ALL_SEMANTIC_RELATIONS
}

ALL_TRUE_AGENT_TYPES = {
    "vehicle": True,
    "car": True,
    "truck": True,
    "bus": True,
    "trailer": True,
    "construction_vehicle": True,
    "pedestrian": True,
    "bicycle": True,
    "motorcycle": True,
    "traffic_cone": True,
    "barrier": True,
    "czone_sign": True,
    "generic_object": True,
    "unknown": True,
}


def _excel_safe_value(value):
    """Convert one metadata value to an Excel-safe scalar."""
    if value is None:
        return None

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, (str, int, float, bool)):
        return value

    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    if isinstance(value, (dict, list, tuple, set)):
        return json.dumps(
            value,
            ensure_ascii=False,
            sort_keys=isinstance(value, dict),
            default=str,
        )

    return str(value)


@contextmanager
def _suppress_master_generation_figures():
    """
    Keep Stage 1 focused on workbook generation.

    The original working function still constructs its two figures internally.
    This context prevents them from being displayed and closes only figures
    created during Stage 1.
    """
    existing_figure_numbers = set(plt.get_fignums())
    original_show = plt.show

    def _hidden_show(*args, **kwargs):
        return None

    plt.show = _hidden_show
    try:
        yield
    finally:
        plt.show = original_show
        new_figure_numbers = (
            set(plt.get_fignums())
            - existing_figure_numbers
        )
        for figure_number in new_figure_numbers:
            plt.close(figure_number)


def generate_complete_master_excel(
    *,
    scenario_selector="random",
    scenario_value=0,
    scenario_seed=42,
    filter_seed=42,
    frame_selector="index",
    frame_value=5,
    frame_seed=42,
    export_root=None,
):
    """
    Read the assertion Parquet data once and save one complete workbook.

    The export is deliberately generated with:
      - every semantic category = True;
      - every semantic relation direction = True;
      - every agent type = True;
      - no selected-agent edge removal;
      - complete assertion rows;
      - complete token columns needed for later Excel-based plotting.
    """
    if export_root is None:
        export_root = (
            Path.cwd()
            / "visual_inspection_exports_master"
        )
    export_root = Path(export_root)

    print("=" * 72)
    print("LOAD AND EXPORT ALL TRUE")
    print("=" * 72)
    print("Scenario selector:", scenario_selector)
    print("Scenario value:", scenario_value)
    print("Frame selector:", frame_selector)
    print("Frame value:", frame_value)
    print("Source assertion output:", OUTPUT_DIR)
    print("Master export root:", export_root)

    with _suppress_master_generation_figures():
        master_result = run_and_export_semantic_inspection(
            semantic_categories=ALL_TRUE_SEMANTIC_CATEGORIES,
            semantic_relations=ALL_TRUE_SEMANTIC_RELATIONS,
            agent_types=ALL_TRUE_AGENT_TYPES,

            scenario_selector=scenario_selector,
            scenario_value=scenario_value,
            scenario_seed=scenario_seed,
            filter_seed=filter_seed,

            frame_selector=frame_selector,
            frame_value=frame_value,
            frame_seed=frame_seed,

            export_root=export_root,

            # Save once below, after adding workbook metadata.
            save_xlsx=False,
            show_tables_in_notebook=False,

            # Required for reliable filtering after Excel reload.
            keep_spatial_tokens=True,

            # Preserve all exported assertions.
            max_saved_assertion_rows=None,
            max_notebook_assertion_rows=0,

            # Interested-agent settings do not restrict the master workbook.
            map_radius_m=40.0,
            interested_distance_threshold_m=float("inf"),
            include_within_distance=False,

            include_forward_corridor=False,
            forward_corridor_length_m=0.0,
            forward_corridor_half_width_m=0.0,

            include_same_lane=False,

            include_predicted_path_intersection=False,
            prediction_horizon_s=5.0,
            prediction_step_s=0.25,
            path_intersection_clearance_m=3.0,

            include_existing_spatial_agents=False,
            filter_edges_to_selected_agents=False,
            manual_highlight_tokens=[],

            include_all_agents_in_graph=True,
            draw_semantic_arrows_on_map=False,
            show_agent_labels=False,
            show_edge_ids=False,
            show_heading_arrows=False,

            max_edge_legend_rows=0,
        )

    metadata = dict(
        master_result.get("metadata", {})
    )

    metadata_row = {
        "excel_format_version":
            MASTER_EXCEL_FORMAT_VERSION,
        "generated_utc":
            datetime.now(timezone.utc).isoformat(),
        "source_output_dir":
            str(OUTPUT_DIR),
        "scenario_selector":
            scenario_selector,
        "scenario_value":
            scenario_value,
        "scenario_seed":
            scenario_seed,
        "filter_seed":
            filter_seed,
        "frame_selector":
            frame_selector,
        "frame_value":
            frame_value,
        "frame_seed":
            frame_seed,
        "scenario_token":
            metadata.get("scenario_token"),
        "scenario_type":
            metadata.get("scenario_type"),
        "log_name":
            metadata.get("log_name"),
        "map_name":
            metadata.get("map_name"),
        "frame_index":
            metadata.get("frame_index"),
        "timestamp_us":
            metadata.get("timestamp_us"),
        "all_categories_true":
            True,
        "all_relations_true":
            True,
        "all_agent_types_true":
            True,
        "filter_edges_to_selected_agents":
            False,
    }

    metadata_table = pd.DataFrame([
        {
            key: _excel_safe_value(value)
            for key, value in metadata_row.items()
        }
    ])

    settings_rows = []

    for group_name, switches in [
        (
            "semantic_category",
            ALL_TRUE_SEMANTIC_CATEGORIES,
        ),
        (
            "semantic_relation",
            ALL_TRUE_SEMANTIC_RELATIONS,
        ),
        (
            "agent_type",
            ALL_TRUE_AGENT_TYPES,
        ),
    ]:
        for setting_name, enabled in switches.items():
            settings_rows.append({
                "setting_group": group_name,
                "setting_name": setting_name,
                "enabled_in_master_excel": bool(enabled),
            })

    master_tables = dict(
        master_result.get("excel_tables", {})
    )

    master_tables["Metadata"] = metadata_table
    master_tables["Master settings"] = pd.DataFrame(
        settings_rows
    )

    excel_path = Path(
        master_result["excel_path"]
    )

    # This is the one and only workbook write in Stage 1.
    save_inspection_workbook(
        excel_path=excel_path,
        tables=master_tables,
    )

    master_result["excel_tables"] = master_tables
    master_result["excel_path"] = excel_path

    print("\nMASTER XLSX SAVED")
    print("File:", excel_path.resolve())
    print("\nAll categories, relations, and agent types were enabled.")
    print(
        "Rerun this stage only when the scenario, frame, seed, "
        "source output, or export root changes."
    )

    return master_result


def _excel_boolean(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)

    if value is None:
        return False

    try:
        if pd.isna(value):
            return False
    except (TypeError, ValueError):
        pass

    return str(value).strip().lower() in {
        "1",
        "true",
        "yes",
        "y",
        "t",
    }


def _excel_predicate_list(value):
    """Restore a predicates list written by pandas/openpyxl."""
    if isinstance(value, list):
        return [
            str(item)
            for item in value
        ]

    if isinstance(value, (tuple, set, np.ndarray)):
        return [
            str(item)
            for item in list(value)
        ]

    if value is None:
        return []

    try:
        if pd.isna(value):
            return []
    except (TypeError, ValueError):
        pass

    text = str(value).strip()

    if not text:
        return []

    for parser in (
        json.loads,
        _python_ast.literal_eval,
    ):
        try:
            parsed = parser(text)
        except Exception:
            continue

        if isinstance(parsed, (list, tuple, set)):
            return [
                str(item)
                for item in parsed
            ]

        if isinstance(parsed, str):
            return [parsed]

    return [
        item.strip().strip("'\"")
        for item in text.strip("[]").split(",")
        if item.strip()
    ]


def _master_metadata_value(metadata, name, default=None):
    if (
        metadata is None
        or metadata.empty
        or name not in metadata.columns
    ):
        return default

    value = metadata.iloc[0][name]

    try:
        if pd.isna(value):
            return default
    except (TypeError, ValueError):
        pass

    return value


def _attach_live_source_objects(
    excel_entities,
    frame,
    scenario,
    scenario_token,
):
    """
    Keep entity coordinates/IDs from the workbook and attach only the live
    nuPlan objects needed by same-lane and predicted-path selection.

    Assertion and predicate information still comes exclusively from Excel.
    """
    entity_table = excel_entities.copy()

    if "track_token" not in entity_table.columns:
        raise KeyError(
            "The Entities sheet has no track_token column. "
            "Regenerate the master workbook with keep_spatial_tokens=True."
        )

    entity_table["track_token"] = (
        entity_table["track_token"]
        .astype(str)
    )

    if "is_ego" in entity_table.columns:
        entity_table["is_ego"] = (
            entity_table["is_ego"]
            .map(_excel_boolean)
        )
    else:
        entity_table["is_ego"] = (
            entity_table["track_token"]
            .astype(str)
            .str.lower()
            .eq("ego")
        )

    for numeric_column in [
        "x",
        "y",
        "heading",
        "length",
        "width",
    ]:
        if numeric_column in entity_table.columns:
            entity_table[numeric_column] = pd.to_numeric(
                entity_table[numeric_column],
                errors="coerce",
            )

    live_entities = add_stable_display_ids(
        build_entity_table(
            frame,
            scenario_token,
        ),
        scenario,
    )

    live_entities["track_token"] = (
        live_entities["track_token"]
        .astype(str)
    )

    live_lookup = (
        live_entities
        .drop_duplicates("track_token")
        .set_index("track_token")
    )

    entity_table["source_object"] = (
        entity_table["track_token"]
        .map(
            live_lookup[
                "source_object"
            ].to_dict()
        )
    )

    # Fill only missing workbook values from the selected nuPlan frame.
    for column in [
        "is_ego",
        "agent_type",
        "x",
        "y",
        "heading",
        "length",
        "width",
        "display_id",
    ]:
        if column not in live_lookup.columns:
            continue

        live_values = entity_table["track_token"].map(
            live_lookup[column].to_dict()
        )

        if column not in entity_table.columns:
            entity_table[column] = live_values
        else:
            entity_table[column] = (
                entity_table[column]
                .where(
                    entity_table[column].notna(),
                    live_values,
                )
            )

    if not entity_table["is_ego"].astype(bool).any():
        entity_table["is_ego"] = (
            entity_table["track_token"]
            .astype(str)
            .str.lower()
            .eq("ego")
        )

    if "display_id" not in entity_table.columns:
        raise KeyError(
            "The Entities sheet has no display_id column."
        )

    return entity_table



def _restore_follow_edges_from_excel(
    semantic_edges_all,
    workbook_tables,
):
    """
    Restore np:follows graph edges from the Follow evidence sheet.

    v9.5.5 master workbooks correctly saved np:follows in Detailed assertions
    and Follow evidence, but the Semantic edges builder removed entity-valued
    assertions because their value_json is null. This compatibility fallback
    allows those existing workbooks to be replotted without re-exporting them.
    New exports also retain np:follows directly in Semantic edges.
    """
    if semantic_edges_all is None:
        semantic_edges_all = pd.DataFrame()

    follow_table = workbook_tables.get(
        "Follow evidence",
        pd.DataFrame(),
    )

    if follow_table is None or follow_table.empty:
        return semantic_edges_all

    required_columns = {
        "subject_id",
        "object_id",
    }

    if not required_columns.issubset(
        set(follow_table.columns)
    ):
        return semantic_edges_all

    existing_follow_pairs = set()

    if not semantic_edges_all.empty:
        for row in semantic_edges_all.itertuples(
            index=False
        ):
            category = str(
                getattr(row, "category", "")
            ).strip().lower()

            predicates = _excel_predicate_list(
                getattr(row, "predicates", [])
            )

            if (
                category == "interaction"
                and "np:follows" in predicates
            ):
                existing_follow_pairs.add(
                    (
                        str(getattr(row, "subject_id", "")),
                        str(getattr(row, "object_id", "")),
                    )
                )

    recovered_rows = []

    for row in follow_table.itertuples(index=False):
        subject_id = str(
            getattr(row, "subject_id", "")
        ).strip()

        object_id = str(
            getattr(row, "object_id", "")
        ).strip()

        if not subject_id or not object_id:
            continue

        pair_key = (
            subject_id,
            object_id,
        )

        if pair_key in existing_follow_pairs:
            continue

        subject_token = str(
            getattr(
                row,
                "subject_track_token",
                entity_id_to_track_token(subject_id),
            )
        ).strip()

        object_token = str(
            getattr(
                row,
                "object_track_token",
                entity_id_to_track_token(object_id),
            )
        ).strip()

        relation_type = classify_semantic_relation(
            subject_id,
            object_id,
        )

        recovered_rows.append(
            {
                "relation_type": relation_type,
                "category": "interaction",
                "subject_id": subject_id,
                "object_id": object_id,
                "subject_token": subject_token,
                "object_token": object_token,
                "predicates": ["np:follows"],
                "predicate_count": 1,
                "relation_labels": "follows",
                "semantic_source_token": subject_token,
                "semantic_target_token": object_token,
                "semantic_statement": (
                    f"{short_token(subject_token)} → "
                    f"{short_token(object_token)}: follows"
                ),
            }
        )

        existing_follow_pairs.add(pair_key)

    if not recovered_rows:
        return semantic_edges_all

    recovered = pd.DataFrame(recovered_rows)

    return pd.concat(
        [semantic_edges_all, recovered],
        ignore_index=True,
        sort=False,
    )


def load_master_excel_for_plotting(excel_path):
    """
    Read all plotting data from a complete master workbook.

    No assertion Parquet file is opened here.
    """
    excel_path = Path(excel_path)

    if not excel_path.exists():
        raise FileNotFoundError(
            f"Master Excel workbook not found: {excel_path}\n"
            "Run LOAD AND EXPORT ALL TRUE first, or set "
            "MASTER_XLSX_TO_PLOT to an existing workbook."
        )

    workbook_tables = pd.read_excel(
        excel_path,
        sheet_name=None,
        engine="openpyxl",
    )

    required_sheets = {
        "Metadata",
        "Entities",
        "Semantic edges",
    }

    missing_sheets = (
        required_sheets
        - set(workbook_tables)
    )

    if missing_sheets:
        raise KeyError(
            "The selected workbook is not a complete master workbook. "
            f"Missing sheets: {sorted(missing_sheets)}. "
            "Generate it with LOAD AND EXPORT ALL TRUE."
        )

    metadata = workbook_tables["Metadata"].copy()

    format_version = _master_metadata_value(
        metadata,
        "excel_format_version",
        None,
    )

    if str(format_version) != MASTER_EXCEL_FORMAT_VERSION:
        raise RuntimeError(
            "The workbook was not generated by the current all-true "
            "master-export cell. Regenerate it before plotting.\n"
            f"Expected format: {MASTER_EXCEL_FORMAT_VERSION}\n"
            f"Found format: {format_version}"
        )

    scenario_token = str(
        _master_metadata_value(
            metadata,
            "scenario_token",
        )
    )

    timestamp_us = int(
        pd.to_numeric(
            _master_metadata_value(
                metadata,
                "timestamp_us",
            ),
            errors="raise",
        )
    )

    stored_frame_index = int(
        pd.to_numeric(
            _master_metadata_value(
                metadata,
                "frame_index",
                0,
            ),
            errors="raise",
        )
    )

    scenario, scenario_catalog_row = select_scenario(
        selector="token",
        value=scenario_token,
        random_seed=0,
        filter_seed=int(
            _master_metadata_value(
                metadata,
                "filter_seed",
                42,
            )
        ),
    )

    frame, resolved_frame_index = select_frame(
        scenario,
        selector="timestamp",
        value=timestamp_us,
        random_seed=0,
    )

    entity_table_all = _attach_live_source_objects(
        workbook_tables["Entities"],
        frame,
        scenario,
        scenario_token,
    )

    semantic_edges_all = workbook_tables[
        "Semantic edges"
    ].copy()

    semantic_edges_all = _restore_follow_edges_from_excel(
        semantic_edges_all,
        workbook_tables,
    )

    if "predicates" not in semantic_edges_all.columns:
        raise KeyError(
            "The Semantic edges sheet has no predicates column."
        )

    semantic_edges_all["predicates"] = (
        semantic_edges_all["predicates"]
        .map(_excel_predicate_list)
    )

    if "category" in semantic_edges_all.columns:
        semantic_edges_all["category"] = (
            semantic_edges_all["category"]
            .astype(str)
            .str.strip()
            .str.lower()
        )

    if "relation_type" in semantic_edges_all.columns:
        semantic_edges_all["relation_type"] = (
            semantic_edges_all["relation_type"]
            .astype(str)
            .str.strip()
        )

    return {
        "excel_path": excel_path,
        "tables": workbook_tables,
        "metadata": metadata,
        "scenario": scenario,
        "scenario_catalog_row": scenario_catalog_row,
        "frame": frame,
        "scenario_token": scenario_token,
        "timestamp_us": timestamp_us,
        "frame_index": resolved_frame_index,
        "stored_frame_index": stored_frame_index,
        "entity_table_all": entity_table_all,
        "semantic_edges_all": semantic_edges_all,
    }


def _filter_master_semantic_edges(
    semantic_edges_all,
    semantic_categories,
    semantic_relations,
):
    active_categories = {
        str(name).strip().lower()
        for name, enabled in semantic_categories.items()
        if enabled
    }

    active_relations = {
        str(name).strip()
        for name, enabled in semantic_relations.items()
        if enabled
    }

    edges = semantic_edges_all.copy()

    if edges.empty:
        return edges

    if "category" not in edges.columns:
        raise KeyError(
            "The Semantic edges sheet has no category column."
        )

    if "relation_type" not in edges.columns:
        raise KeyError(
            "The Semantic edges sheet has no relation_type column."
        )

    return edges[
        edges["category"].astype(str).str.lower().isin(
            active_categories
        )
        & edges["relation_type"].astype(str).isin(
            active_relations
        )
    ].copy()


def _temporal_agent_tokens_from_excel(
    temporal_values,
    relation_switches,
    entity_table,
):
    """
    Recover temporal predicate-bearing agents from the Temporal values sheet.
    """
    if temporal_values is None or temporal_values.empty:
        return set()

    enabled_relations = enabled_names(
        relation_switches
    )

    enabled_tokens = set(
        entity_table["track_token"].astype(str)
    )

    result = set()

    for row in temporal_values.itertuples(
        index=False
    ):
        scope = safe_string(
            getattr(row, "scope", "")
        ).strip().lower()

        subject_id = safe_string(
            getattr(row, "subject_id", "")
        )

        object_id = safe_string(
            getattr(row, "object_id", "")
        )

        if scope == "pair":
            relation_type = classify_semantic_relation(
                subject_id,
                object_id,
                "",
            )

            if relation_type not in enabled_relations:
                continue

        for entity_id in (
            subject_id,
            object_id,
        ):
            token = safe_string(
                entity_id_to_track_token(
                    entity_id
                )
            )

            if (
                token
                and token.lower() != "ego"
                and token in enabled_tokens
            ):
                result.add(token)

    return result


def _display_relation_type(
    subject_display,
    object_display,
):
    subject_display = safe_string(
        subject_display
    ).strip()

    object_display = safe_string(
        object_display
    ).strip()

    subject_is_ego = (
        subject_display.upper() == "EGO"
    )

    object_is_ego = (
        object_display.upper() == "EGO"
    )

    if subject_is_ego and object_display:
        return "ego_to_agent"

    if object_is_ego and subject_display:
        return "agent_to_ego"

    if subject_display and object_display:
        return "agent_to_agent"

    return "other"


def _filter_excel_value_table(
    table,
    *,
    relation_switches,
    entity_table,
    selected_tokens=None,
):
    """
    Apply relation, agent-type, and optional interested-agent filters to one
    small Excel value table.
    """
    if table is None or table.empty:
        return pd.DataFrame(
            columns=getattr(
                table,
                "columns",
                None,
            )
        )

    result = table.copy()

    enabled_display_ids = set(
        entity_table["display_id"].astype(str)
    )

    selected_display_ids = None

    if selected_tokens is not None:
        selected_tokens = set(
            map(str, selected_tokens)
        )

        selected_display_ids = set(
            entity_table.loc[
                entity_table["is_ego"].astype(bool)
                | entity_table["track_token"].astype(str).isin(
                    selected_tokens
                ),
                "display_id",
            ].astype(str)
        )

    enabled_relations = enabled_names(
        relation_switches
    )

    keep_rows = []

    for row in result.itertuples(
        index=False
    ):
        row_data = row._asdict()

        subject_display = safe_string(
            row_data.get("subject", "")
        ).strip()

        object_display = safe_string(
            row_data.get("object", "")
        ).strip()

        scope = safe_string(
            row_data.get("scope", "")
        ).strip().lower()

        if (
            subject_display
            and subject_display not in enabled_display_ids
        ):
            keep_rows.append(False)
            continue

        if (
            object_display
            and object_display not in enabled_display_ids
        ):
            keep_rows.append(False)
            continue

        if scope == "pair":
            relation_type = safe_string(
                row_data.get("pair_type", "")
            ).strip()

            if not relation_type:
                relation_type = _display_relation_type(
                    subject_display,
                    object_display,
                )

            if relation_type not in enabled_relations:
                keep_rows.append(False)
                continue

        if selected_display_ids is not None:
            non_ego_displays = {
                display_id
                for display_id in (
                    subject_display,
                    object_display,
                )
                if (
                    display_id
                    and display_id.upper() != "EGO"
                )
            }

            if (
                non_ego_displays
                and not non_ego_displays.issubset(
                    selected_display_ids
                )
            ):
                keep_rows.append(False)
                continue

        keep_rows.append(True)

    return result.loc[
        keep_rows
    ].reset_index(drop=True)


def plot_from_master_excel(
    *,
    excel_path,
    semantic_categories,
    semantic_relations,
    agent_types,
    semantic_predicates="all",

    interested_distance_threshold_m=40.0,
    include_within_distance=True,

    include_forward_corridor=True,
    forward_corridor_length_m=40.0,
    forward_corridor_half_width_m=5.0,

    include_same_lane=True,

    include_predicted_path_intersection=True,
    prediction_horizon_s=5.0,
    prediction_step_s=0.25,
    path_intersection_clearance_m=3.0,

    include_existing_spatial_agents=False,
    filter_edges_to_selected_agents=True,
    manual_highlight_tokens=None,

    map_radius_m=40.0,
    include_all_agents_in_graph=False,
    draw_semantic_arrows_on_map=True,
    show_agent_labels=True,
    show_agent_lane_info=False,
    show_map_structure_ids=False,
    show_edge_ids=False,
    show_heading_arrows=True,

    show_edge_legend=True,
    max_edge_legend_rows=None,
    show_filtered_tables=True,
    max_filtered_table_rows=200,
):
    """
    Read the complete workbook, apply the current switches, and replot.

    This function does not read assertion Parquet files, does not rebuild the
    complete assertion dataset, and does not write the workbook.
    """
    if manual_highlight_tokens is None:
        manual_highlight_tokens = []

    active_categories = enabled_names(
        semantic_categories
    )

    active_relations = enabled_names(
        semantic_relations
    )

    validate_semantic_predicate_selection(
        semantic_predicates,
        DEFINITIONS,
        active_categories=active_categories,
    )

    if not active_categories:
        raise ValueError(
            "Enable at least one SEMANTIC_CATEGORIES entry."
        )

    master = load_master_excel_for_plotting(
        excel_path
    )

    entity_table = filter_entities_by_agent_type(
        master["entity_table_all"],
        agent_types,
    )

    enabled_agent_tokens = set(
        entity_table.loc[
            ~entity_table["is_ego"].astype(bool),
            "track_token",
        ].astype(str)
    )

    semantic_edges_all = _filter_master_semantic_edges(
        master["semantic_edges_all"],
        semantic_categories,
        semantic_relations,
    )

    # Second visualization filter: exact predicate(s) inside the enabled family.
    semantic_edges_all = filter_semantic_edges_by_predicates(
        semantic_edges_all,
        semantic_predicates,
    )

    semantic_edges_all = filter_edges_by_enabled_entity_tokens(
        semantic_edges_all,
        entity_table,
    )

    all_spatial_edges = (
        master["semantic_edges_all"]
        .loc[
            master["semantic_edges_all"][
                "category"
            ].astype(str).str.lower().eq(
                "spatial"
            )
        ]
        .copy()
    )

    all_spatial_edges = filter_edges_by_enabled_entity_tokens(
        all_spatial_edges,
        entity_table,
    )

    candidate_tokens, selection_audit = (
        select_interesting_agents_for_plot(
            master["frame"],
            entity_table,
            all_spatial_edges,

            distance_threshold_m=(
                interested_distance_threshold_m
            ),
            include_within_distance=(
                include_within_distance
            ),

            include_forward_corridor=(
                include_forward_corridor
            ),
            forward_corridor_length_m=(
                forward_corridor_length_m
            ),
            forward_corridor_half_width_m=(
                forward_corridor_half_width_m
            ),

            include_same_lane=(
                include_same_lane
            ),

            include_predicted_path_intersection=(
                include_predicted_path_intersection
            ),
            prediction_horizon_s=(
                prediction_horizon_s
            ),
            prediction_step_s=(
                prediction_step_s
            ),
            path_intersection_clearance_m=(
                path_intersection_clearance_m
            ),

            include_existing_spatial_agents=(
                include_existing_spatial_agents
            ),

            manual_tokens=(
                manual_highlight_tokens
            ),
        )
    )

    candidate_tokens = (
        set(candidate_tokens)
        & enabled_agent_tokens
    )

    if not selection_audit.empty:
        selection_audit = selection_audit[
            selection_audit[
                "track_token"
            ].astype(str).isin(
                enabled_agent_tokens
            )
        ].copy()

    semantic_edges = (
        filter_edges_by_selected_agents(
            semantic_edges_all,
            candidate_tokens,
        )
        if filter_edges_to_selected_agents
        else semantic_edges_all.copy()
    )

    plot_edges = prepare_plot_edges(
        semantic_edges,
        entity_table,
    )

    predicate_tokens = (
        _agent_tokens_with_displayed_predicates(
            plot_edges
        )
    )

    temporal_values = master[
        "tables"
    ].get(
        "Temporal values",
        pd.DataFrame(),
    )

    if bool(
        semantic_categories.get(
            "temporal",
            False,
        )
    ):
        predicate_tokens = (
            set(predicate_tokens)
            | _temporal_agent_tokens_from_excel(
                temporal_values,
                semantic_relations,
                entity_table,
            )
        )

    displayed_selected_tokens = (
        set(candidate_tokens)
        & set(predicate_tokens)
    )

    if not selection_audit.empty:
        selection_audit = (
            selection_audit.copy()
        )

        selection_audit[
            "has_displayed_predicate"
        ] = (
            selection_audit[
                "track_token"
            ]
            .astype(str)
            .isin(predicate_tokens)
        )

        selection_audit[
            "displayed_selected"
        ] = (
            selection_audit[
                "track_token"
            ]
            .astype(str)
            .isin(
                displayed_selected_tokens
            )
        )

    edge_legend = edge_legend_dataframe(
        plot_edges
    )

    fig, axes = plt.subplots(
        2,
        1,
        figsize=(20, 22),
        constrained_layout=True,
    )

    ax_map, ax_graph = axes

    plot_map_with_agents(
        ax_map,
        master["frame"],
        entity_table,
        plot_edges,
        frame_assertions=master.get("frame_assertions"),

        scenario_token=(
            master["scenario_token"]
        ),
        frame_index=(
            master["frame_index"]
        ),
        selected_tokens=(
            displayed_selected_tokens
        ),
        predicate_tokens=(
            predicate_tokens
        ),

        map_radius_m=map_radius_m,

        manual_highlight_tokens=(
            manual_highlight_tokens
        ),

        draw_semantic_arrows=(
            draw_semantic_arrows_on_map
        ),

        show_agent_labels=(
            show_agent_labels
        ),

        show_agent_lane_info=(
            show_agent_lane_info
        ),

        show_map_structure_ids=(
            show_map_structure_ids
        ),

        show_edge_ids=(
            show_edge_ids
        ),

        agent_type_switches=(
            agent_types
        ),
    )

    plot_semantic_graph_real_coordinates(
        ax_graph,
        entity_table,
        plot_edges,

        selected_tokens=(
            displayed_selected_tokens
        ),

        predicate_tokens=(
            predicate_tokens
        ),

        include_all_agents=(
            include_all_agents_in_graph
            or bool(
                semantic_categories.get(
                    "temporal",
                    False,
                )
            )
        ),

        label_nodes=(
            show_agent_labels
        ),

        show_edge_ids=(
            show_edge_ids
        ),

        show_heading_arrows=(
            show_heading_arrows
        ),

        agent_type_switches=(
            agent_types
        ),
    )

    plt.show()

    if show_edge_legend:
        print_edge_legend(
            edge_legend,
            max_rows=max_edge_legend_rows,
        )

    selected_tokens_for_tables = (
        candidate_tokens
        if filter_edges_to_selected_agents
        else None
    )

    filtered_tables = {
        "Semantic edges": semantic_edges,
    }

    category_to_sheet = {
        "motion": "Motion values",
        "temporal": "Temporal values",
        "spatial": "Spatial values",
        "interaction": "Follow evidence",
    }

    for category_name, sheet_name in (
        category_to_sheet.items()
    ):
        if not bool(
            semantic_categories.get(
                category_name,
                False,
            )
        ):
            continue

        filtered_tables[sheet_name] = (
            _filter_excel_value_table(
                master["tables"].get(
                    sheet_name,
                    pd.DataFrame(),
                ),
                relation_switches=(
                    semantic_relations
                ),
                entity_table=(
                    entity_table
                ),
                selected_tokens=(
                    selected_tokens_for_tables
                ),
            )
        )

    if show_filtered_tables:
        print("\nEXCEL-BASED FILTERED TABLES")

        for table_name, table in (
            filtered_tables.items()
        ):
            print(
                f"\n{table_name.upper()} "
                f"({len(table):,} rows)"
            )

            if table.empty:
                print(
                    "No rows match the current plotting configuration."
                )
            else:
                display(
                    table.head(
                        int(
                            max_filtered_table_rows
                        )
                    )
                )

                if (
                    len(table)
                    > int(
                        max_filtered_table_rows
                    )
                ):
                    print(
                        "Showing "
                        f"{int(max_filtered_table_rows):,} "
                        f"of {len(table):,} rows."
                    )

    metadata = {
        "excel_path":
            str(master["excel_path"]),
        "scenario_token":
            master["scenario_token"],
        "semantic_predicates":
            semantic_predicate_selection_label(semantic_predicates),
        "frame_index":
            master["frame_index"],
        "timestamp_us":
            master["timestamp_us"],
        "active_categories":
            sorted(active_categories),
        "active_relations":
            sorted(active_relations),
        "active_agent_types":
            sorted(
                enabled_agent_type_names(
                    agent_types
                )
            ),
        "number_of_enabled_agents":
            max(0, len(entity_table) - 1),
        "number_of_candidate_agents":
            len(candidate_tokens),
        "number_of_displayed_selected_agents":
            len(displayed_selected_tokens),
        "number_of_filtered_semantic_rows":
            len(semantic_edges),
        "number_of_plot_edges":
            len(plot_edges),
    }

    return {
        **master,
        "metadata_plot": metadata,
        "entity_table": entity_table,
        "semantic_edges": semantic_edges,
        "plot_edges": plot_edges,
        "edge_legend": edge_legend,
        "selection_audit": selection_audit,
        "candidate_tokens": candidate_tokens,
        "selected_tokens": displayed_selected_tokens,
        "predicate_tokens": predicate_tokens,
        "filtered_tables": filtered_tables,
        "figure": fig,
    }


print(
    "Master-Excel workflow is ready: "
    "export all True once, then read/filter/plot from XLSX."
)

## LOAD AND EXPORT ALL TRUE

Run this cell once for the selected scenario and frame. The resulting workbook is the complete source for later plotting.

In [ ]:
# ============================================================
# 14. LOAD AND EXPORT ALL TRUE
# ============================================================
#
# RERUN THIS CELL ONLY WHEN YOU CHANGE:
#   - scenario selector, index, token, or random seed;
#   - frame selector, index, timestamp, token, or random seed;
#   - OUTPUT_DIR in the paths cell;
#   - MASTER_EXPORT_ROOT.
#
# This cell:
#   - reads the assertion Parquet files;
#   - enables every category, relation direction, and agent type;
#   - keeps all semantic edges;
#   - saves one complete Excel workbook;
#   - does not display the inspection plots.
#
# Do NOT rerun this cell for plotting/filter changes.
# ============================================================

MASTER_EXPORT_ROOT = (
    Path.cwd()
    / "visual_inspection_exports_master"
)

# Scenario:
#   "index", "token", or "random"
MASTER_SCENARIO_SELECTOR = "random"
MASTER_SCENARIO_VALUE = 0
MASTER_SCENARIO_SEED = 42
MASTER_FILTER_SEED = 42

# Frame:
#   "index", "timestamp", "token", or "random"
MASTER_FRAME_SELECTOR = "index"
MASTER_FRAME_VALUE = 5
MASTER_FRAME_SEED = 42


MASTER_RESULT = generate_complete_master_excel(
    scenario_selector=(
        MASTER_SCENARIO_SELECTOR
    ),
    scenario_value=(
        MASTER_SCENARIO_VALUE
    ),
    scenario_seed=(
        MASTER_SCENARIO_SEED
    ),
    filter_seed=(
        MASTER_FILTER_SEED
    ),

    frame_selector=(
        MASTER_FRAME_SELECTOR
    ),
    frame_value=(
        MASTER_FRAME_VALUE
    ),
    frame_seed=(
        MASTER_FRAME_SEED
    ),

    export_root=(
        MASTER_EXPORT_ROOT
    ),
)

MASTER_XLSX_PATH = Path(
    MASTER_RESULT["excel_path"]
)

print(
    "\nWorkbook ready for the plotting cell:"
)
print(
    MASTER_XLSX_PATH.resolve()
)

## READ EXCEL, CHANGE CONFIGURATION, AND REPLOT

Edit and rerun this cell repeatedly. It reads the saved workbook and applies the current category, relation, agent-type, interested-agent, and display switches.

In [ ]:
# ============================================================
# 15. READ EXCEL, CHANGE CONFIGURATION, AND REPLOT
# ============================================================
#
# RERUN ONLY THIS CELL WHEN YOU CHANGE:
#   - SEMANTIC_CATEGORIES;
#   - SEMANTIC_RELATIONS;
#   - AGENT_TYPES;
#   - interested-agent rules and thresholds;
#   - map radius and manual highlights;
#   - selected-agent edge filtering;
#   - semantic arrows, labels, headings, edge IDs, or legend limits.
#
# This cell:
#   - reads the saved master Excel workbook;
#   - applies the current True/False switches;
#   - recalculates interested agents;
#   - creates only the plots and small filtered tables;
#   - does NOT read assertion Parquet files;
#   - does NOT call run_and_export_semantic_inspection();
#   - does NOT rewrite the Excel workbook.
# ============================================================


# Normally this uses the workbook produced by the previous cell.
# After restarting the kernel, you may replace the right side with:
# Path("/absolute/path/to/scene_inspection_tables.xlsx")
MASTER_XLSX_TO_PLOT = globals().get(
    "MASTER_XLSX_PATH",
    None,
)

if MASTER_XLSX_TO_PLOT is None:
    raise RuntimeError(
        "No master Excel workbook is selected. "
        "Run LOAD AND EXPORT ALL TRUE first, or set "
        "MASTER_XLSX_TO_PLOT to an existing master XLSX path."
    )


# ============================================================
# CATEGORY SELECTION FOR THIS PLOT
# ============================================================

SEMANTIC_CATEGORIES = {
    "structure": False,
    "scenario": False,
    "position_values": False,
    "geometry": False,

    "motion": False,
    "motion_state": False,
    "temporal": False,
    "map": False,
    "route": False,
    "pairwise": False,
    "relevance": False,

    "spatial": False,

    "heading": False,
    "interaction": True,
    "risk": False,
    "traffic_light": False,
    "visibility": False,
    "future_observation": False,
    "maneuver": False,
    "intent": False,
    "other": False,
}


# ============================================================
# EXACT PREDICATE SELECTION FOR THIS PLOT
# ============================================================
#
# Family filter first, exact-predicate filter second.
#
# Show every predicate inside the enabled family/families:
SEMANTIC_PREDICATES = "all"
#
# Examples:
# SEMANTIC_PREDICATES = ["follows"]
# SEMANTIC_PREDICATES = ["crossesInFrontOf"]
# SEMANTIC_PREDICATES = ["follows", "overtakes"]
# Full IDs are also accepted, e.g. ["np:follows"].


# ============================================================
# RELATION SELECTION FOR THIS PLOT
# ============================================================

SEMANTIC_RELATIONS = {
    # EGO → A1, EGO → A2, ...
    "ego_to_agent": True,

    # A1 → EGO, A2 → EGO, ...
    "agent_to_ego": False,

    # A1 → A2, A2 → A3, ...
    "agent_to_agent": True,

    "ego_to_structure": True,
    "agent_to_structure": True,
    "structure_to_structure": False,
    "structure_to_entity": False,
    "other": False,
}


# ============================================================
# AGENT-TYPE SELECTION FOR THIS PLOT
# ============================================================

AGENT_TYPES = {
    # v9.5.33 default: show every tracked object type in the video/map.
    "vehicle": True,
    "car": True,
    "truck": True,
    "bus": True,
    "trailer": True,
    "construction_vehicle": True,
    "pedestrian": True,
    "bicycle": True,
    "motorcycle": True,
    "traffic_cone": True,
    "barrier": True,
    "czone_sign": True,
    "generic_object": True,
    "unknown": True,
}


# ============================================================
# INTERESTED-AGENT SELECTION FOR THIS PLOT
# ============================================================

INTERESTED_DISTANCE_THRESHOLD_M = 40.0
INCLUDE_WITHIN_DISTANCE = True

INCLUDE_FORWARD_CORRIDOR = True
FORWARD_CORRIDOR_LENGTH_M = 40.0
FORWARD_CORRIDOR_HALF_WIDTH_M = 5.0

INCLUDE_SAME_LANE = True

INCLUDE_PREDICTED_PATH_INTERSECTION = True
PREDICTION_HORIZON_S = 5.0
PREDICTION_STEP_S = 0.25
PATH_INTERSECTION_CLEARANCE_M = 3.0

INCLUDE_EXISTING_SPATIAL_AGENTS = False

FILTER_EDGES_TO_SELECTED_AGENTS = False

MANUAL_HIGHLIGHT_TOKENS = []


# ============================================================
# MAP, GRAPH, LABEL, AND LEGEND OPTIONS
# ============================================================

MAP_RADIUS_M = 55.0

INCLUDE_ALL_AGENTS_IN_GRAPH = False

DRAW_SEMANTIC_ARROWS_ON_MAP = True
SHOW_AGENT_LABELS = True

# Lane-change debugging switches.
# True: show extractor-consistent primary/candidate lane information above every entity.
SHOW_AGENT_LANE_INFO = True
# True: print native lane/lane-connector/roadblock IDs directly on the map.
SHOW_MAP_STRUCTURE_IDS = True

SHOW_EDGE_IDS = False
SHOW_HEADING_ARROWS = True

SHOW_EDGE_LEGEND = True
MAX_EDGE_LEGEND_ROWS = None

SHOW_FILTERED_TABLES = True
MAX_FILTERED_TABLE_ROWS = 200


# ============================================================
# READ XLSX, FILTER, AND PLOT
# ============================================================

PLOT_RESULT = plot_from_master_excel(
    excel_path=(
        MASTER_XLSX_TO_PLOT
    ),

    semantic_categories=(
        SEMANTIC_CATEGORIES
    ),
    semantic_relations=(
        SEMANTIC_RELATIONS
    ),
    semantic_predicates=(
        SEMANTIC_PREDICATES
    ),
    agent_types=(
        AGENT_TYPES
    ),

    interested_distance_threshold_m=(
        INTERESTED_DISTANCE_THRESHOLD_M
    ),
    include_within_distance=(
        INCLUDE_WITHIN_DISTANCE
    ),

    include_forward_corridor=(
        INCLUDE_FORWARD_CORRIDOR
    ),
    forward_corridor_length_m=(
        FORWARD_CORRIDOR_LENGTH_M
    ),
    forward_corridor_half_width_m=(
        FORWARD_CORRIDOR_HALF_WIDTH_M
    ),

    include_same_lane=(
        INCLUDE_SAME_LANE
    ),

    include_predicted_path_intersection=(
        INCLUDE_PREDICTED_PATH_INTERSECTION
    ),
    prediction_horizon_s=(
        PREDICTION_HORIZON_S
    ),
    prediction_step_s=(
        PREDICTION_STEP_S
    ),
    path_intersection_clearance_m=(
        PATH_INTERSECTION_CLEARANCE_M
    ),

    include_existing_spatial_agents=(
        INCLUDE_EXISTING_SPATIAL_AGENTS
    ),

    filter_edges_to_selected_agents=(
        FILTER_EDGES_TO_SELECTED_AGENTS
    ),

    manual_highlight_tokens=(
        MANUAL_HIGHLIGHT_TOKENS
    ),

    map_radius_m=(
        MAP_RADIUS_M
    ),

    include_all_agents_in_graph=(
        INCLUDE_ALL_AGENTS_IN_GRAPH
    ),

    draw_semantic_arrows_on_map=(
        DRAW_SEMANTIC_ARROWS_ON_MAP
    ),

    show_agent_labels=(
        SHOW_AGENT_LABELS
    ),

    show_agent_lane_info=(
        SHOW_AGENT_LANE_INFO
    ),

    show_map_structure_ids=(
        SHOW_MAP_STRUCTURE_IDS
    ),

    show_edge_ids=(
        SHOW_EDGE_IDS
    ),

    show_heading_arrows=(
        SHOW_HEADING_ARROWS
    ),

    show_edge_legend=(
        SHOW_EDGE_LEGEND
    ),

    max_edge_legend_rows=(
        MAX_EDGE_LEGEND_ROWS
    ),

    show_filtered_tables=(
        SHOW_FILTERED_TABLES
    ),

    max_filtered_table_rows=(
        MAX_FILTERED_TABLE_ROWS
    ),
)

In [ ]:
# OPTIONAL RESULT ACCESS
# display(PLOT_RESULT["semantic_edges"])
# display(PLOT_RESULT["entity_table"])
# display(PLOT_RESULT["selection_audit"])
# display(PLOT_RESULT["edge_legend"])
# display(PLOT_RESULT["filtered_tables"].get("Motion values"))
# display(PLOT_RESULT["filtered_tables"].get("Temporal values"))
# display(PLOT_RESULT["filtered_tables"].get("Spatial values"))
# print(PLOT_RESULT["excel_path"])
# display(PLOT_RESULT["filtered_tables"].get("Follow evidence"))


## MAP-ONLY SEMANTIC VIDEO FOR THE FULL SCENE

This version also renders active `np:changesLane(vehicle_like, target_lane)` relations. No source/target lane highlighting is added. During every active maneuver frame, one purple arrow starts at the current subject position and points to the fixed target-lane completion point stored in the predicate evidence.

In [ ]:
# ============================================================
# 16. FULL-SCENE CONFIGURABLE MAP-ONLY SEMANTIC VIDEO
# ============================================================
#
# This cell reuses the CURRENT configuration from Cell 15:
#   - SEMANTIC_CATEGORIES;
#   - SEMANTIC_RELATIONS;
#   - SEMANTIC_PREDICATES ("all" or exact predicate list);
#   - AGENT_TYPES;
#   - interested-agent rules and thresholds;
#   - selected-agent edge filtering;
#   - map radius, semantic arrows, labels, edge IDs, and legend limits.
#
# Change the switches in Cell 15, rerun Cell 15 (or at least its
# configuration statements), and then rerun this cell.
#
# This cell:
#   - reads assertion Parquet data because the master Excel contains only
#     one selected frame and cannot represent the complete scene;
#   - iterates through the complete selected nuPlan scene;
#   - renders ONLY the upper map figure;
#   - applies the same category/predicate/relation/agent/selection filters as Cell 15;
#   - saves one browser-compatible H.264 MP4 with yuv420p;
#   - keeps frames with zero enabled semantic relations.
#
# Run all function-definition cells above before running this cell.
# ============================================================

from collections import Counter as _VideoCounter
import shutil as _video_shutil
import subprocess as _video_subprocess
from IPython.display import Video as _NotebookVideo


# ============================================================
# VERIFY THAT CELL 15 CONFIGURATION EXISTS
# ============================================================

_REQUIRED_CELL_15_GLOBALS = [
    "SEMANTIC_CATEGORIES",
    "SEMANTIC_RELATIONS",
    "AGENT_TYPES",
    "INTERESTED_DISTANCE_THRESHOLD_M",
    "INCLUDE_WITHIN_DISTANCE",
    "INCLUDE_FORWARD_CORRIDOR",
    "FORWARD_CORRIDOR_LENGTH_M",
    "FORWARD_CORRIDOR_HALF_WIDTH_M",
    "INCLUDE_SAME_LANE",
    "INCLUDE_PREDICTED_PATH_INTERSECTION",
    "PREDICTION_HORIZON_S",
    "PREDICTION_STEP_S",
    "PATH_INTERSECTION_CLEARANCE_M",
    "INCLUDE_EXISTING_SPATIAL_AGENTS",
    "FILTER_EDGES_TO_SELECTED_AGENTS",
    "MANUAL_HIGHLIGHT_TOKENS",
    "MAP_RADIUS_M",
    "DRAW_SEMANTIC_ARROWS_ON_MAP",
    "SHOW_AGENT_LABELS",
    "SHOW_AGENT_LANE_INFO",
    "SHOW_MAP_STRUCTURE_IDS",
    "SHOW_EDGE_IDS",
    "SHOW_EDGE_LEGEND",
    "MAX_EDGE_LEGEND_ROWS",
    "SHOW_FILTERED_TABLES",
    "MAX_FILTERED_TABLE_ROWS",
]

_missing_cell_15_globals = [
    name
    for name in _REQUIRED_CELL_15_GLOBALS
    if name not in globals()
]

if _missing_cell_15_globals:
    raise RuntimeError(
        "Run the Cell 15 configuration first. Missing variables: "
        + ", ".join(_missing_cell_15_globals)
    )


# ============================================================
# SNAPSHOT THE CURRENT CELL 15 CONFIGURATION
# ============================================================
#
# These copies ensure that every rendered frame uses one consistent
# configuration, even if a notebook variable is edited while rendering.
# ============================================================

VIDEO_SEMANTIC_CATEGORIES = dict(SEMANTIC_CATEGORIES)
VIDEO_SEMANTIC_RELATIONS = dict(SEMANTIC_RELATIONS)
VIDEO_SEMANTIC_PREDICATES = globals().get("SEMANTIC_PREDICATES", "all")
VIDEO_AGENT_TYPES = dict(AGENT_TYPES)

VIDEO_INTERESTED_DISTANCE_THRESHOLD_M = float(
    INTERESTED_DISTANCE_THRESHOLD_M
)
VIDEO_INCLUDE_WITHIN_DISTANCE = bool(INCLUDE_WITHIN_DISTANCE)

VIDEO_INCLUDE_FORWARD_CORRIDOR = bool(INCLUDE_FORWARD_CORRIDOR)
VIDEO_FORWARD_CORRIDOR_LENGTH_M = float(FORWARD_CORRIDOR_LENGTH_M)
VIDEO_FORWARD_CORRIDOR_HALF_WIDTH_M = float(
    FORWARD_CORRIDOR_HALF_WIDTH_M
)

VIDEO_INCLUDE_SAME_LANE = bool(INCLUDE_SAME_LANE)

VIDEO_INCLUDE_PREDICTED_PATH_INTERSECTION = bool(
    INCLUDE_PREDICTED_PATH_INTERSECTION
)
VIDEO_PREDICTION_HORIZON_S = float(PREDICTION_HORIZON_S)
VIDEO_PREDICTION_STEP_S = float(PREDICTION_STEP_S)
VIDEO_PATH_INTERSECTION_CLEARANCE_M = float(
    PATH_INTERSECTION_CLEARANCE_M
)

VIDEO_INCLUDE_EXISTING_SPATIAL_AGENTS = bool(
    INCLUDE_EXISTING_SPATIAL_AGENTS
)
VIDEO_FILTER_EDGES_TO_SELECTED_AGENTS = bool(
    FILTER_EDGES_TO_SELECTED_AGENTS
)
VIDEO_MANUAL_HIGHLIGHT_TOKENS = list(MANUAL_HIGHLIGHT_TOKENS)

VIDEO_MAP_RADIUS_M = float(MAP_RADIUS_M)
VIDEO_DRAW_SEMANTIC_ARROWS_ON_MAP = bool(
    DRAW_SEMANTIC_ARROWS_ON_MAP
)
VIDEO_SHOW_AGENT_LABELS = bool(SHOW_AGENT_LABELS)
VIDEO_SHOW_AGENT_LANE_INFO = bool(SHOW_AGENT_LANE_INFO)
VIDEO_SHOW_MAP_STRUCTURE_IDS = bool(SHOW_MAP_STRUCTURE_IDS)
VIDEO_SHOW_EDGE_IDS = bool(SHOW_EDGE_IDS)

VIDEO_SHOW_EDGE_LEGEND = bool(SHOW_EDGE_LEGEND)
VIDEO_MAX_EDGE_LEGEND_ROWS = MAX_EDGE_LEGEND_ROWS
VIDEO_SHOW_FILTERED_TABLES = bool(SHOW_FILTERED_TABLES)
VIDEO_MAX_FILTERED_TABLE_ROWS = int(MAX_FILTERED_TABLE_ROWS)

_active_video_categories = sorted(
    enabled_names(VIDEO_SEMANTIC_CATEGORIES)
)
_active_video_relations = sorted(
    enabled_names(VIDEO_SEMANTIC_RELATIONS)
)

if not _active_video_categories:
    raise ValueError(
        "Enable at least one SEMANTIC_CATEGORIES entry in Cell 15."
    )

if not _active_video_relations:
    raise ValueError(
        "Enable at least one SEMANTIC_RELATIONS entry in Cell 15."
    )


# ============================================================
# VIDEO-ONLY CONFIGURATION
# ============================================================

VIDEO_EXPORT_ROOT = Path.cwd() / "visual_inspection_semantic_videos"

# Reuse the scenario selected for the master workbook by default.
VIDEO_SCENARIO_SELECTOR = globals().get(
    "MASTER_SCENARIO_SELECTOR",
    "random",
)
VIDEO_SCENARIO_VALUE = globals().get(
    "MASTER_SCENARIO_VALUE",
    0,
)
VIDEO_SCENARIO_SEED = int(
    globals().get("MASTER_SCENARIO_SEED", 42)
)
VIDEO_FILTER_SEED = int(
    globals().get("MASTER_FILTER_SEED", 42)
)

# Full scene by default. END=None means the last available frame.
VIDEO_START_FRAME = 0
VIDEO_END_FRAME = None
VIDEO_FRAME_STEP = 1

# None preserves approximately real-time nuPlan playback.
# Example: 2.0 saves a video that plays twice as fast.
VIDEO_PLAYBACK_SPEED = 1.0
VIDEO_FPS = None

VIDEO_FIGSIZE = (12, 12)
VIDEO_DPI = 120
VIDEO_SHOW_STATUS_BOX = False

# crossesInFrontOf debugging: draw the exact same finite 10 m forward
# segments used by the paper crossing detector for every active crossing.
# Set False to hide these debugging rays from the saved videos.
VIDEO_DRAW_CROSSES_REFERENCE_LINES = True
VIDEO_CROSSES_REFERENCE_RAY_LENGTH_M = 10.0


# Used only to obtain every spatial relation for the optional
# INCLUDE_EXISTING_SPATIAL_AGENTS candidate-selection rule.
_VIDEO_SPATIAL_ONLY_CATEGORIES = {
    name: (name == "spatial")
    for name in ALL_SEMANTIC_CATEGORIES
}
_VIDEO_ALL_RELATIONS = {
    name: True
    for name in ALL_SEMANTIC_RELATIONS
}


def _video_default_fps(frames, frame_indices, playback_speed):
    """Derive playback FPS from the selected nuPlan timestamps."""
    timestamps = np.asarray(
        [int(frames[index].timestamp_us) for index in frame_indices],
        dtype=np.int64,
    )

    if len(timestamps) < 2:
        return max(1.0, float(playback_speed))

    positive_deltas_s = np.diff(timestamps).astype(float) / 1e6
    positive_deltas_s = positive_deltas_s[positive_deltas_s > 0]

    if len(positive_deltas_s) == 0:
        return max(1.0, float(playback_speed))

    source_fps = 1.0 / float(np.median(positive_deltas_s))

    return float(
        np.clip(
            source_fps * float(playback_speed),
            0.25,
            60.0,
        )
    )


def _video_relation_label_counts(plot_edges):
    """Count the displayed semantic labels at one frame."""
    counts = _VideoCounter()

    if plot_edges is None or plot_edges.empty:
        return counts

    for row in plot_edges.itertuples(index=False):
        labels = str(
            getattr(row, "relation_labels", "")
            or "unspecified"
        )
        for label in [part.strip() for part in labels.split(",")]:
            if label:
                counts[label] += 1

    return counts


def _video_displayed_follow_mode_counts(frame_assertions, plot_edges):
    """Count follow modes only for np:follows edges displayed in this frame."""
    counts = _VideoCounter()

    if (
        frame_assertions is None
        or frame_assertions.empty
        or plot_edges is None
        or plot_edges.empty
    ):
        return counts

    displayed_follow_pairs = set()

    for edge in plot_edges.itertuples(index=False):
        predicates = {
            str(predicate)
            for predicate in getattr(edge, "predicates", [])
        }

        if "np:follows" not in predicates:
            continue

        displayed_follow_pairs.add((
            str(getattr(edge, "subject_token", "")),
            str(getattr(edge, "object_token", "")),
        ))

    if not displayed_follow_pairs:
        return counts

    follows = frame_assertions.loc[
        frame_assertions["predicate_id"]
        .astype(str)
        .eq("np:follows")
    ]

    for row in follows.itertuples(index=False):
        subject_token = str(
            entity_id_to_track_token(
                getattr(row, "subject_id", "")
            )
        )
        object_token = str(
            entity_id_to_track_token(
                getattr(row, "object_id", "")
            )
        )

        if (subject_token, object_token) not in displayed_follow_pairs:
            continue

        evidence = _evidence_mapping(
            getattr(row, "evidence_json", None)
        )
        mode = str(
            evidence.get("follow_mode")
            or "unspecified"
        )
        counts[mode] += 1

    return counts



def _video_follow_mode_by_pair(frame_assertions):
    """Return {(subject_token, object_token): follow_mode} for one frame."""
    result = {}

    if frame_assertions is None or frame_assertions.empty:
        return result

    follows = frame_assertions.loc[
        frame_assertions["predicate_id"]
        .astype(str)
        .eq("np:follows")
    ]

    for row in follows.itertuples(index=False):
        subject_token = str(
            entity_id_to_track_token(
                getattr(row, "subject_id", "")
            )
        )
        object_token = str(
            entity_id_to_track_token(
                getattr(row, "object_id", "")
            )
        )
        evidence = _evidence_mapping(
            getattr(row, "evidence_json", None)
        )
        result[(subject_token, object_token)] = str(
            evidence.get("follow_mode")
            or "unspecified"
        )

    return result

def _video_overtake_evidence_by_pair(frame_assertions):
    """Return active full-maneuver overtake evidence by directed pair."""
    result = {}

    if frame_assertions is None or frame_assertions.empty:
        return result

    rows = frame_assertions.loc[
        frame_assertions["predicate_id"]
        .astype(str)
        .eq("np:overtakes")
    ]

    for row in rows.itertuples(index=False):
        subject_token = str(
            entity_id_to_track_token(
                getattr(row, "subject_id", "")
            )
        )
        object_token = str(
            entity_id_to_track_token(
                getattr(row, "object_id", "")
            )
        )
        evidence = _evidence_mapping(
            getattr(row, "evidence_json", None)
        )
        result[(subject_token, object_token)] = evidence

    return result



def _video_restore_overtake_edges(
    semantic_edges_all,
    frame_assertions,
):
    """
    Restore np:overtakes edges that the generic semantic-edge converter may
    omit because np:overtakes is entity-valued and stores its target in
    object_id while value_json is null.

    This mirrors the existing compatibility recovery used for np:follows.
    """
    if semantic_edges_all is None:
        semantic_edges_all = pd.DataFrame()

    if frame_assertions is None or frame_assertions.empty:
        return semantic_edges_all

    if "predicate_id" not in frame_assertions.columns:
        return semantic_edges_all

    overtakes = frame_assertions.loc[
        frame_assertions["predicate_id"]
        .astype(str)
        .eq("np:overtakes")
    ]

    if overtakes.empty:
        return semantic_edges_all

    existing_pairs = set()

    if not semantic_edges_all.empty:
        for row in semantic_edges_all.itertuples(index=False):
            predicates = {
                str(predicate)
                for predicate in _excel_predicate_list(
                    getattr(row, "predicates", [])
                )
            }

            if "np:overtakes" not in predicates:
                continue

            existing_pairs.add((
                str(getattr(row, "subject_id", "")).strip(),
                str(getattr(row, "object_id", "")).strip(),
            ))

    recovered_rows = []

    for row in overtakes.itertuples(index=False):
        subject_id = str(
            getattr(row, "subject_id", "")
        ).strip()
        object_id = str(
            getattr(row, "object_id", "")
        ).strip()

        if not subject_id or not object_id:
            continue

        pair_key = (subject_id, object_id)

        if pair_key in existing_pairs:
            continue

        subject_token = str(
            entity_id_to_track_token(subject_id)
        ).strip()
        object_token = str(
            entity_id_to_track_token(object_id)
        ).strip()

        recovered_rows.append({
            "relation_type": classify_semantic_relation(
                subject_id,
                object_id,
            ),
            "category": "interaction",
            "subject_id": subject_id,
            "object_id": object_id,
            "subject_token": subject_token,
            "object_token": object_token,
            "predicates": ["np:overtakes"],
            "predicate_count": 1,
            "relation_labels": "overtakes",
            "semantic_source_token": subject_token,
            "semantic_target_token": object_token,
            "semantic_statement": (
                f"{short_token(subject_token)} → "
                f"{short_token(object_token)}: overtakes"
            ),
        })

        existing_pairs.add(pair_key)

    if not recovered_rows:
        return semantic_edges_all

    return pd.concat(
        [
            semantic_edges_all,
            pd.DataFrame(recovered_rows),
        ],
        ignore_index=True,
        sort=False,
    )


def _video_restore_merge_edges(semantic_edges_all, frame_assertions):
    """Restore entity-valued merge/crossing relations from current-frame assertions."""
    if semantic_edges_all is None:
        semantic_edges_all = pd.DataFrame()
    if frame_assertions is None or frame_assertions.empty:
        return semantic_edges_all
    if "predicate_id" not in frame_assertions.columns:
        return semantic_edges_all

    supported = {
        "np:mergesInFrontOf": "mergesInFrontOf",
        "np:mergesBehind": "mergesBehind",
        "np:crossesInFrontOf": "crossesInFrontOf",
        "np:yieldsTo": "yieldsTo",
    }
    rows = frame_assertions.loc[
        frame_assertions["predicate_id"].astype(str).isin(set(supported))
    ]
    if rows.empty:
        return semantic_edges_all

    existing = set()
    if not semantic_edges_all.empty:
        for row in semantic_edges_all.itertuples(index=False):
            predicates = {
                str(predicate)
                for predicate in _excel_predicate_list(getattr(row, "predicates", []))
            }
            for predicate in predicates.intersection(supported):
                existing.add((
                    str(getattr(row, "subject_id", "")).strip(),
                    str(getattr(row, "object_id", "")).strip(),
                    predicate,
                ))

    recovered = []
    for row in rows.itertuples(index=False):
        predicate = str(getattr(row, "predicate_id", ""))
        subject_id = str(getattr(row, "subject_id", "")).strip()
        object_id = str(getattr(row, "object_id", "")).strip()
        if not subject_id or not object_id or predicate not in supported:
            continue
        key = (subject_id, object_id, predicate)
        if key in existing:
            continue
        subject_token = str(entity_id_to_track_token(subject_id)).strip()
        object_token = str(entity_id_to_track_token(object_id)).strip()
        label = supported[predicate]
        recovered.append({
            "relation_type": classify_semantic_relation(subject_id, object_id),
            "category": "interaction",
            "subject_id": subject_id,
            "object_id": object_id,
            "subject_token": subject_token,
            "object_token": object_token,
            "predicates": [predicate],
            "predicate_count": 1,
            "relation_labels": label,
            "semantic_source_token": subject_token,
            "semantic_target_token": object_token,
            "semantic_statement": (
                f"{short_token(subject_token)} → {short_token(object_token)}: {label}"
            ),
        })
        existing.add(key)

    if not recovered:
        return semantic_edges_all
    return pd.concat(
        [semantic_edges_all, pd.DataFrame(recovered)],
        ignore_index=True,
        sort=False,
    )


def _video_restore_risk_edges(semantic_edges_all, frame_assertions):
    """Restore exact current-frame risk relations from assertion rows.

    Risk predicates are entity-valued agent/ego relations.  The generic
    semantic-edge conversion should normally retain them, but the video
    pipeline historically needed compatibility restoration for several
    entity-valued predicates.  Keep risk validation robust by reconstructing
    the stored SUBJECT -> OBJECT relation directly from the assertion row when
    it is absent.
    """
    if semantic_edges_all is None:
        semantic_edges_all = pd.DataFrame()
    if frame_assertions is None or frame_assertions.empty:
        return semantic_edges_all
    if "predicate_id" not in frame_assertions.columns:
        return semantic_edges_all

    predicate = "np:hasConflictRiskWith"
    rows = frame_assertions.loc[
        frame_assertions["predicate_id"].astype(str).eq(predicate)
    ]
    if rows.empty:
        return semantic_edges_all

    existing = set()
    if not semantic_edges_all.empty:
        for edge in semantic_edges_all.itertuples(index=False):
            predicates = {
                str(value)
                for value in _excel_predicate_list(
                    getattr(edge, "predicates", [])
                )
            }
            if predicate not in predicates:
                continue
            existing.add((
                str(getattr(edge, "subject_id", "")).strip(),
                str(getattr(edge, "object_id", "")).strip(),
            ))

    recovered = []
    for row in rows.itertuples(index=False):
        subject_id = str(getattr(row, "subject_id", "") or "").strip()
        object_id = str(getattr(row, "object_id", "") or "").strip()
        if not subject_id or not object_id:
            continue
        key = (subject_id, object_id)
        if key in existing:
            continue

        subject_token = str(entity_id_to_track_token(subject_id)).strip()
        object_token = str(entity_id_to_track_token(object_id)).strip()
        recovered.append({
            "relation_type": classify_semantic_relation(subject_id, object_id),
            "category": "risk",
            "subject_id": subject_id,
            "object_id": object_id,
            "subject_token": subject_token,
            "object_token": object_token,
            "predicates": [predicate],
            "predicate_count": 1,
            "relation_labels": "hasConflictRiskWith",
            "semantic_source_token": subject_token,
            "semantic_target_token": object_token,
            "semantic_statement": (
                f"{short_token(subject_token)} → "
                f"{short_token(object_token)}: hasConflictRiskWith"
            ),
        })
        existing.add(key)

    if not recovered:
        return semantic_edges_all

    return pd.concat(
        [semantic_edges_all, pd.DataFrame(recovered)],
        ignore_index=True,
        sort=False,
    )


def _video_active_lane_change_overlays(
    frame_assertions,
    semantic_relations,
    entity_table=None,
):
    """Return active np:changesLane vehicle-to-structure evidence rows."""
    result = []

    if frame_assertions is None or frame_assertions.empty:
        return result
    if "predicate_id" not in frame_assertions.columns:
        return result

    rows = frame_assertions.loc[
        frame_assertions["predicate_id"]
        .astype(str)
        .eq("np:changesLane")
    ]

    enabled_subject_tokens = None
    if entity_table is not None and not entity_table.empty:
        enabled_subject_tokens = set(
            entity_table["track_token"].dropna().astype(str)
        )

    seen_event_ids = set()
    for row in rows.itertuples(index=False):
        subject_id = str(getattr(row, "subject_id", "")).strip()
        object_id = str(getattr(row, "object_id", "")).strip()
        if not subject_id or not object_id:
            continue

        relation_type = str(
            classify_semantic_relation(subject_id, object_id)
        )
        if not bool(semantic_relations.get(relation_type, False)):
            continue

        subject_token = str(
            entity_id_to_track_token(subject_id)
        )
        if (
            enabled_subject_tokens is not None
            and subject_token not in enabled_subject_tokens
        ):
            continue

        evidence = _evidence_mapping(
            getattr(row, "evidence_json", None)
        )
        event_id = str(
            evidence.get("event_id")
            or f"{subject_id}|{object_id}"
        )
        if event_id in seen_event_ids:
            continue
        seen_event_ids.add(event_id)

        result.append({
            "event_id": event_id,
            "subject_id": subject_id,
            "subject_token": subject_token,
            "object_id": object_id,
            "relation_type": relation_type,
            "evidence": evidence,
        })

    return result


def _video_draw_lane_change_overlays(
    axis,
    frame,
    frame_assertions,
    semantic_relations,
    entity_table=None,
):
    """Draw only the active subject-to-completion-point lane-change arrow."""
    del frame  # Kept in the public signature for notebook compatibility.

    overlays = _video_active_lane_change_overlays(
        frame_assertions,
        semantic_relations,
        entity_table=entity_table,
    )

    for overlay in overlays:
        evidence = overlay["evidence"]
        coordinate_names = (
            "lane_change_arrow_start_x",
            "lane_change_arrow_start_y",
            "lane_change_arrow_end_x",
            "lane_change_arrow_end_y",
        )
        coordinates = []
        for name in coordinate_names:
            try:
                value = float(evidence.get(name))
            except (TypeError, ValueError):
                value = float("nan")
            coordinates.append(value)

        if not all(math.isfinite(value) for value in coordinates):
            continue

        start_x, start_y, end_x, end_y = coordinates
        if math.hypot(end_x - start_x, end_y - start_y) <= 1e-6:
            continue

        axis.annotate(
            "",
            xy=(end_x, end_y),
            xytext=(start_x, start_y),
            arrowprops={
                "arrowstyle": "-|>",
                "color": "#6a1b9a",
                "linewidth": 4.0,
                "mutation_scale": 18,
                "shrinkA": 0,
                "shrinkB": 0,
                "alpha": 0.95,
            },
            zorder=18,
        )

    return overlays


def _video_safe_filename_fragment(values, fallback):
    """Create a short filename fragment from enabled configuration names."""
    cleaned = []

    for value in values:
        text = "".join(
            character
            if character.isalnum() or character in {"-", "_"}
            else "_"
            for character in str(value)
        ).strip("_")

        if text:
            cleaned.append(text)

    if not cleaned:
        return fallback

    joined = "-".join(cleaned)
    return joined[:120]


# ============================================================
# EGO ENDPOINT NORMALIZATION FOR VIDEO EDGES
# ============================================================
#
# The assertion/RDF layer and the frame entity table may represent EGO with
# different identifiers. Normal agents usually retain their track token, while
# EGO can appear as "ego", "EGO", an RDF URI ending in /ego or #ego, or an
# EgoVehicle URI. Normalize edge endpoints before relation-direction filtering
# and before prepare_plot_edges().
# ============================================================


def _video_is_ego_identifier(value):
    """Return True for the common EGO identifiers used by the pipeline."""
    if value is None:
        return False

    text = str(value).strip()
    if not text:
        return False

    lowered = text.lower().rstrip("/")

    if lowered in {
        "ego",
        "np:ego",
        "ego_vehicle",
        "egovehicle",
        "np:egovehicle",
    }:
        return True

    return (
        "egovehicle" in lowered
        or lowered.endswith("/ego")
        or lowered.endswith("#ego")
        or lowered.endswith(":ego")
    )


def _video_canonical_ego_token(entity_table):
    """Return the exact EGO token used by the current frame entity table."""
    if entity_table is None or entity_table.empty:
        return "ego"

    if "is_ego" not in entity_table.columns:
        return "ego"

    ego_rows = entity_table.loc[
        entity_table["is_ego"].fillna(False).astype(bool)
    ]
    if ego_rows.empty:
        return "ego"

    for column in ("track_token", "entity_token", "token"):
        if column in ego_rows.columns:
            value = ego_rows.iloc[0][column]
            if value is not None and str(value).strip():
                return str(value)

    return "ego"


def _video_normalize_endpoint(value, canonical_ego_token):
    """Normalize one semantic-edge endpoint to a frame entity token."""
    if _video_is_ego_identifier(value):
        return str(canonical_ego_token)

    try:
        converted = entity_id_to_track_token(value)
    except Exception:
        converted = value

    if _video_is_ego_identifier(converted):
        return str(canonical_ego_token)

    return str(converted)


def _video_normalize_edge_endpoints(edge_table, entity_table):
    """
    Normalize EGO/agent endpoint identifiers and recompute relation direction.

    This must run before _filter_master_semantic_edges(), because that function
    uses relation_type to apply ego_to_agent, agent_to_ego, and agent_to_agent
    switches.
    """
    if edge_table is None:
        return pd.DataFrame()
    if edge_table.empty:
        return edge_table.copy()

    result = edge_table.copy()
    canonical_ego_token = _video_canonical_ego_token(entity_table)

    subject_columns = [
        column
        for column in ("subject_token", "subject", "subject_id")
        if column in result.columns
    ]
    object_columns = [
        column
        for column in ("object_token", "object", "object_id")
        if column in result.columns
    ]

    for column in subject_columns:
        result[column] = result[column].map(
            lambda value: _video_normalize_endpoint(
                value,
                canonical_ego_token,
            )
        )

    for column in object_columns:
        result[column] = result[column].map(
            lambda value: _video_normalize_endpoint(
                value,
                canonical_ego_token,
            )
        )

    subject_column = next(
        (column for column in ("subject_token", "subject", "subject_id")
         if column in result.columns),
        None,
    )
    object_column = next(
        (column for column in ("object_token", "object", "object_id")
         if column in result.columns),
        None,
    )

    if subject_column is not None and object_column is not None:
        subject_is_ego = result[subject_column].astype(str).eq(
            str(canonical_ego_token)
        )
        object_is_ego = result[object_column].astype(str).eq(
            str(canonical_ego_token)
        )

        if "relation_type" not in result.columns:
            result["relation_type"] = "other"

        entity_tokens = set()
        if "track_token" in entity_table.columns:
            entity_tokens = set(
                entity_table["track_token"].dropna().astype(str)
            )

        subject_is_entity = result[subject_column].astype(str).isin(
            entity_tokens
        )
        object_is_entity = result[object_column].astype(str).isin(
            entity_tokens
        )

        result.loc[
            subject_is_ego & object_is_entity & ~object_is_ego,
            "relation_type",
        ] = "ego_to_agent"
        result.loc[
            ~subject_is_ego & subject_is_entity & object_is_ego,
            "relation_type",
        ] = "agent_to_ego"
        result.loc[
            ~subject_is_ego
            & ~object_is_ego
            & subject_is_entity
            & object_is_entity,
            "relation_type",
        ] = "agent_to_agent"

    return result


def _video_count_ego_edges(edge_table, entity_table):
    """Count rows whose subject or object is EGO."""
    if edge_table is None or edge_table.empty:
        return 0

    canonical_ego_token = _video_canonical_ego_token(entity_table)
    subject_column = next(
        (column for column in ("subject_token", "subject", "subject_id")
         if column in edge_table.columns),
        None,
    )
    object_column = next(
        (column for column in ("object_token", "object", "object_id")
         if column in edge_table.columns),
        None,
    )

    if subject_column is None or object_column is None:
        return 0

    return int(
        (
            edge_table[subject_column].astype(str).eq(canonical_ego_token)
            | edge_table[object_column].astype(str).eq(canonical_ego_token)
        ).sum()
    )


def _video_count_raw_ego_follows(frame_assertions):
    """Count raw np:follows assertions involving EGO before visualization."""
    if frame_assertions is None or frame_assertions.empty:
        return 0

    follows = frame_assertions.loc[
        frame_assertions["predicate_id"].astype(str).eq("np:follows")
    ]
    if follows.empty:
        return 0

    return int(
        (
            follows["subject_id"].map(_video_is_ego_identifier)
            | follows["object_id"].map(_video_is_ego_identifier)
        ).sum()
    )



def _video_convert_mp4v_to_browser_h264(
    temporary_video_path,
    final_video_path,
):
    """Convert OpenCV MP4V output to browser-compatible H.264/yuv420p."""
    temporary_video_path = Path(temporary_video_path)
    final_video_path = Path(final_video_path)

    ffmpeg_executable = _video_shutil.which("ffmpeg")
    if ffmpeg_executable is None:
        raise RuntimeError(
            "FFmpeg is required for browser-compatible H.264 output. "
            "Install it in the active environment, for example with: "
            "conda install -c conda-forge ffmpeg -y"
        )

    if (
        not temporary_video_path.exists()
        or temporary_video_path.stat().st_size == 0
    ):
        raise RuntimeError(
            "OpenCV did not create a valid temporary MP4V video: "
            f"{temporary_video_path}"
        )

    final_video_path.parent.mkdir(parents=True, exist_ok=True)
    final_video_path.unlink(missing_ok=True)

    command = [
        ffmpeg_executable,
        "-hide_banner",
        "-loglevel",
        "error",
        "-y",
        "-i",
        str(temporary_video_path),
        "-c:v",
        "libx264",
        "-preset",
        "medium",
        "-crf",
        "20",
        "-pix_fmt",
        "yuv420p",
        "-tag:v",
        "avc1",
        "-movflags",
        "+faststart",
        "-vf",
        "scale=trunc(iw/2)*2:trunc(ih/2)*2",
        "-an",
        str(final_video_path),
    ]

    completed = _video_subprocess.run(
        command,
        stdout=_video_subprocess.PIPE,
        stderr=_video_subprocess.PIPE,
        text=True,
        check=False,
    )

    if completed.returncode != 0:
        raise RuntimeError(
            "FFmpeg H.264 conversion failed. The temporary MP4V file "
            f"was kept at {temporary_video_path}.\n"
            f"FFmpeg error:\n{completed.stderr.strip()}"
        )

    if (
        not final_video_path.exists()
        or final_video_path.stat().st_size == 0
    ):
        raise RuntimeError(
            "FFmpeg completed without creating a valid H.264 file: "
            f"{final_video_path}"
        )

    temporary_video_path.unlink(missing_ok=True)
    return final_video_path



def _video_draw_crosses_reference_lines(
    axis,
    plot_edges,
    entity_table,
    *,
    map_radius_m,
):
    """Draw the exact paper 10 m forward segments for active crossings.

    Each segment starts at the current agent position and extends exactly
    5 m in the same current heading direction used by the detector.
    """
    if (
        plot_edges is None
        or plot_edges.empty
        or entity_table is None
        or entity_table.empty
    ):
        return 0

    entity_by_token = entity_table.set_index("track_token", drop=False)
    drawn_pairs = set()
    line_length_m = float(VIDEO_CROSSES_REFERENCE_RAY_LENGTH_M)
    drawn_count = 0

    def _row_for_token(token):
        if token not in entity_by_token.index:
            return None
        row = entity_by_token.loc[token]
        if isinstance(row, pd.DataFrame):
            row = row.iloc[0]
        return row

    def _draw_line(row, *, color, linestyle, label_text, zorder):
        try:
            x = float(row["x"])
            y = float(row["y"])
            heading = float(row["heading"])
        except (TypeError, ValueError, KeyError):
            return False

        if not all(math.isfinite(value) for value in (x, y, heading)):
            return False

        dx = line_length_m * math.cos(heading)
        dy = line_length_m * math.sin(heading)
        x0, y0 = x, y
        x1, y1 = x + dx, y + dy

        axis.plot(
            [x0, x1],
            [y0, y1],
            linestyle=linestyle,
            linewidth=2.0,
            color=color,
            alpha=0.90,
            zorder=zorder,
        )

        # Put an inline label close to the agent rather than adding a legend.
        label_distance_m = min(3.5, 0.70 * line_length_m)
        label_x = x + label_distance_m * math.cos(heading)
        label_y = y + label_distance_m * math.sin(heading)
        axis.text(
            label_x,
            label_y,
            label_text,
            fontsize=7.0,
            fontweight="bold",
            color=color,
            ha="left",
            va="bottom",
            zorder=zorder + 0.2,
            clip_on=True,
            bbox={
                "boxstyle": "round,pad=0.12",
                "facecolor": "white",
                "edgecolor": color,
                "alpha": 0.82,
                "linewidth": 0.5,
            },
        )
        return True

    for edge in plot_edges.itertuples(index=False):
        edge_predicates = {
            str(predicate)
            for predicate in _excel_predicate_list(
                getattr(edge, "predicates", [])
            )
        }
        if "np:crossesInFrontOf" not in edge_predicates:
            continue

        subject_token = str(getattr(edge, "subject_token", ""))
        object_token = str(getattr(edge, "object_token", ""))
        if not subject_token or not object_token:
            continue

        pair_key = (subject_token, object_token)
        if pair_key in drawn_pairs:
            continue
        drawn_pairs.add(pair_key)

        subject_row = _row_for_token(subject_token)
        object_row = _row_for_token(object_token)
        if subject_row is None or object_row is None:
            continue

        subject_drawn = _draw_line(
            subject_row,
            color="#8e24aa",
            linestyle="-.",
            label_text="S forward",
            zorder=8.15,
        )
        object_drawn = _draw_line(
            object_row,
            color="#fb8c00",
            linestyle="--",
            label_text="O forward",
            zorder=8.20,
        )

        if subject_drawn or object_drawn:
            drawn_count += 1

    return drawn_count


# ============================================================
# v9.5.38 TRAFFIC-LIGHT VIDEO OVERLAY
# ============================================================

_VIDEO_TL_SEARCH_RADIUS_M = 20.0
_VIDEO_TL_RELEVANCE_COLOR = "#d81b60"


def _video_tl_signal_connector_id(signal_id):
    text = str(signal_id or "").strip()
    marker = ":control:"
    if text.startswith("traffic_signal:") and marker in text:
        connector_id = text.rsplit(marker, 1)[-1].strip()
        return connector_id or None
    return None


def _video_tl_movement_connector_id(movement_id):
    text = str(movement_id or "").strip()
    marker = ":lane_connector:"
    if text.startswith("controlled_movement:") and marker in text:
        connector_id = text.rsplit(marker, 1)[-1].strip()
        return connector_id or None
    return None


def _video_tl_state_name(state_id):
    text = str(state_id or "").strip()
    prefix = "signal_state:"
    if text.startswith(prefix):
        value = text[len(prefix):].strip().upper()
        if value in {"RED", "YELLOW", "GREEN", "UNKNOWN"}:
            return value
    return "UNKNOWN"


def _video_tl_state_color(state):
    return {
        "RED": "#d32f2f",
        "YELLOW": "#f9a825",
        "GREEN": "#2e7d32",
        "UNKNOWN": "#616161",
    }.get(str(state or "UNKNOWN").upper(), "#616161")


def _video_tl_get_map_object(map_api, object_id, layer):
    if map_api is None:
        return None
    for method_name in ("get_map_object", "get_one_map_object"):
        method = getattr(map_api, method_name, None)
        if method is None:
            continue
        try:
            obj = method(str(object_id), layer)
        except Exception:
            continue
        if obj is not None:
            return obj
    return None


def _video_tl_get_connector(map_api, connector_id):
    if not NUPLAN_MAP_IMPORTS_AVAILABLE:
        return None
    return _video_tl_get_map_object(
        map_api,
        connector_id,
        SemanticMapLayer.LANE_CONNECTOR,
    )


def _video_tl_baseline_points(connector):
    baseline = getattr(connector, "baseline_path", None)
    path = getattr(baseline, "discrete_path", None)
    if path is None:
        return np.empty((0, 2), dtype=float)

    points = []
    for pose in list(path):
        try:
            points.append((float(pose.x), float(pose.y)))
        except Exception:
            continue
    return np.asarray(points, dtype=float)


def _video_tl_connector_polygon(connector):
    polygon = getattr(connector, "polygon", None)
    exterior = getattr(polygon, "exterior", None)
    if exterior is None:
        return np.empty((0, 2), dtype=float)
    try:
        return np.asarray(exterior.coords, dtype=float)
    except Exception:
        return np.empty((0, 2), dtype=float)


def _video_tl_connector_entry(connector):
    points = _video_tl_baseline_points(connector)
    if len(points):
        return np.asarray(points[0], dtype=float)

    polygon = getattr(connector, "polygon", None)
    centroid = getattr(polygon, "centroid", None)
    if centroid is not None:
        try:
            return np.array(
                [float(centroid.x), float(centroid.y)],
                dtype=float,
            )
        except Exception:
            pass
    return None


def _video_tl_connector_center(connector):
    points = _video_tl_baseline_points(connector)
    if len(points):
        return np.mean(points, axis=0)

    polygon = getattr(connector, "polygon", None)
    centroid = getattr(polygon, "centroid", None)
    if centroid is not None:
        try:
            return np.array(
                [float(centroid.x), float(centroid.y)],
                dtype=float,
            )
        except Exception:
            pass
    return None


def _video_tl_geometry(obj):
    for name in ("polygon", "linestring", "line", "geometry"):
        geometry = getattr(obj, name, None)
        if geometry is not None:
            return geometry
    return None


def _video_tl_anchor(obj):
    if obj is None:
        return None

    geometry = _video_tl_geometry(obj)
    if geometry is not None:
        for attr_name in ("representative_point", "centroid"):
            try:
                attr = getattr(geometry, attr_name)
                point = attr() if callable(attr) else attr
                return np.array(
                    [float(point.x), float(point.y)],
                    dtype=float,
                )
            except Exception:
                continue

    for candidate in (
        getattr(obj, "point", None),
        getattr(obj, "center", None),
    ):
        if candidate is None:
            continue
        try:
            return np.array(
                [float(candidate.x), float(candidate.y)],
                dtype=float,
            )
        except Exception:
            continue

    baseline = getattr(obj, "baseline_path", None)
    path = getattr(baseline, "discrete_path", None)
    if path:
        for pose in list(path):
            try:
                return np.array(
                    [float(pose.x), float(pose.y)],
                    dtype=float,
                )
            except Exception:
                continue

    return None


def _video_tl_proximal_objects(map_api, center_xy, radius_m, layer):
    if (
        map_api is None
        or center_xy is None
        or not NUPLAN_MAP_IMPORTS_AVAILABLE
    ):
        return []

    try:
        result = map_api.get_proximal_map_objects(
            Point2D(
                float(center_xy[0]),
                float(center_xy[1]),
            ),
            float(radius_m),
            [layer],
        )
        return list(result.get(layer, []))
    except Exception:
        return []


def _video_tl_select_physical_signal(map_api, connector):
    entry = _video_tl_connector_entry(connector)
    if entry is None or not hasattr(SemanticMapLayer, "TRAFFIC_LIGHT"):
        return None

    candidates = _video_tl_proximal_objects(
        map_api,
        entry,
        _VIDEO_TL_SEARCH_RADIUS_M,
        SemanticMapLayer.TRAFFIC_LIGHT,
    )

    scored = []
    for obj in candidates:
        anchor = _video_tl_anchor(obj)
        if anchor is None:
            continue
        scored.append((
            float(np.linalg.norm(anchor - entry)),
            obj,
        ))

    if not scored:
        return None

    scored.sort(key=lambda item: item[0])
    return scored[0][1]


def _video_tl_draw_map_object(
    ax,
    obj,
    *,
    facecolor,
    edgecolor,
    alpha=0.55,
    linewidth=2.5,
    zorder=12,
):
    geometry = _video_tl_geometry(obj)

    if geometry is None:
        anchor = _video_tl_anchor(obj)
        if anchor is not None:
            ax.scatter(
                [float(anchor[0])],
                [float(anchor[1])],
                s=150,
                marker="s",
                c=[facecolor],
                edgecolors=edgecolor,
                linewidths=linewidth,
                alpha=alpha,
                zorder=zorder,
            )
        return

    geom_type = str(getattr(geometry, "geom_type", ""))

    if geom_type in {"Polygon", "MultiPolygon"}:
        try:
            parts = list(_iter_polygon_parts(geometry))
        except Exception:
            parts = [geometry]

        for polygon in parts:
            try:
                coords = np.asarray(
                    polygon.exterior.coords,
                    dtype=float,
                )
            except Exception:
                continue
            ax.add_patch(
                MplPolygon(
                    coords,
                    closed=True,
                    facecolor=facecolor,
                    edgecolor=edgecolor,
                    alpha=alpha,
                    linewidth=linewidth,
                    zorder=zorder,
                )
            )
        return

    if geom_type in {"LineString", "LinearRing"}:
        try:
            coords = np.asarray(geometry.coords, dtype=float)
        except Exception:
            coords = np.empty((0, 2), dtype=float)
        if len(coords):
            ax.plot(
                coords[:, 0],
                coords[:, 1],
                color=edgecolor,
                alpha=alpha,
                linewidth=linewidth,
                zorder=zorder,
            )
        return

    anchor = _video_tl_anchor(obj)
    if anchor is not None:
        ax.scatter(
            [float(anchor[0])],
            [float(anchor[1])],
            s=150,
            marker="s",
            c=[facecolor],
            edgecolors=edgecolor,
            linewidths=linewidth,
            alpha=alpha,
            zorder=zorder,
        )


def _video_tl_canonical_state_by_signal(frame_assertions):
    result = {}

    if (
        frame_assertions is None
        or frame_assertions.empty
        or "predicate_id" not in frame_assertions.columns
    ):
        return result

    rows = frame_assertions.loc[
        frame_assertions["predicate_id"]
        .astype(str)
        .eq("np:hasSignalState")
    ]

    for row in rows.itertuples(index=False):
        signal_id = str(
            getattr(row, "subject_id", "")
        ).strip()
        state_id = str(
            getattr(row, "object_id", "")
        ).strip()
        if signal_id:
            result[signal_id] = _video_tl_state_name(state_id)

    return result


def _video_tl_native_state_by_connector(frame):
    result = {}
    for light in getattr(frame, "traffic_lights", []) or []:
        connector_id = str(
            getattr(light, "lane_connector_id", "")
        ).strip()
        if not connector_id:
            continue

        raw = getattr(light, "status", "UNKNOWN")
        state = str(getattr(raw, "name", raw)).upper()
        if "." in state:
            state = state.rsplit(".", 1)[-1]
        if state == "AMBER":
            state = "YELLOW"
        if state not in {"RED", "YELLOW", "GREEN"}:
            state = "UNKNOWN"
        result[connector_id] = state
    return result


def _video_tl_selected(predicate_id, semantic_predicates):
    return semantic_predicate_is_selected(
        predicate_id,
        semantic_predicates,
    )


def _video_tl_relation_enabled(relation_type, semantic_relations):
    return bool(
        semantic_relations.get(
            str(relation_type),
            False,
        )
    )



_VIDEO_TL_FRAME_MATCH_TOLERANCE_US = 300_000


def _video_tl_nearest_assertion_snapshot(
    scenario_assertions,
    timestamp_us,
    tolerance_us=_VIDEO_TL_FRAME_MATCH_TOLERANCE_US,
):
    """
    Return the nearest traffic-light assertion snapshot to one nuPlan frame.

    Traffic-light assertion timestamps are not guaranteed to be bit-identical
    to frame.timestamp_us.  The generic video path historically required exact
    equality, which caused valid traffic-light predicates to disappear from
    batch videos and video_edge_events.csv.

    The nearest traffic-light assertion timestamp is accepted only when it is
    within tolerance_us of the displayed frame.
    """
    if (
        scenario_assertions is None
        or scenario_assertions.empty
        or "predicate_id" not in scenario_assertions.columns
        or "valid_time_us" not in scenario_assertions.columns
    ):
        return scenario_assertions.iloc[0:0].copy() if scenario_assertions is not None else None

    traffic_ids = {
        "np:controls",
        "np:hasSignalState",
        "np:isRelevantSignal",
    }

    rows = scenario_assertions.loc[
        scenario_assertions["predicate_id"].astype(str).isin(traffic_ids)
    ].copy()

    if rows.empty:
        return rows

    times = pd.to_numeric(rows["valid_time_us"], errors="coerce")
    good = times.notna()
    if not good.any():
        return rows.iloc[0:0].copy()

    rows = rows.loc[good].copy()
    times = times.loc[good].astype("int64")

    target = int(timestamp_us)
    deltas = (times - target).abs()
    nearest_index = deltas.idxmin()
    nearest_time = int(times.loc[nearest_index])
    nearest_delta = abs(nearest_time - target)

    if nearest_delta > int(tolerance_us):
        return rows.iloc[0:0].copy()

    snapshot = rows.loc[times.eq(nearest_time)].copy()
    snapshot["_video_frame_timestamp_us"] = target
    snapshot["_video_assertion_timestamp_us"] = nearest_time
    snapshot["_video_time_delta_us"] = nearest_delta
    return snapshot


def _video_tl_build_overlay_rows(
    frame,
    frame_assertions,
    entity_table,
    *,
    semantic_categories,
    semantic_relations,
    semantic_predicates,
    scenario_assertions=None,
    timestamp_us=None,
    frame_match_tolerance_us=_VIDEO_TL_FRAME_MATCH_TOLERANCE_US,
):
    """
    Build selected traffic-light events directly from the assertion table.

    This intentionally bypasses semantic_edge_table(), whose generic
    entity-token filtering is designed for tracked actors and historically
    removed canonical traffic_signal / controlled_movement resources.
    """
    if not bool(
        semantic_categories.get("traffic_light", False)
    ):
        return []

    selected_predicates = {
        predicate_id
        for predicate_id in (
            "np:controls",
            "np:hasSignalState",
            "np:isRelevantSignal",
        )
        if _video_tl_selected(
            predicate_id,
            semantic_predicates,
        )
    }

    if not selected_predicates:
        return []

    # Traffic-light assertions need tolerant timestamp alignment.  Prefer a
    # nearest complete traffic-light snapshot from the whole scenario; fall
    # back to the generic exact-frame assertions only when necessary.
    tl_assertions = None

    if (
        scenario_assertions is not None
        and timestamp_us is not None
    ):
        tl_assertions = _video_tl_nearest_assertion_snapshot(
            scenario_assertions,
            timestamp_us,
            tolerance_us=frame_match_tolerance_us,
        )

    if (
        tl_assertions is None
        or tl_assertions.empty
    ):
        if (
            frame_assertions is None
            or frame_assertions.empty
            or "predicate_id" not in frame_assertions.columns
        ):
            return []
        tl_assertions = frame_assertions

    rows = tl_assertions.loc[
        tl_assertions["predicate_id"]
        .astype(str)
        .isin(selected_predicates)
    ]

    if rows.empty:
        return []

    display_lookup = {}
    if (
        entity_table is not None
        and not entity_table.empty
        and "track_token" in entity_table.columns
    ):
        display_lookup = (
            entity_table
            .drop_duplicates("track_token")
            .set_index("track_token")["display_id"]
            .astype(str)
            .to_dict()
        )

    canonical_states = _video_tl_canonical_state_by_signal(
        tl_assertions
    )
    native_states = _video_tl_native_state_by_connector(frame)

    events = []
    seen = set()

    for assertion in rows.itertuples(index=False):
        predicate_id = str(
            getattr(assertion, "predicate_id", "")
        )
        subject_id = str(
            getattr(assertion, "subject_id", "")
        ).strip()
        object_id = str(
            getattr(assertion, "object_id", "")
        ).strip()

        connector_id = None
        relation_type = "structure_to_structure"
        subject_track_token = None

        if predicate_id == "np:isRelevantSignal":
            connector_id = _video_tl_signal_connector_id(
                object_id
            )
            subject_track_token = str(
                entity_id_to_track_token(subject_id)
            )
            relation_type = (
                "ego_to_structure"
                if subject_track_token == "ego"
                else "agent_to_structure"
            )
            signal_id = object_id

        elif predicate_id == "np:controls":
            connector_id = (
                _video_tl_movement_connector_id(object_id)
                or _video_tl_signal_connector_id(subject_id)
            )
            signal_id = subject_id

        elif predicate_id == "np:hasSignalState":
            connector_id = _video_tl_signal_connector_id(
                subject_id
            )
            signal_id = subject_id

        else:
            continue

        if not connector_id:
            continue

        # In exact-predicate video mode, the selected traffic-light predicate
        # is authoritative. Generic relation switches were designed for the
        # agent-only graph renderer and historically suppressed
        # structure_to_structure traffic-light relations such as controls and
        # hasSignalState. Respect relation switches only when displaying all
        # predicates together.
        _exact_tl_selection = normalize_semantic_predicate_selection(
            semantic_predicates
        )
        if (
            _exact_tl_selection is None
            and not _video_tl_relation_enabled(
                relation_type,
                semantic_relations,
            )
        ):
            continue

        state = canonical_states.get(
            signal_id,
            native_states.get(
                connector_id,
                "UNKNOWN",
            ),
        )

        if predicate_id == "np:isRelevantSignal":
            subject_display = display_lookup.get(
                subject_track_token,
                short_token(subject_track_token),
            )
            object_display = f"TrafficSignal:{connector_id}"
        elif predicate_id == "np:controls":
            subject_display = f"TrafficSignal:{connector_id}"
            object_display = f"ControlledMovement:{connector_id}"
        else:
            subject_display = f"TrafficSignal:{connector_id}"
            object_display = state

        key = (
            predicate_id,
            subject_id,
            object_id,
            connector_id,
        )
        if key in seen:
            continue
        seen.add(key)

        events.append({
            "predicate_id": predicate_id,
            "relation_labels": predicate_id.replace("np:", ""),
            "category": "traffic_light",
            "relation_type": relation_type,
            "subject_id": subject_id,
            "object_id": object_id,
            "subject": subject_display,
            "object": object_display,
            "subject_track_token": subject_track_token,
            "signal_id": signal_id,
            "connector_id": connector_id,
            "state": state,
            "assertion_timestamp_us": (
                int(getattr(assertion, "_video_assertion_timestamp_us"))
                if hasattr(assertion, "_video_assertion_timestamp_us")
                else int(getattr(assertion, "valid_time_us", timestamp_us or 0))
            ),
            "frame_time_delta_us": (
                int(getattr(assertion, "_video_time_delta_us"))
                if hasattr(assertion, "_video_time_delta_us")
                else 0
            ),
        })

    return events


def _video_tl_draw_overlays(
    ax,
    frame,
    entity_table,
    events,
    *,
    draw_semantic_arrows_on_map=True,
):
    """
    Draw the traffic-light family on the ordinary ego-centered semantic map.

      Agent --isRelevantSignal--> TrafficSignal
      TrafficSignal --controls--> ControlledMovement
      TrafficSignal [RED/GREEN/YELLOW/UNKNOWN]
    """
    if not events:
        return {
            "event_count": 0,
            "drawn_is_relevant_signal": 0,
            "drawn_controls": 0,
            "drawn_states": 0,
        }

    map_api = getattr(frame, "map_api", None)

    entity_lookup = {}
    if (
        entity_table is not None
        and not entity_table.empty
        and "track_token" in entity_table.columns
    ):
        entity_lookup = (
            entity_table
            .drop_duplicates("track_token")
            .set_index("track_token", drop=False)
        )

    connector_ids = sorted({
        str(event["connector_id"])
        for event in events
        if event.get("connector_id")
    })

    connector_cache = {}
    signal_anchor_cache = {}
    state_by_connector = {}

    for event in events:
        connector_id = str(event["connector_id"])
        state_by_connector.setdefault(
            connector_id,
            str(event.get("state") or "UNKNOWN"),
        )

    # Draw each controlled connector / physical signal once as context.
    for connector_id in connector_ids:
        connector = _video_tl_get_connector(
            map_api,
            connector_id,
        )
        if connector is None:
            continue

        connector_cache[connector_id] = connector
        state = state_by_connector.get(
            connector_id,
            "UNKNOWN",
        )
        state_color = _video_tl_state_color(state)

        polygon = _video_tl_connector_polygon(connector)
        baseline = _video_tl_baseline_points(connector)
        center = _video_tl_connector_center(connector)
        entry = _video_tl_connector_entry(connector)

        if len(polygon):
            ax.add_patch(
                MplPolygon(
                    polygon,
                    closed=True,
                    facecolor=state_color,
                    edgecolor="#006064",
                    alpha=0.16,
                    linewidth=2.2,
                    zorder=6.0,
                )
            )

        if len(baseline):
            ax.plot(
                baseline[:, 0],
                baseline[:, 1],
                color="#006064",
                linewidth=3.0,
                alpha=0.90,
                zorder=7.0,
            )

        physical_signal = _video_tl_select_physical_signal(
            map_api,
            connector,
        )

        if physical_signal is not None:
            _video_tl_draw_map_object(
                ax,
                physical_signal,
                facecolor=state_color,
                edgecolor="#212121",
                alpha=0.72,
                linewidth=2.8,
                zorder=13,
            )
            signal_anchor = _video_tl_anchor(
                physical_signal
            )
        else:
            signal_anchor = entry
            if signal_anchor is not None:
                ax.scatter(
                    [float(signal_anchor[0])],
                    [float(signal_anchor[1])],
                    s=180,
                    marker="s",
                    c=[state_color],
                    edgecolors="#212121",
                    linewidths=2.0,
                    zorder=13,
                )

        if signal_anchor is not None:
            signal_anchor_cache[connector_id] = signal_anchor
            ax.text(
                float(signal_anchor[0]),
                float(signal_anchor[1]) + 1.7,
                f"TrafficSignal\n{state}",
                ha="center",
                va="bottom",
                fontsize=7.5,
                fontweight="bold",
                zorder=16,
                bbox={
                    "boxstyle": "round,pad=0.20",
                    "facecolor": "white",
                    "edgecolor": state_color,
                    "alpha": 0.92,
                },
            )

        if center is not None:
            ax.text(
                float(center[0]),
                float(center[1]),
                f"LC:{connector_id}",
                ha="center",
                va="center",
                fontsize=6.5,
                zorder=15,
                bbox={
                    "boxstyle": "round,pad=0.16",
                    "facecolor": "white",
                    "edgecolor": "#006064",
                    "alpha": 0.82,
                },
            )

    diagnostics = {
        "event_count": int(len(events)),
        "drawn_is_relevant_signal": 0,
        "drawn_controls": 0,
        "drawn_states": 0,
    }

    for event in events:
        predicate_id = str(event["predicate_id"])
        connector_id = str(event["connector_id"])
        connector = connector_cache.get(connector_id)
        signal_anchor = signal_anchor_cache.get(
            connector_id
        )

        if predicate_id == "np:hasSignalState":
            # The state is a semantic node without its own map coordinate.
            # Its value is therefore represented by the label on the signal.
            if signal_anchor is not None:
                diagnostics["drawn_states"] += 1
            continue

        if not bool(draw_semantic_arrows_on_map):
            continue

        if (
            predicate_id == "np:controls"
            and connector is not None
            and signal_anchor is not None
        ):
            target = _video_tl_connector_center(
                connector
            )
            if target is not None:
                ax.annotate(
                    "controls",
                    xy=(
                        float(target[0]),
                        float(target[1]),
                    ),
                    xytext=(
                        float(signal_anchor[0]),
                        float(signal_anchor[1]),
                    ),
                    arrowprops={
                        "arrowstyle": "-|>",
                        "color": "#c62828",
                        "linewidth": 2.6,
                        "shrinkA": 8,
                        "shrinkB": 7,
                    },
                    color="#c62828",
                    fontsize=8,
                    fontweight="bold",
                    ha="center",
                    va="center",
                    zorder=17,
                )
                diagnostics["drawn_controls"] += 1
            continue

        if (
            predicate_id == "np:isRelevantSignal"
            and signal_anchor is not None
        ):
            token = str(
                event.get("subject_track_token") or ""
            )
            if token not in entity_lookup.index:
                continue

            source = entity_lookup.loc[token]
            if isinstance(source, pd.DataFrame):
                source = source.iloc[0]

            source_x = float(source["x"])
            source_y = float(source["y"])

            ax.annotate(
                "isRelevantSignal",
                xy=(
                    float(signal_anchor[0]),
                    float(signal_anchor[1]),
                ),
                xytext=(source_x, source_y),
                arrowprops={
                    "arrowstyle": "-|>",
                    "color": _VIDEO_TL_RELEVANCE_COLOR,
                    "linewidth": 3.3,
                    "alpha": 0.96,
                    "shrinkA": 9,
                    "shrinkB": 9,
                },
                color=_VIDEO_TL_RELEVANCE_COLOR,
                fontsize=8,
                fontweight="bold",
                ha="center",
                va="center",
                zorder=18,
            )

            # Strong outline so the participating road user is obvious.
            try:
                corners = oriented_rectangle_corners(
                    source_x,
                    source_y,
                    float(source["heading"]),
                    source["length"],
                    source["width"],
                )
                ax.add_patch(
                    MplPolygon(
                        corners,
                        closed=True,
                        facecolor="none",
                        edgecolor=_VIDEO_TL_RELEVANCE_COLOR,
                        linewidth=3.0,
                        zorder=17,
                    )
                )
            except Exception:
                pass

            diagnostics["drawn_is_relevant_signal"] += 1

    return diagnostics



# ============================================================
# DIRECT EXACT-PREDICATE VISUALIZATION FALLBACK (v9.5.38 fix2)
# ============================================================
#
# The generic semantic-edge renderer is intentionally graph-oriented. It
# therefore drops unary literal predicates (object_id is null) and historically
# discards agent->map-structure edges because map structure IDs are not tracked
# agent tokens. Exact-predicate validation needs both classes to remain visible
# and to be written to video_edge_events.csv.
#
# This fallback is active only when one or more exact predicates are selected.
# Pairwise predicates already represented by the normal semantic edge table are
# left untouched.
# ============================================================

_VIDEO_DIRECT_MAP_LAYER_NAMES = {
    "lane": "LANE",
    "lane_connector": "LANE_CONNECTOR",
    "roadblock": "ROADBLOCK",
    "roadblock_connector": "ROADBLOCK_CONNECTOR",
    "intersection": "INTERSECTION",
    "crosswalk": "CROSSWALK",
    "stop_line": "STOP_LINE",
    "traffic_light": "TRAFFIC_LIGHT",
    "walkway": "WALKWAYS",
    "carpark": "CARPARK_AREA",
}


def _video_direct_definition_meta(predicate_id):
    if (
        DEFINITIONS is None
        or DEFINITIONS.empty
        or "predicate_id" not in DEFINITIONS.columns
    ):
        return {}
    rows = DEFINITIONS.loc[
        DEFINITIONS["predicate_id"].astype(str).eq(str(predicate_id))
    ]
    if rows.empty:
        return {}
    row = rows.iloc[0]
    result = {}
    for key in ("category", "label", "units", "value_type"):
        if key in rows.columns:
            value = row.get(key)
            if value is not None and str(value).lower() != "nan":
                result[key] = value
    return result


def _video_direct_value(value_json):
    try:
        return _decoded_assertion_value(value_json)
    except Exception:
        pass
    if value_json is None:
        return None
    text = str(value_json).strip()
    if not text or text.lower() in {"none", "nan", "null"}:
        return None
    try:
        return json.loads(text)
    except Exception:
        return text


def _video_direct_value_text(value, unit=None):
    if isinstance(value, float):
        if math.isfinite(value):
            text = f"{value:.3f}".rstrip("0").rstrip(".")
        else:
            text = str(value)
    elif isinstance(value, (list, tuple)):
        text = "[" + ", ".join(map(str, value)) + "]"
    else:
        text = str(value)
    unit_text = ""
    if unit is not None and str(unit).strip() and str(unit).lower() != "nan":
        unit_text = f" {unit}"
    return text + unit_text


def _video_direct_map_object(map_api, entity_id):
    if map_api is None or not NUPLAN_MAP_IMPORTS_AVAILABLE:
        return None
    text_id = str(entity_id or "").strip()
    if ":" not in text_id:
        return None
    kind, native_id = text_id.split(":", 1)
    layer_name = _VIDEO_DIRECT_MAP_LAYER_NAMES.get(kind)
    if not layer_name or not hasattr(SemanticMapLayer, layer_name):
        return None
    layer = getattr(SemanticMapLayer, layer_name)
    return _video_tl_get_map_object(map_api, native_id, layer)


def _video_direct_entity_lookup(entity_table):
    if (
        entity_table is None
        or entity_table.empty
        or "track_token" not in entity_table.columns
    ):
        return {}
    result = {}
    for row in entity_table.to_dict("records"):
        result[str(row.get("track_token"))] = row
    return result


def _video_direct_exact_assertion_events(
    frame,
    frame_assertions,
    entity_table,
    *,
    semantic_categories,
    semantic_predicates,
):
    """Build direct drawable events omitted by the generic edge renderer."""
    selected = normalize_semantic_predicate_selection(semantic_predicates)
    if selected is None:
        return []
    if frame_assertions is None or frame_assertions.empty:
        return []
    if "predicate_id" not in frame_assertions.columns:
        return []

    active_categories = enabled_names(semantic_categories)
    entity_lookup = _video_direct_entity_lookup(entity_table)
    events = []
    seen = set()

    for predicate_id in selected:
        # Traffic-light predicates have their own physically grounded overlay.
        if predicate_id in {
            "np:controls",
            "np:hasSignalState",
            "np:isRelevantSignal",
        }:
            continue

        meta = _video_direct_definition_meta(predicate_id)
        category = str(meta.get("category", ""))
        if category and category not in active_categories:
            continue
        value_type = str(meta.get("value_type", "")).strip().lower()
        unit = meta.get("units")

        rows = frame_assertions.loc[
            frame_assertions["predicate_id"].astype(str).eq(predicate_id)
        ]
        if rows.empty:
            continue

        for assertion in rows.itertuples(index=False):
            subject_id = str(getattr(assertion, "subject_id", "") or "")
            object_raw = getattr(assertion, "object_id", None)
            object_id = "" if object_raw is None else str(object_raw)
            if object_id.lower() in {"none", "nan", "null"}:
                object_id = ""

            subject_kind = _entity_kind(subject_id)
            object_kind = _entity_kind(object_id) if object_id else "other"
            subject_token = str(entity_id_to_track_token(subject_id))

            # Pairwise measurements are already converted to subject->object
            # edges by semantic_edge_table(). Do not duplicate them here.
            if subject_id.startswith("pair:"):
                continue

            # Unary literal/state predicates: attach a value label to the
            # physical subject (ego or tracked agent).
            if value_type != "entity" and subject_kind in {"ego", "agent"}:
                if subject_token not in entity_lookup:
                    continue
                value = _video_direct_value(
                    getattr(assertion, "value_json", None)
                )
                if value is None:
                    continue
                value_text = _video_direct_value_text(value, unit)
                key = (predicate_id, subject_id, value_text)
                if key in seen:
                    continue
                seen.add(key)
                events.append({
                    "visualization_type": "subject_value",
                    "predicate_id": predicate_id,
                    "relation_labels": predicate_id.replace("np:", ""),
                    "category": category,
                    "relation_type": "entity_value",
                    "subject_id": subject_id,
                    "object_id": "",
                    "subject_track_token": subject_token,
                    "value": value,
                    "value_text": value_text,
                    "unit": unit,
                })
                continue

            # Entity-valued agent/ego -> map structure predicates. The normal
            # graph renderer cannot draw these because a lane/crosswalk/etc. is
            # not a tracked-agent token. Draw the map geometry directly.
            if (
                value_type == "entity"
                and subject_kind in {"ego", "agent"}
                and object_kind == "structure"
            ):
                if subject_token not in entity_lookup:
                    continue
                key = (predicate_id, subject_id, object_id)
                if key in seen:
                    continue
                seen.add(key)
                events.append({
                    "visualization_type": "entity_to_map_structure",
                    "predicate_id": predicate_id,
                    "relation_labels": predicate_id.replace("np:", ""),
                    "category": category,
                    "relation_type": (
                        "ego_to_structure"
                        if subject_kind == "ego"
                        else "agent_to_structure"
                    ),
                    "subject_id": subject_id,
                    "object_id": object_id,
                    "subject_track_token": subject_token,
                    "value": None,
                    "value_text": object_id,
                    "unit": None,
                })

    return events


def _video_direct_predicate_agent_tokens(events):
    return {
        str(event.get("subject_track_token"))
        for event in events
        if event.get("subject_track_token")
    }


def _video_direct_draw_events(
    ax,
    frame,
    entity_table,
    events,
    *,
    draw_semantic_arrows_on_map=True,
):
    if not events:
        return {
            "drawn_value_labels": 0,
            "drawn_map_relations": 0,
        }

    entity_lookup = _video_direct_entity_lookup(entity_table)
    map_api = getattr(frame, "map_api", None)
    value_stack = {}
    drawn_value = 0
    drawn_map = 0

    for event in events:
        token = str(event.get("subject_track_token") or "")
        subject = entity_lookup.get(token)
        if subject is None:
            continue
        sx = float(subject.get("x", 0.0))
        sy = float(subject.get("y", 0.0))
        predicate_label = str(event.get("relation_labels") or "predicate")
        vtype = event.get("visualization_type")

        if vtype == "subject_value":
            stack_index = value_stack.get(token, 0)
            value_stack[token] = stack_index + 1
            dy = 1.7 + 1.15 * stack_index
            label = f"{predicate_label}={event.get('value_text', '')}"
            ax.text(
                sx,
                sy + dy,
                label,
                fontsize=7.0,
                ha="center",
                va="bottom",
                zorder=22,
                clip_on=True,
                bbox={
                    "boxstyle": "round,pad=0.22",
                    "facecolor": "white",
                    "edgecolor": "#37474f",
                    "alpha": 0.94,
                    "linewidth": 0.9,
                },
            )
            drawn_value += 1
            continue

        if vtype == "entity_to_map_structure":
            object_id = str(event.get("object_id") or "")
            map_obj = _video_direct_map_object(map_api, object_id)
            anchor = _video_tl_anchor(map_obj) if map_obj is not None else None

            if map_obj is not None:
                _video_tl_draw_map_object(
                    ax,
                    map_obj,
                    facecolor="#ffd54f",
                    edgecolor="#6d4c41",
                    alpha=0.38,
                    linewidth=2.4,
                    zorder=10,
                )

            if anchor is not None:
                tx, ty = float(anchor[0]), float(anchor[1])
                if draw_semantic_arrows_on_map:
                    ax.annotate(
                        predicate_label,
                        xy=(tx, ty),
                        xytext=(sx, sy),
                        arrowprops={
                            "arrowstyle": "-|>",
                            "linewidth": 2.0,
                            "color": "#6a1b9a",
                            "alpha": 0.92,
                            "shrinkA": 8,
                            "shrinkB": 5,
                        },
                        color="#6a1b9a",
                        fontsize=7.2,
                        fontweight="bold",
                        ha="center",
                        va="center",
                        zorder=21,
                    )
                ax.text(
                    tx,
                    ty,
                    object_id,
                    fontsize=6.5,
                    ha="center",
                    va="center",
                    zorder=20,
                    bbox={
                        "boxstyle": "round,pad=0.18",
                        "facecolor": "white",
                        "edgecolor": "#6d4c41",
                        "alpha": 0.88,
                    },
                )
            else:
                # Geometry lookup should normally succeed. This fallback keeps
                # the assertion visible even on map API/version mismatches.
                ax.text(
                    sx,
                    sy + 1.7,
                    f"{predicate_label} → {object_id}",
                    fontsize=7.0,
                    ha="center",
                    va="bottom",
                    zorder=22,
                    bbox={
                        "boxstyle": "round,pad=0.22",
                        "facecolor": "white",
                        "edgecolor": "#6a1b9a",
                        "alpha": 0.94,
                    },
                )
            drawn_map += 1

    return {
        "drawn_value_labels": int(drawn_value),
        "drawn_map_relations": int(drawn_map),
    }

def render_full_scene_configurable_semantic_video(
    *,
    scenario_selector="random",
    scenario_value=0,
    scenario_seed=42,
    filter_seed=42,
    output_dir=OUTPUT_DIR,
    export_root=VIDEO_EXPORT_ROOT,
    start_frame=0,
    end_frame=None,
    frame_step=1,
    fps=None,
    playback_speed=1.0,
    figsize=(12, 12),
    dpi=120,
    semantic_categories=None,
    semantic_relations=None,
    semantic_predicates="all",
    agent_types=None,
    interested_distance_threshold_m=40.0,
    include_within_distance=True,
    include_forward_corridor=True,
    forward_corridor_length_m=40.0,
    forward_corridor_half_width_m=5.0,
    include_same_lane=True,
    include_predicted_path_intersection=True,
    prediction_horizon_s=5.0,
    prediction_step_s=0.25,
    path_intersection_clearance_m=3.0,
    include_existing_spatial_agents=False,
    filter_edges_to_selected_agents=True,
    manual_highlight_tokens=None,
    map_radius_m=40.0,
    draw_semantic_arrows_on_map=True,
    show_agent_labels=True,
    show_agent_lane_info=False,
    show_map_structure_ids=False,
    show_edge_ids=False,
    show_status_box=True,
    draw_crosses_reference_lines=True,
):
    """
    Render one map-only MP4 using the same filters as Cell 15.

    Only semantic relations whose two endpoints have map coordinates in the
    frame can be drawn as arrows. Literal-valued predicates can still be
    selected in tables, but they do not create a subject-to-object map arrow.
    """
    try:
        import cv2
    except ImportError as exc:
        raise ImportError(
            "OpenCV is required for MP4 writing. Install opencv-python "
            "in the active nuPlan environment."
        ) from exc

    if semantic_categories is None:
        semantic_categories = VIDEO_SEMANTIC_CATEGORIES

    if semantic_relations is None:
        semantic_relations = VIDEO_SEMANTIC_RELATIONS

    if semantic_predicates is None:
        semantic_predicates = "all"

    if agent_types is None:
        agent_types = VIDEO_AGENT_TYPES

    if manual_highlight_tokens is None:
        manual_highlight_tokens = []

    active_categories = sorted(
        enabled_names(semantic_categories)
    )
    active_relations = sorted(
        enabled_names(semantic_relations)
    )
    active_predicate_filter = semantic_predicate_selection_label(
        semantic_predicates
    )

    validate_semantic_predicate_selection(
        semantic_predicates,
        DEFINITIONS,
        active_categories=active_categories,
    )

    if not active_categories:
        raise ValueError(
            "Enable at least one semantic category."
        )

    if not active_relations:
        raise ValueError(
            "Enable at least one semantic relation direction."
        )

    scenario, scenario_catalog_row = select_scenario(
        selector=scenario_selector,
        value=scenario_value,
        random_seed=scenario_seed,
        filter_seed=filter_seed,
    )

    scenario_token = str(scenario.token)
    frames = get_scenario_frames(scenario)

    if not frames:
        raise ValueError(
            "The selected scenario has no frames."
        )

    start_index = max(0, int(start_frame))
    stop_index = (
        len(frames)
        if end_frame is None
        else min(len(frames), int(end_frame) + 1)
    )
    step = max(1, int(frame_step))
    frame_indices = list(
        range(start_index, stop_index, step)
    )

    if not frame_indices:
        raise ValueError(
            "The configured frame range does not contain any frames."
        )

    scenario_assertions = load_assertions_for_scenario(
        str(Path(output_dir)),
        scenario_token,
    )

    if scenario_assertions.empty:
        raise RuntimeError(
            "No assertion rows were found for scenario "
            f"{scenario_token} in {Path(output_dir)}."
        )

    actual_fps = (
        float(fps)
        if fps is not None
        else _video_default_fps(
            frames,
            frame_indices,
            playback_speed,
        )
    )

    if not math.isfinite(actual_fps) or actual_fps <= 0:
        raise ValueError(
            "fps must be a positive finite number."
        )

    export_root = Path(export_root)
    export_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    category_fragment = _video_safe_filename_fragment(
        active_categories,
        "semantic",
    )
    video_path = export_root / (
        f"{scenario_token}_{category_fragment}_map_only.mp4"
    )
    temporary_video_path = export_root / (
        f"{scenario_token}_{category_fragment}_map_only_mp4v_temporary.mp4"
    )

    # Remove stale files from an interrupted or overwritten run.
    video_path.unlink(missing_ok=True)
    temporary_video_path.unlink(missing_ok=True)

    writer = None
    summary_rows = []
    edge_event_rows = []

    first_timestamp_us = int(
        frames[frame_indices[0]].timestamp_us
    )

    try:
        for rendered_index, frame_index in enumerate(frame_indices):
            frame = frames[frame_index]
            timestamp_us = int(frame.timestamp_us)

            frame_assertions = assertions_at_timestamp(
                scenario_assertions,
                timestamp_us,
                definitions=DEFINITIONS,
            )

            entity_table = add_stable_display_ids(
                build_entity_table(
                    frame,
                    scenario_token,
                ),
                scenario,
            )

            entity_table = filter_entities_by_agent_type(
                entity_table,
                agent_types,
            )

            enabled_agent_tokens = set(
                entity_table.loc[
                    ~entity_table["is_ego"].astype(bool),
                    "track_token",
                ].astype(str)
            )

            # Build the current-frame semantic edges using the normal
            # assertion-to-edge conversion.
            semantic_edges_all = semantic_edge_table(
                frame_assertions,
                DEFINITIONS,
                semantic_categories,
                semantic_relations,
                pair_assertions=scenario_assertions,
            )

            # IMPORTANT: use the exact same np:follows compatibility path as
            # the working single-frame Excel plot.  np:follows is an
            # entity-valued relation whose target is stored in object_id and
            # whose value_json is legitimately null.  Older output/definition
            # combinations may therefore omit it from semantic_edge_table().
            # The one-frame plot recovers it from the Follow evidence sheet;
            # here we build that same table in memory for every timestamp and
            # call the same recovery helper before applying the switches.
            if bool(semantic_categories.get("interaction", False)):
                semantic_edges_all = _restore_follow_edges_from_excel(
                    semantic_edges_all,
                    {
                        "Follow evidence": follow_evidence_table(
                            frame_assertions
                        )
                    },
                )

                # np:overtakes is also an entity-valued interaction relation.
                # Recover it directly from the current-frame assertions for
                # the same reason np:follows needs a compatibility fallback.
                semantic_edges_all = _video_restore_overtake_edges(
                    semantic_edges_all,
                    frame_assertions,
                )

                semantic_edges_all = _video_restore_merge_edges(
                    semantic_edges_all,
                    frame_assertions,
                )

            # Risk is rebuilt independently from the interaction family.  For
            # exact risk validation, restore the entity-valued relation
            # directly from the current-frame assertion if the generic edge
            # converter did not retain it.
            if bool(semantic_categories.get("risk", False)):
                semantic_edges_all = _video_restore_risk_edges(
                    semantic_edges_all,
                    frame_assertions,
                )

            # Normalize EGO and agent endpoints BEFORE relation-direction
            # filtering. This prevents an RDF-style EGO ID from being
            # classified or filtered differently from the frame-table EGO.
            semantic_edges_all = _video_normalize_edge_endpoints(
                semantic_edges_all,
                entity_table,
            )
            restored_ego_edge_count = _video_count_ego_edges(
                semantic_edges_all,
                entity_table,
            )

            # Apply the identical category/relation filtering used by
            # plot_from_master_excel() after follow-edge recovery.
            semantic_edges_all = _filter_master_semantic_edges(
                semantic_edges_all,
                semantic_categories,
                semantic_relations,
            )

            # Second visualization filter: exact predicate(s) inside the
            # enabled semantic family/families. The edge predicate list is
            # trimmed so arrow styling and labels cannot leak another
            # predicate carried by the same S->O edge.
            semantic_edges_all = filter_semantic_edges_by_predicates(
                semantic_edges_all,
                semantic_predicates,
            )

            semantic_edges_all = filter_edges_by_enabled_entity_tokens(
                semantic_edges_all,
                entity_table,
            )
            enabled_ego_edge_count = _video_count_ego_edges(
                semantic_edges_all,
                entity_table,
            )

            # Spatial edges are calculated independently because they may be
            # needed by INCLUDE_EXISTING_SPATIAL_AGENTS even when the spatial
            # category itself is disabled for display.
            all_spatial_edges = semantic_edge_table(
                frame_assertions,
                DEFINITIONS,
                _VIDEO_SPATIAL_ONLY_CATEGORIES,
                _VIDEO_ALL_RELATIONS,
                pair_assertions=scenario_assertions,
            )

            all_spatial_edges = _video_normalize_edge_endpoints(
                all_spatial_edges,
                entity_table,
            )
            all_spatial_edges = filter_edges_by_enabled_entity_tokens(
                all_spatial_edges,
                entity_table,
            )

            raw_follow_assertion_count = int(
                frame_assertions["predicate_id"]
                .astype(str)
                .eq("np:follows")
                .sum()
            )
            raw_overtake_assertion_count = int(
                frame_assertions["predicate_id"]
                .astype(str)
                .eq("np:overtakes")
                .sum()
            )
            raw_lane_change_assertion_count = int(
                frame_assertions["predicate_id"]
                .astype(str)
                .eq("np:changesLane")
                .sum()
            )
            recovered_follow_edge_count = int(
                semantic_edges_all["predicates"].map(
                    lambda values: "np:follows" in set(
                        _excel_predicate_list(values)
                    )
                ).sum()
            ) if (
                semantic_edges_all is not None
                and not semantic_edges_all.empty
                and "predicates" in semantic_edges_all.columns
            ) else 0

            candidate_tokens, selection_audit = (
                select_interesting_agents_for_plot(
                    frame,
                    entity_table,
                    all_spatial_edges,
                    distance_threshold_m=(
                        interested_distance_threshold_m
                    ),
                    include_within_distance=(
                        include_within_distance
                    ),
                    include_forward_corridor=(
                        include_forward_corridor
                    ),
                    forward_corridor_length_m=(
                        forward_corridor_length_m
                    ),
                    forward_corridor_half_width_m=(
                        forward_corridor_half_width_m
                    ),
                    include_same_lane=(
                        include_same_lane
                    ),
                    include_predicted_path_intersection=(
                        include_predicted_path_intersection
                    ),
                    prediction_horizon_s=(
                        prediction_horizon_s
                    ),
                    prediction_step_s=(
                        prediction_step_s
                    ),
                    path_intersection_clearance_m=(
                        path_intersection_clearance_m
                    ),
                    include_existing_spatial_agents=(
                        include_existing_spatial_agents
                    ),
                    manual_tokens=(
                        manual_highlight_tokens
                    ),
                )
            )

            candidate_tokens = (
                set(candidate_tokens)
                & enabled_agent_tokens
            )

            semantic_edges = (
                filter_edges_by_selected_agents(
                    semantic_edges_all,
                    candidate_tokens,
                )
                if filter_edges_to_selected_agents
                else semantic_edges_all.copy()
            )
            selected_ego_edge_count = _video_count_ego_edges(
                semantic_edges,
                entity_table,
            )

            # Normalize once more immediately before plotting so that no helper
            # between filtering and prepare_plot_edges() can reintroduce an
            # assertion-side EGO identifier.
            semantic_edges = _video_normalize_edge_endpoints(
                semantic_edges,
                entity_table,
            )
            plot_edges = prepare_plot_edges(
                semantic_edges,
                entity_table,
            )
            plotted_ego_edge_count = _video_count_ego_edges(
                plot_edges,
                entity_table,
            )

            traffic_light_events = _video_tl_build_overlay_rows(
                frame,
                frame_assertions,
                entity_table,
                semantic_categories=semantic_categories,
                semantic_relations=semantic_relations,
                semantic_predicates=semantic_predicates,
                scenario_assertions=scenario_assertions,
                timestamp_us=timestamp_us,
                frame_match_tolerance_us=_VIDEO_TL_FRAME_MATCH_TOLERANCE_US,
            )

            direct_predicate_events = _video_direct_exact_assertion_events(
                frame,
                frame_assertions,
                entity_table,
                semantic_categories=semantic_categories,
                semantic_predicates=semantic_predicates,
            )

            traffic_light_agent_tokens = {
                str(event["subject_track_token"])
                for event in traffic_light_events
                if (
                    event.get("predicate_id") == "np:isRelevantSignal"
                    and event.get("subject_track_token")
                )
            }
            direct_predicate_agent_tokens = (
                _video_direct_predicate_agent_tokens(
                    direct_predicate_events
                )
            )

            predicate_tokens = (
                _agent_tokens_with_displayed_predicates(
                    plot_edges
                )
                | traffic_light_agent_tokens
                | direct_predicate_agent_tokens
            )

            displayed_selected_tokens = (
                set(candidate_tokens)
                & set(predicate_tokens)
            )

            fig, ax = plt.subplots(
                1,
                1,
                figsize=figsize,
                dpi=dpi,
                constrained_layout=True,
            )

            plot_map_with_agents(
                ax,
                frame,
                entity_table,
                plot_edges,
                frame_assertions=frame_assertions,
                scenario_token=scenario_token,
                frame_index=frame_index,
                selected_tokens=displayed_selected_tokens,
                predicate_tokens=predicate_tokens,
                map_radius_m=float(map_radius_m),
                manual_highlight_tokens=(
                    manual_highlight_tokens
                ),
                draw_semantic_arrows=bool(
                    draw_semantic_arrows_on_map
                ),
                show_agent_labels=bool(
                    show_agent_labels
                ),
                show_agent_lane_info=bool(
                    show_agent_lane_info
                ),
                show_map_structure_ids=bool(
                    show_map_structure_ids
                ),
                show_edge_ids=bool(
                    show_edge_ids
                ),
                agent_type_switches=agent_types,
            )

            crosses_reference_pair_count = 0
            if bool(draw_crosses_reference_lines):
                crosses_reference_pair_count = (
                    _video_draw_crosses_reference_lines(
                        ax,
                        plot_edges,
                        entity_table,
                        map_radius_m=float(map_radius_m),
                    )
                )

            traffic_light_diagnostics = _video_tl_draw_overlays(
                ax,
                frame,
                entity_table,
                traffic_light_events,
                draw_semantic_arrows_on_map=bool(
                    draw_semantic_arrows_on_map
                ),
            )

            direct_predicate_diagnostics = _video_direct_draw_events(
                ax,
                frame,
                entity_table,
                direct_predicate_events,
                draw_semantic_arrows_on_map=bool(
                    draw_semantic_arrows_on_map
                ),
            )

            # Remove upper-right legend.
            legend = ax.get_legend()
            if legend is not None:
                legend.remove()

            lane_change_overlays = []
            if (
                bool(semantic_categories.get("interaction", False))
                and bool(draw_semantic_arrows_on_map)
                and semantic_predicate_is_selected(
                    "np:changesLane",
                    semantic_predicates,
                )
            ):
                lane_change_overlays = _video_draw_lane_change_overlays(
                    ax,
                    frame,
                    frame_assertions,
                    semantic_relations,
                    entity_table=entity_table,
                )

            elapsed_s = (
                timestamp_us - first_timestamp_us
            ) / 1e6

            overtake_evidence_by_pair = _video_overtake_evidence_by_pair(
                frame_assertions
            )
            overtake_phase_counts = _VideoCounter(
                str(evidence.get("overtake_phase_label") or "unspecified")
                for evidence in overtake_evidence_by_pair.values()
            )
            overtake_phase_text = ", ".join(
                f"{phase}: {count}"
                for phase, count in sorted(overtake_phase_counts.items())
            )

            lane_change_phase_counts = _VideoCounter(
                str(
                    overlay["evidence"].get("lane_change_phase_label")
                    or "unspecified"
                )
                for overlay in lane_change_overlays
            )
            lane_change_phase_text = ", ".join(
                f"{phase}: {count}"
                for phase, count in sorted(lane_change_phase_counts.items())
            )

            label_counts = _video_relation_label_counts(
                plot_edges
            )
            if lane_change_overlays:
                label_counts["changesLane"] += len(lane_change_overlays)

            for traffic_event in traffic_light_events:
                label_counts[
                    str(traffic_event["relation_labels"])
                ] += 1
            for direct_event in direct_predicate_events:
                label_counts[
                    str(direct_event["relation_labels"])
                ] += 1
            label_text = ", ".join(
                f"{label}: {count}"
                for label, count in sorted(label_counts.items())
            )
            if not label_text:
                label_text = "none"

            follow_mode_counts = (
                _video_displayed_follow_mode_counts(
                    frame_assertions,
                    plot_edges,
                )
                if bool(
                    semantic_categories.get(
                        "interaction",
                        False,
                    )
                )
                else _VideoCounter()
            )
            follow_mode_text = ", ".join(
                f"{mode}: {count}"
                for mode, count in sorted(
                    follow_mode_counts.items()
                )
            )

            category_text = ", ".join(active_categories)
            relation_text = ", ".join(active_relations)

            ax.set_title(
                "Configurable semantic map video\n"
                f"scenario={scenario_token} | frame={frame_index} | "
                f"time={elapsed_s:.1f} s | active relations="
                f"{len(plot_edges) + len(lane_change_overlays) + len(traffic_light_events) + len(direct_predicate_events)}"
            )

            if show_status_box:
                status_lines = [
                    f"Categories: {category_text}",
                    f"Predicate filter: {active_predicate_filter}",
                    f"Relations: {relation_text}",
                    f"Predicates: {label_text}",
                ]

                if follow_mode_text:
                    status_lines.append(
                        f"Follow modes: {follow_mode_text}"
                    )
                if overtake_phase_text:
                    status_lines.append(
                        f"Overtake phase: {overtake_phase_text}"
                    )

                if lane_change_phase_text:
                    status_lines.append(
                        f"Lane-change phase: {lane_change_phase_text}"
                    )

                ax.text(
                    0.015,
                    0.985,
                    "\n".join(status_lines),
                    transform=ax.transAxes,
                    ha="left",
                    va="top",
                    fontsize=8.5,
                    zorder=30,
                    bbox={
                        "boxstyle": "round,pad=0.35",
                        "facecolor": "white",
                        "edgecolor": "#6a1b9a",
                        "alpha": 0.92,
                    },
                )

            fig.canvas.draw()
            rgba = np.asarray(
                fig.canvas.buffer_rgba()
            )
            rgb = np.ascontiguousarray(
                rgba[:, :, :3]
            )
            height, width = rgb.shape[:2]

            if writer is None:
                fourcc = cv2.VideoWriter_fourcc(
                    *"mp4v"
                )
                writer = cv2.VideoWriter(
                    str(temporary_video_path),
                    fourcc,
                    actual_fps,
                    (width, height),
                )

                if not writer.isOpened():
                    writer.release()
                    writer = None
                    raise RuntimeError(
                        "OpenCV could not open the MP4 writer. Confirm "
                        "that the active environment supports mp4v."
                    )

            bgr = cv2.cvtColor(
                rgb,
                cv2.COLOR_RGB2BGR,
            )
            writer.write(bgr)
            plt.close(fig)

            raw_ego_follow_assertion_count = (
                _video_count_raw_ego_follows(frame_assertions)
            )

            summary_rows.append({
                "rendered_frame": rendered_index,
                "scene_frame_index": frame_index,
                "timestamp_us": timestamp_us,
                "elapsed_s": elapsed_s,
                "selected_agent_count": int(
                    len(candidate_tokens)
                ),
                "displayed_predicate_agent_count": int(
                    len(predicate_tokens)
                ),
                "semantic_edge_count": int(
                    len(plot_edges)
                ),
                "active_relation_count": int(
                    len(plot_edges)
                    + len(lane_change_overlays)
                    + len(traffic_light_events)
                    + len(direct_predicate_events)
                ),
                "direct_predicate_event_count": int(
                    len(direct_predicate_events)
                ),
                "drawn_direct_value_label_count": int(
                    direct_predicate_diagnostics[
                        "drawn_value_labels"
                    ]
                ),
                "drawn_direct_map_relation_count": int(
                    direct_predicate_diagnostics[
                        "drawn_map_relations"
                    ]
                ),
                "traffic_light_event_count": int(
                    len(traffic_light_events)
                ),
                "drawn_isRelevantSignal_count": int(
                    traffic_light_diagnostics[
                        "drawn_is_relevant_signal"
                    ]
                ),
                "drawn_controls_count": int(
                    traffic_light_diagnostics[
                        "drawn_controls"
                    ]
                ),
                "drawn_signal_state_count": int(
                    traffic_light_diagnostics[
                        "drawn_states"
                    ]
                ),
                "raw_follow_assertion_count": int(
                    raw_follow_assertion_count
                ),
                "raw_overtake_assertion_count": int(
                    raw_overtake_assertion_count
                ),
                "raw_lane_change_assertion_count": int(
                    raw_lane_change_assertion_count
                ),
                "lane_change_overlay_count": int(
                    len(lane_change_overlays)
                ),
                "recovered_follow_edge_count": int(
                    recovered_follow_edge_count
                ),
                "raw_ego_follow_assertion_count": int(
                    raw_ego_follow_assertion_count
                ),
                "restored_ego_edge_count": int(
                    restored_ego_edge_count
                ),
                "enabled_ego_edge_count": int(
                    enabled_ego_edge_count
                ),
                "selected_ego_edge_count": int(
                    selected_ego_edge_count
                ),
                "plotted_ego_edge_count": int(
                    plotted_ego_edge_count
                ),
                "predicate_labels": label_text,
                "crosses_reference_pair_count": int(
                    crosses_reference_pair_count
                ),
                "follow_modes": (
                    follow_mode_text
                    if follow_mode_text
                    else "none"
                ),
                "overtake_phases": (
                    overtake_phase_text
                    if overtake_phase_text
                    else "none"
                ),
                "lane_change_phases": (
                    lane_change_phase_text
                    if lane_change_phase_text
                    else "none"
                ),
            })

            for traffic_event_index, traffic_event in enumerate(
                traffic_light_events,
                start=1,
            ):
                edge_event_rows.append({
                    "scene_frame_index": frame_index,
                    "timestamp_us": timestamp_us,
                    "elapsed_s": elapsed_s,
                    "edge_id_in_frame": (
                        f"TL{traffic_event_index}"
                    ),
                    "subject": traffic_event["subject"],
                    "object": traffic_event["object"],
                    "relation_type": traffic_event[
                        "relation_type"
                    ],
                    "categories": "traffic_light",
                    "relation_labels": traffic_event[
                        "relation_labels"
                    ],
                    "predicate_id": traffic_event[
                        "predicate_id"
                    ],
                    "traffic_signal_id": traffic_event[
                        "signal_id"
                    ],
                    "native_lane_connector_id": traffic_event[
                        "connector_id"
                    ],
                    "signal_state": traffic_event[
                        "state"
                    ],
                    "assertion_timestamp_us": traffic_event.get(
                        "assertion_timestamp_us"
                    ),
                    "frame_time_delta_us": traffic_event.get(
                        "frame_time_delta_us"
                    ),
                    "is_traffic_light_event": True,
                })

            for direct_event_index, direct_event in enumerate(
                direct_predicate_events,
                start=1,
            ):
                subject_token = str(
                    direct_event.get("subject_track_token") or ""
                )
                subject_display = subject_token
                if (
                    entity_table is not None
                    and not entity_table.empty
                    and subject_token
                    and "track_token" in entity_table.columns
                ):
                    _subject_rows = entity_table.loc[
                        entity_table["track_token"].astype(str).eq(
                            subject_token
                        )
                    ]
                    if not _subject_rows.empty:
                        subject_display = str(
                            _subject_rows.iloc[0].get(
                                "display_id",
                                subject_token,
                            )
                        )

                edge_event_rows.append({
                    "scene_frame_index": frame_index,
                    "timestamp_us": timestamp_us,
                    "elapsed_s": elapsed_s,
                    "edge_id_in_frame": f"D{direct_event_index}",
                    "subject": subject_display,
                    "object": (
                        direct_event.get("object_id")
                        or direct_event.get("value_text")
                    ),
                    "relation_type": direct_event.get(
                        "relation_type"
                    ),
                    "categories": direct_event.get("category"),
                    "relation_labels": direct_event.get(
                        "relation_labels"
                    ),
                    "predicate_id": direct_event.get("predicate_id"),
                    "predicate_value": direct_event.get("value"),
                    "predicate_value_text": direct_event.get(
                        "value_text"
                    ),
                    "predicate_unit": direct_event.get("unit"),
                    "direct_visualization_type": direct_event.get(
                        "visualization_type"
                    ),
                    "is_direct_predicate_event": True,
                })

            for overlay in lane_change_overlays:
                evidence = overlay["evidence"]
                edge_event_rows.append({
                    "scene_frame_index": frame_index,
                    "timestamp_us": timestamp_us,
                    "elapsed_s": elapsed_s,
                    "edge_id_in_frame": evidence.get("event_id"),
                    "subject": overlay["subject_token"],
                    "object": evidence.get("target_lane_entity_id") or overlay["object_id"],
                    "relation_type": overlay["relation_type"],
                    "categories": "interaction",
                    "relation_labels": "changesLane",
                    "is_active_lane_change": True,
                    "lane_change_side": evidence.get("lane_change_side"),
                    "lane_change_phase": evidence.get("lane_change_phase"),
                    "lane_change_phase_number": evidence.get("lane_change_phase_number"),
                    "lane_change_phase_label": evidence.get("lane_change_phase_label"),
                    "source_lane_id": evidence.get("source_lane_id"),
                    "target_lane_id": evidence.get("target_lane_id"),
                    "lane_change_arrow_start_x": evidence.get("lane_change_arrow_start_x"),
                    "lane_change_arrow_start_y": evidence.get("lane_change_arrow_start_y"),
                    "lane_change_arrow_end_x": evidence.get("lane_change_arrow_end_x"),
                    "lane_change_arrow_end_y": evidence.get("lane_change_arrow_end_y"),
                })

            follow_mode_by_pair = _video_follow_mode_by_pair(
                frame_assertions
            )

            if plot_edges is not None and not plot_edges.empty:
                for edge in plot_edges.itertuples(index=False):
                    edge_subject_token = str(
                        getattr(edge, "subject_token", "")
                    )
                    edge_object_token = str(
                        getattr(edge, "object_token", "")
                    )
                    edge_predicates = {
                        str(predicate)
                        for predicate in getattr(
                            edge, "predicates", []
                        )
                    }
                    follow_mode = (
                        follow_mode_by_pair.get((
                            edge_subject_token,
                            edge_object_token,
                        ))
                        if "np:follows" in edge_predicates
                        else None
                    )
                    overtake_evidence = (
                        overtake_evidence_by_pair.get((
                            edge_subject_token,
                            edge_object_token,
                        ), {})
                        if "np:overtakes" in edge_predicates
                        else {}
                    )

                    edge_predicate_list = sorted(edge_predicates)
                    edge_event_rows.append({
                        "scene_frame_index": frame_index,
                        "timestamp_us": timestamp_us,
                        "elapsed_s": elapsed_s,
                        "predicate_id": (
                            edge_predicate_list[0]
                            if len(edge_predicate_list) == 1
                            else None
                        ),
                        "predicate_ids": ",".join(edge_predicate_list),
                        "edge_id_in_frame": getattr(
                            edge,
                            "edge_id",
                            None,
                        ),
                        "subject": getattr(
                            edge,
                            "subject_display_id",
                            getattr(edge, "subject_token", None),
                        ),
                        "object": getattr(
                            edge,
                            "object_display_id",
                            getattr(edge, "object_token", None),
                        ),
                        "relation_type": getattr(
                            edge,
                            "relation_type",
                            None,
                        ),
                        "categories": ", ".join(
                            map(
                                str,
                                getattr(edge, "categories", []),
                            )
                        ),
                        "relation_labels": getattr(
                            edge,
                            "relation_labels",
                            None,
                        ),
                        "follow_mode": follow_mode,
                        "is_moving_headway": (
                            follow_mode == "moving_headway"
                        ),
                        "is_queue_or_stop_and_go": (
                            follow_mode
                            == "queue_or_stop_and_go"
                        ),
                        "is_active_overtake": (
                            "np:overtakes" in edge_predicates
                        ),
                        "is_completed_overtake": overtake_evidence.get(
                            "overtake_completed"
                        ),
                        "overtake_phase": overtake_evidence.get(
                            "overtake_phase"
                        ),
                        "overtake_phase_number": overtake_evidence.get(
                            "overtake_phase_number"
                        ),
                        "overtake_phase_label": overtake_evidence.get(
                            "overtake_phase_label"
                        ),
                        "overtake_assignment_scope": overtake_evidence.get(
                            "overtake_assignment_scope"
                        ),
                        "overtake_case": overtake_evidence.get(
                            "overtake_case"
                        ),
                        "overtake_case_number": overtake_evidence.get(
                            "overtake_case_number"
                        ),
                        "overtake_case_label": overtake_evidence.get(
                            "overtake_case_label"
                        ),
                        "overtake_initial_lane_relation": overtake_evidence.get(
                            "overtake_initial_lane_relation"
                        ),
                        "overtake_observation_mode": overtake_evidence.get(
                            "overtake_observation_mode"
                        ),
                        "relation_family": overtake_evidence.get(
                            "relation_family"
                        ),
                        "lane_departure_observed": overtake_evidence.get(
                            "lane_departure_observed"
                        ),
                        "overtake_entry_mode": overtake_evidence.get(
                            "overtake_entry_mode"
                        ),
                        "initial_follow_observed": overtake_evidence.get(
                            "initial_follow_observed"
                        ),
                        "initial_follow_mode": overtake_evidence.get(
                            "initial_follow_mode"
                        ),
                        "overtake_completion_mode": overtake_evidence.get(
                            "overtake_completion_mode"
                        ),
                        "returned_to_original_lane": overtake_evidence.get(
                            "returned_to_original_lane"
                        ),
                        "remained_in_passing_lane": overtake_evidence.get(
                            "remained_in_passing_lane"
                        ),
                        "passing_side": overtake_evidence.get(
                            "passing_side"
                        ),
                        "initial_lane_id": overtake_evidence.get(
                            "initial_lane_id"
                        ),
                        "initial_subject_lane_id": overtake_evidence.get(
                            "initial_subject_lane_id"
                        ),
                        "initial_object_lane_id": overtake_evidence.get(
                            "initial_object_lane_id"
                        ),
                        "passing_lane_id": overtake_evidence.get(
                            "passing_lane_id"
                        ),
                        "final_lane_id": overtake_evidence.get(
                            "final_lane_id"
                        ),
                        "overtake_start_time_us": overtake_evidence.get(
                            "start_time_us"
                        ),
                        "lane_departure_time_us": overtake_evidence.get(
                            "lane_departure_time_us"
                        ),
                        "passing_lane_observation_start_time_us": overtake_evidence.get(
                            "passing_lane_observation_start_time_us"
                        ),
                        "side_by_side_time_us": overtake_evidence.get(
                            "side_by_side_time_us"
                        ),
                        "order_reversal_time_us": overtake_evidence.get(
                            "order_reversal_time_us"
                        ),
                        "clearance_time_us": overtake_evidence.get(
                            "clearance_time_us"
                        ),
                        "return_time_us": overtake_evidence.get(
                            "return_time_us"
                        ),
                        "final_order_stable_start_time_us": overtake_evidence.get(
                            "final_order_stable_start_time_us"
                        ),
                        "completion_time_us": overtake_evidence.get(
                            "completion_time_us"
                        ),
                        "final_order_frame_count": overtake_evidence.get(
                            "final_order_frame_count"
                        ),
                        "final_clearance_gap_m": overtake_evidence.get(
                            "final_clearance_gap_m"
                        ),
                        "maximum_relative_speed_mps": overtake_evidence.get(
                            "maximum_relative_speed_mps"
                        ),
                    })

            if (
                rendered_index == 0
                or (rendered_index + 1) % 10 == 0
                or rendered_index + 1 == len(frame_indices)
            ):
                print(
                    f"Rendered {rendered_index + 1:,}/"
                    f"{len(frame_indices):,} frames; "
                    f"active semantic edges={len(plot_edges):,}; "
                    f"traffic-light events={len(traffic_light_events):,}; "
                    f"isRelevantSignal drawn="
                    f"{traffic_light_diagnostics['drawn_is_relevant_signal']:,}; "
                    f"raw follows={raw_follow_assertion_count:,}; "
                    f"raw overtakes={raw_overtake_assertion_count:,}; "
                    f"recovered follows={recovered_follow_edge_count:,}; "
                    f"raw/restored/enabled/selected/plotted EGO="
                    f"{raw_ego_follow_assertion_count:,}/"
                    f"{restored_ego_edge_count:,}/"
                    f"{enabled_ego_edge_count:,}/"
                    f"{selected_ego_edge_count:,}/"
                    f"{plotted_ego_edge_count:,}"
                )

    finally:
        if writer is not None:
            writer.release()
        plt.close("all")

    if (
        not temporary_video_path.exists()
        or temporary_video_path.stat().st_size == 0
    ):
        raise RuntimeError(
            "The OpenCV video writer did not create a valid temporary "
            f"MP4V file: {temporary_video_path}"
        )

    _video_convert_mp4v_to_browser_h264(
        temporary_video_path,
        video_path,
    )

    summary = pd.DataFrame(summary_rows)
    edge_events = pd.DataFrame(edge_event_rows)

    active_frame_count = (
        int(
            (summary["active_relation_count"] > 0).sum()
        )
        if not summary.empty
        else 0
    )

    print("\nMAP-ONLY SEMANTIC VIDEO SAVED")
    print("Scenario catalog row:", scenario_catalog_row)
    print("Scenario token:", scenario_token)
    print("Categories:", active_categories)
    print("Predicate filter:", active_predicate_filter)
    print("Relations:", active_relations)
    print("Rendered frames:", len(frame_indices))
    print("Frames containing enabled semantic relations:", active_frame_count)
    print("FPS:", actual_fps)
    print("Codec: H.264 (libx264), pixel format: yuv420p")
    print("Web optimization: faststart enabled")
    print("File:", video_path.resolve())

    return {
        "video_path": video_path,
        "scenario": scenario,
        "scenario_token": scenario_token,
        "scenario_catalog_row": scenario_catalog_row,
        "fps": actual_fps,
        "frame_indices": frame_indices,
        "summary": summary,
        "edge_events": edge_events,
        "active_frame_count": active_frame_count,
        "active_categories": active_categories,
        "active_predicates": active_predicate_filter,
        "active_relations": active_relations,
    }


VIDEO_RESULT = render_full_scene_configurable_semantic_video(
    scenario_selector=VIDEO_SCENARIO_SELECTOR,
    scenario_value=VIDEO_SCENARIO_VALUE,
    scenario_seed=VIDEO_SCENARIO_SEED,
    filter_seed=VIDEO_FILTER_SEED,
    output_dir=OUTPUT_DIR,
    export_root=VIDEO_EXPORT_ROOT,
    start_frame=VIDEO_START_FRAME,
    end_frame=VIDEO_END_FRAME,
    frame_step=VIDEO_FRAME_STEP,
    fps=VIDEO_FPS,
    playback_speed=VIDEO_PLAYBACK_SPEED,
    figsize=VIDEO_FIGSIZE,
    dpi=VIDEO_DPI,
    semantic_categories=VIDEO_SEMANTIC_CATEGORIES,
    semantic_relations=VIDEO_SEMANTIC_RELATIONS,
    semantic_predicates=VIDEO_SEMANTIC_PREDICATES,
    agent_types=VIDEO_AGENT_TYPES,
    interested_distance_threshold_m=(
        VIDEO_INTERESTED_DISTANCE_THRESHOLD_M
    ),
    include_within_distance=(
        VIDEO_INCLUDE_WITHIN_DISTANCE
    ),
    include_forward_corridor=(
        VIDEO_INCLUDE_FORWARD_CORRIDOR
    ),
    forward_corridor_length_m=(
        VIDEO_FORWARD_CORRIDOR_LENGTH_M
    ),
    forward_corridor_half_width_m=(
        VIDEO_FORWARD_CORRIDOR_HALF_WIDTH_M
    ),
    include_same_lane=(
        VIDEO_INCLUDE_SAME_LANE
    ),
    include_predicted_path_intersection=(
        VIDEO_INCLUDE_PREDICTED_PATH_INTERSECTION
    ),
    prediction_horizon_s=(
        VIDEO_PREDICTION_HORIZON_S
    ),
    prediction_step_s=(
        VIDEO_PREDICTION_STEP_S
    ),
    path_intersection_clearance_m=(
        VIDEO_PATH_INTERSECTION_CLEARANCE_M
    ),
    include_existing_spatial_agents=(
        VIDEO_INCLUDE_EXISTING_SPATIAL_AGENTS
    ),
    filter_edges_to_selected_agents=(
        VIDEO_FILTER_EDGES_TO_SELECTED_AGENTS
    ),
    manual_highlight_tokens=(
        VIDEO_MANUAL_HIGHLIGHT_TOKENS
    ),
    map_radius_m=VIDEO_MAP_RADIUS_M,
    draw_semantic_arrows_on_map=(
        VIDEO_DRAW_SEMANTIC_ARROWS_ON_MAP
    ),
    show_agent_labels=VIDEO_SHOW_AGENT_LABELS,
    show_agent_lane_info=VIDEO_SHOW_AGENT_LANE_INFO,
    show_map_structure_ids=VIDEO_SHOW_MAP_STRUCTURE_IDS,
    show_edge_ids=VIDEO_SHOW_EDGE_IDS,
    show_status_box=VIDEO_SHOW_STATUS_BOX,
    draw_crosses_reference_lines=(
        VIDEO_DRAW_CROSSES_REFERENCE_LINES
    ),
)


# Display the saved MP4 without embedding all video bytes.
display(
    _NotebookVideo(
        filename=str(VIDEO_RESULT["video_path"]),
        embed=False,
    )
)


# ============================================================
# OPTIONAL OUTPUTS CONTROLLED BY CELL 15 SWITCHES
# ============================================================

active_video_frames = VIDEO_RESULT["summary"].loc[
    VIDEO_RESULT["summary"]["semantic_edge_count"] > 0
].reset_index(drop=True)

if VIDEO_SHOW_FILTERED_TABLES:
    print("\nFRAMES WITH ACTIVE ENABLED SEMANTIC RELATIONS")

    if active_video_frames.empty:
        print(
            "No enabled semantic relation occurred in the rendered "
            "frame range."
        )
    else:
        display(
            active_video_frames.head(
                VIDEO_MAX_FILTERED_TABLE_ROWS
            )
        )

        if (
            len(active_video_frames)
            > VIDEO_MAX_FILTERED_TABLE_ROWS
        ):
            print(
                "Showing "
                f"{VIDEO_MAX_FILTERED_TABLE_ROWS:,} of "
                f"{len(active_video_frames):,} active frames."
            )

if VIDEO_SHOW_EDGE_LEGEND:
    print("\nSEMANTIC RELATIONS OBSERVED IN THE VIDEO")

    edge_events = VIDEO_RESULT["edge_events"]

    if edge_events.empty:
        print(
            "No enabled semantic relation was drawable in the video."
        )
    else:
        video_edge_legend = (
            edge_events.groupby(
                [
                    "subject",
                    "object",
                    "relation_type",
                    "categories",
                    "relation_labels",
                ],
                dropna=False,
            )
            .agg(
                first_frame=("scene_frame_index", "min"),
                last_frame=("scene_frame_index", "max"),
                active_frame_count=("scene_frame_index", "nunique"),
            )
            .reset_index()
            .sort_values(
                [
                    "first_frame",
                    "subject",
                    "object",
                    "relation_labels",
                ]
            )
            .reset_index(drop=True)
        )

        if VIDEO_MAX_EDGE_LEGEND_ROWS is None:
            display(video_edge_legend)
        else:
            display(
                video_edge_legend.head(
                    int(VIDEO_MAX_EDGE_LEGEND_ROWS)
                )
            )
